In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import sys, os, platform, shutil, subprocess, zipfile
from pathlib import Path

print("=== ENVIRONMENT ===")
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
for mod in ["torch", "transformers", "numpy", "PIL", "pandas", "wandb"]:
    try:
        m = __import__(mod)
        print(f"{mod:<12}:", getattr(m, "__version__", "?"))
    except Exception as e:
        print(f"{mod:<12}: NOT INSTALLED ({type(e).__name__})")

try:
    import torch
    print("CUDA avail  :", torch.cuda.is_available())
    print("CUDA (torch):", torch.version.cuda)
    print("cuDNN       :", torch.backends.cudnn.version())
    print("GPU count   :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, capability {p.major}.{p.minor}")
except Exception as e:
    print("torch check failed:", e)

print("\n=== nvidia-smi ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)

print("=== DISK ===")
for p in ["/kaggle/working", "/kaggle/input", "/tmp"]:
    if os.path.exists(p):
        u = shutil.disk_usage(p)
        print(f"{p:<16} total {u.total/1e9:7.1f} GB | free {u.free/1e9:7.1f} GB")

print("\n=== /kaggle/input TREE (depth <= 3) ===")
root = Path("/kaggle/input")
def show(d, depth=0, max_depth=3):
    try:
        items = sorted(d.iterdir())
    except Exception as e:
        print("  " * depth, "[unreadable]", e); return
    files = [x for x in items if x.is_file()]
    dirs  = [x for x in items if x.is_dir()]
    for f in files[:15]:
        print("  " * depth + f"{f.name}  ({f.stat().st_size/1e6:.2f} MB)")
    if len(files) > 15:
        print("  " * depth + f"... and {len(files)-15} more files in this folder")
    for sd in dirs:
        n = sum(1 for _ in sd.rglob("*") if _.is_file())
        print("  " * depth + f"[DIR] {sd.name}/  ({n} files inside)")
        if depth + 1 < max_depth:
            show(sd, depth + 1, max_depth)
show(root)

print("\n=== ZIP CONTENTS (if any ZIPs are present) ===")
zips = sorted(root.rglob("*.zip"))
print("ZIP files found:", len(zips))
for z in zips:
    try:
        with zipfile.ZipFile(z) as zf:
            names = zf.namelist()
            print(f"\n{z}\n  entries: {len(names)}")
            for n in names[:6]:
                print("   ", n)
    except Exception as e:
        print(f"{z}: cannot open ({e})")

=== ENVIRONMENT ===
Python      : 3.13.16
Platform    : Linux-6.18.55+-x86_64-with-glibc2.39
torch       : 2.11.0+cu130
transformers: 5.20.2
numpy       : 2.1.3
PIL         : 13.0.0
pandas      : 2.3.3
wandb       : 0.28.1
CUDA avail  : True
CUDA (torch): 13.0
cuDNN       : 92700
GPU count   : 2
  GPU 0: Tesla T4, 15.6 GB, capability 7.5
  GPU 1: Tesla T4, 15.6 GB, capability 7.5

=== nvidia-smi ===
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04

=== DISK ===
/kaggle/working  total    21.0 GB | free    20.9 GB
/kaggle/input    total    21.0 GB | free    20.9 GB
/tmp             total  8758.4 GB | free  1038.2 GB

=== /kaggle/input TREE (depth <= 3) ===
[DIR] datasets/  (23008 files inside)
  [DIR] hudafatimah/  (23008 files inside)
    [DIR] mars-terrain-v3-512/  (23008 files inside)

=== ZIP CONTENTS (if any ZIPs are present) ===
ZIP files found: 0


Datasets under /kaggle/input/datasets/hudafatimah : ['mars-terrain-v3-512']

DATASET: /kaggle/input/datasets/hudafatimah/mars-terrain-v3-512
.                                                            1 files        0.0 MB  {'.json': 1}
dataset_v3_512_train_part1                                   0 files        0.0 MB  {}
dataset_v3_512_train_part1/dataset_v3_512                    0 files        0.0 MB  {}
dataset_v3_512_train_part1/dataset_v3_512/train/img       3268 files      433.6 MB  {'.png': 3268}
dataset_v3_512_train_part1/dataset_v3_512/train/mask      3268 files        8.2 MB  {'.png': 3268}
dataset_v3_512_train_part2                                   0 files        0.0 MB  {}
dataset_v3_512_train_part2/dataset_v3_512                    0 files        0.0 MB  {}
dataset_v3_512_train_part2/dataset_v3_512/train/img       3268 files      427.2 MB  {'.png': 3268}
dataset_v3_512_train_part2/dataset_v3_512/train/mask      3268 files        9.5 MB  {'.png': 3268}
dataset_v3_512_tra

In [ ]:
# Kaggle Cell 2b: read-only integrity check

Kaggle auto-extracted your ZIPs, so the original MD5 checks can't be run here. They were already verified on your laptop and in earlier Colab sessions. On Kaggle I verify the extracted files directly instead.

Your counts match the frozen split:

| Split | Expected | Found |
|---|---:|---:|
| Train (part1 + part2 + part3) | 9,803 | 3,268 + 3,268 + 3,267 = 9,803 |
| Val | 1,192 | 1,192 |
| MSL test | 311 | 311 |
| MER test | 195 | 195 |

Two things to know about the layout:
- **Train is split across three folders**, each with its own `img/` and `mask/`. The loader will have to look in all three. In the next step I'll copy everything into one `/kaggle/working/mars_work/dataset_v3_512/` tree, so it matches your laptop layout.
- **The input folder is read-only.** Nothing in this cell writes anywhere.

This cell checks the following, without assuming the CSV column names:
1. The image and mask filename stems pair up exactly in every split.
2. There are no duplicate stems across the three train parts.
3. Every image is 512×512 grayscale, and every mask is 512×512 single-channel.
4. The label values across all masks are exactly `{0,1,2,3,255}`, with pixel shares per class.
5. It prints the CSV columns and the keys of `split_v3_FROZEN.json`.

```python
import json, collections
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

R = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
VT = R / "dataset_v3_512_val_test_csv" / "dataset_v3_512"

dirs = {
    "train": [(R / f"dataset_v3_512_train_part{i}" / "dataset_v3_512" / "train") for i in (1, 2, 3)],
    "val":      [VT / "val"],
    "test_msl": [VT / "test_msl"],
    "test_mer": [VT / "test_mer"],
}
expected = {"train": 9803, "val": 1192, "test_msl": 311, "test_mer": 195}

summary = {}
all_ok = True
for split, parts in dirs.items():
    imgs, masks = {}, {}
    dup = 0
    for p in parts:
        for f in (p / "img").glob("*.png"):
            if f.stem in imgs: dup += 1
            imgs[f.stem] = f
        for f in (p / "mask").glob("*.png"):
            masks[f.stem] = f
    only_img = set(imgs) - set(masks)
    only_msk = set(masks) - set(imgs)

    bad_img, bad_msk = [], []
    counts = np.zeros(256, dtype=np.int64)
    for s, f in imgs.items():
        with Image.open(f) as im:
            if im.size != (512, 512) or im.mode != "L":
                bad_img.append((f.name, im.size, im.mode))
    for s, f in masks.items():
        with Image.open(f) as m:
            if m.size != (512, 512) or m.mode not in ("L", "P"):
                bad_msk.append((f.name, m.size, m.mode))
            counts += np.bincount(np.asarray(m, dtype=np.uint8).ravel(), minlength=256)

    present = [int(v) for v in np.nonzero(counts)[0]]
    ok = (len(imgs) == expected[split] and not only_img and not only_msk
          and dup == 0 and not bad_img and not bad_msk and present == [0, 1, 2, 3, 255])
    all_ok &= ok
    lab = counts[[0, 1, 2, 3]].sum()
    shares = {k: round(100 * counts[i] / lab, 2) for i, k in enumerate(["soil", "bedrock", "sand", "bigrock"])}
    print(f"\n[{split}] images={len(imgs)} masks={len(masks)} (expected {expected[split]}) "
          f"dup_stems={dup} unpaired_img={len(only_img)} unpaired_mask={len(only_msk)}")
    print(f"   bad image format: {len(bad_img)} {bad_img[:2]} | bad mask format: {len(bad_msk)} {bad_msk[:2]}")
    print(f"   label values present: {present} | ignore(255) share of all px: {100*counts[255]/counts.sum():.2f}%")
    print(f"   class share of labelled px (%): {shares}")
    print("   ->", "OK" if ok else "PROBLEM")
    summary[split] = {"n": len(imgs), "ok": bool(ok)}

print("\n=== CSVs ===")
for name in ["train", "val", "test_msl", "test_mer"]:
    df = pd.read_csv(VT.parent / "dataset_v3_512_val_test_csv" / "dataset_v3_512" / f"{name}.csv")
    print(f"{name}.csv rows={len(df)} columns={list(df.columns)}")
    if name == "train":
        print(df.head(2).to_string())

print("\n=== split_v3_FROZEN.json ===")
sp = json.load(open(R / "split_v3_FROZEN.json"))
def desc(x, depth=0):
    if isinstance(x, dict):
        for k, v in list(x.items())[:12]:
            t = f"len={len(v)}" if isinstance(v, (list, dict)) else repr(v)[:60]
            print("  " * depth + f"{k}: {type(v).__name__} {t}")
            if isinstance(v, dict) and depth < 1: desc(v, depth + 1)
    else:
        print(type(x).__name__, len(x) if hasattr(x, "__len__") else x)
desc(sp)

print("\nOVERALL:", "ALL CHECKS PASSED" if all_ok else "SOMETHING FAILED, DO NOT PROCEED")
```

## What to expect
- Four `[split]` blocks, each ending with `-> OK`, and `ALL CHECKS PASSED` at the bottom.
- Label values `[0, 1, 2, 3, 255]` in every split. Big Rock should be roughly 0.8% of labelled pixels in train.
- The CSV columns and head, so I can write the loader against the real column names.
- The JSON structure, so I can check it against the CSVs.

It reads about 23,000 small PNGs, so it may take a minute or two. Paste the full output unedited. If anything says PROBLEM, stop and send it to me before we go further.

Next I'll give you the cell that copies the dataset into `/kaggle/working/mars_work/`, creates the folder tree and records the environment file.

In [6]:
import json, os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

R  = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
VT = R / "dataset_v3_512_val_test_csv" / "dataset_v3_512"

dirs = {
    "train": [R / f"dataset_v3_512_train_part{i}" / "dataset_v3_512" / "train" for i in (1, 2, 3)],
    "val": [VT / "val"], "test_msl": [VT / "test_msl"], "test_mer": [VT / "test_mer"],
}
expected = {"train": 9803, "val": 1192, "test_msl": 311, "test_mer": 195}
VALID = {0, 1, 2, 3, 255}

# Output folder tree (outputs only; dataset is NOT copied)
W = Path("/kaggle/working/mars_work")
for sub in ["checkpoints/segformer_v1", "code", "configs", "figures", "logs", "splits",
            "results/lr_pilot", "results/baseline", "results/compression", "backups"]:
    (W / sub).mkdir(parents=True, exist_ok=True)

audit, all_ok, stems_by_split = {}, True, {}
for split, parts in dirs.items():
    imgs, masks, dup_img, dup_msk = {}, {}, 0, 0
    for p in parts:
        for f in (p / "img").glob("*.png"):
            dup_img += f.stem in imgs; imgs[f.stem] = f
        for f in (p / "mask").glob("*.png"):
            dup_msk += f.stem in masks; masks[f.stem] = f
    unpaired_img, unpaired_msk = set(imgs) - set(masks), set(masks) - set(imgs)

    bad_img, bad_msk = [], []
    counts = np.zeros(256, dtype=np.int64)
    for f in imgs.values():
        with Image.open(f) as im:
            if im.size != (512, 512) or im.mode != "L":
                bad_img.append((f.name, im.size, im.mode))
    for f in masks.values():
        with Image.open(f) as m:
            if m.size != (512, 512) or m.mode not in ("L", "P"):
                bad_msk.append((f.name, m.size, m.mode))
            counts += np.bincount(np.asarray(m, dtype=np.uint8).ravel(), minlength=256)

    present = {int(v) for v in np.nonzero(counts)[0]}
    unexpected, missing = sorted(present - VALID), sorted(VALID - present)
    ok = (len(imgs) == expected[split] and not unpaired_img and not unpaired_msk
          and dup_img == 0 and dup_msk == 0 and not bad_img and not bad_msk and not unexpected)
    all_ok &= ok
    stems_by_split[split] = set(imgs)
    print(f"[{split}] img={len(imgs)} mask={len(masks)} (expected {expected[split]}) "
          f"dup_img={dup_img} dup_mask={dup_msk} unpaired_img={len(unpaired_img)} unpaired_mask={len(unpaired_msk)}")
    print(f"   bad img: {len(bad_img)} | bad mask: {len(bad_msk)} | unexpected labels: {unexpected} | "
          f"classes absent from this split: {missing} -> {'OK' if ok else 'PROBLEM'}")
    audit[split] = {"images": len(imgs), "masks": len(masks), "ok": bool(ok),
                    "label_pixel_counts": {int(k): int(counts[k]) for k in present}}

# Stems must not overlap between splits (guards against leakage)
names = list(stems_by_split)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        ov = stems_by_split[names[i]] & stems_by_split[names[j]]
        print(f"overlap {names[i]} & {names[j]}: {len(ov)}")
        all_ok &= (len(ov) == 0)

print("\n=== CSVs ===")
for name in ["train", "val", "test_msl", "test_mer"]:
    df = pd.read_csv(VT / f"{name}.csv")
    print(f"{name}.csv rows={len(df)} (expected {expected[name]}) columns={list(df.columns)}")
    all_ok &= (len(df) == expected[name])
    for c in df.columns:
        if df[c].dtype == object:
            st = df[c].astype(str).map(lambda v: Path(v).stem)
            hit = st.isin(stems_by_split[name]).sum()
            if hit:
                print(f"   column '{c}': {hit}/{len(df)} values match image stems of this split")
    if name == "train":
        print(df.head(2).to_string())

print("\n=== split_v3_FROZEN.json ===")
sp = json.load(open(R / "split_v3_FROZEN.json"))
def desc(x, depth=0):
    if isinstance(x, dict):
        for k, v in list(x.items())[:12]:
            t = f"len={len(v)}" if isinstance(v, (list, dict)) else repr(v)[:60]
            print("  " * depth + f"{k}: {type(v).__name__} {t}")
            if isinstance(v, dict) and depth < 1: desc(v, depth + 1)
    else:
        print(type(x).__name__, len(x) if hasattr(x, "__len__") else x)
desc(sp)

audit["all_ok"] = bool(all_ok)
json.dump(audit, open(W / "logs" / "dataset_audit_kaggle.json", "w"), indent=2)
print("\nOVERALL:", "ALL CHECKS PASSED" if all_ok else "SOMETHING FAILED, DO NOT PROCEED")
print("Audit saved to:", W / "logs" / "dataset_audit_kaggle.json")

KeyboardInterrupt: 

In [1]:
from pathlib import Path
import torch

R = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")

print("Dataset exists:", R.exists())
print("Dataset folders:", [p.name for p in R.iterdir()])

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

Dataset exists: True
Dataset folders: ['dataset_v3_512_train_part1', 'dataset_v3_512_train_part2', 'dataset_v3_512_train_part3', 'dataset_v3_512_val_test_csv', 'split_v3_FROZEN.json']
CUDA available: True
GPU: Tesla T4
GPU count: 2


In [2]:
import time, random, sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image

R = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
imgs = []
for i in (1, 2, 3):
    imgs += list((R / f"dataset_v3_512_train_part{i}" / "dataset_v3_512" / "train" / "img").glob("*.png"))
print("train images listed:", len(imgs), flush=True)

random.seed(0)
random.shuffle(imgs)
A, B = imgs[:300], imgs[300:600]

def load(f):
    with Image.open(f) as im:
        return np.asarray(im).shape

t = time.time()
for k, f in enumerate(A):
    load(f)
    if (k + 1) % 50 == 0:
        print(f"  sequential {k+1}/300  {time.time()-t:.1f}s", flush=True)
t_seq = time.time() - t
print(f"1) sequential cold : {t_seq:.1f}s  ({t_seq/300*1000:.0f} ms/file)", flush=True)

t = time.time()
with ThreadPoolExecutor(16) as ex:
    list(ex.map(load, B))
t_thr = time.time() - t
print(f"2) 16 threads cold : {t_thr:.1f}s  ({t_thr/300*1000:.0f} ms/file)", flush=True)

t = time.time()
for f in A:
    load(f)
t_warm = time.time() - t
print(f"3) sequential warm : {t_warm:.1f}s  ({t_warm/300*1000:.0f} ms/file)", flush=True)

print(f"\nProjection for all 23,008 files, sequential cold: {t_seq/300*23008/60:.1f} min")
print(f"Projection with 16 threads, cold              : {t_thr/300*23008/60:.1f} min")

train images listed: 9803
  sequential 50/300  4.3s
  sequential 100/300  8.5s
  sequential 150/300  12.8s
  sequential 200/300  16.7s
  sequential 250/300  20.8s
  sequential 300/300  24.7s
1) sequential cold : 24.7s  (82 ms/file)
2) 16 threads cold : 1.7s  (6 ms/file)
3) sequential warm : 1.4s  (5 ms/file)

Projection for all 23,008 files, sequential cold: 31.6 min
Projection with 16 threads, cold              : 2.2 min


In [5]:

import json
import shutil
import time
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


# ============================================================
# 0. PATHS AND SETUP
# ============================================================

R = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
VT = R / "dataset_v3_512_val_test_csv" / "dataset_v3_512"

L = Path("/tmp/mars_data")
W = Path("/kaggle/working/mars_work")

assert R.exists(), f"Dataset not found: {R}"

for sub in [
    "checkpoints/segformer_v1",
    "code",
    "configs",
    "figures",
    "logs",
    "splits",
    "results/lr_pilot",
    "results/baseline",
    "results/compression",
    "backups"
]:
    (W / sub).mkdir(parents=True, exist_ok=True)

src_dirs = {
    "train": [
        R / f"dataset_v3_512_train_part{i}" / "dataset_v3_512" / "train"
        for i in (1, 2, 3)
    ],
    "val": [VT / "val"],
    "test_msl": [VT / "test_msl"],
    "test_mer": [VT / "test_mer"]
}

expected = {
    "train": 9803,
    "val": 1192,
    "test_msl": 311,
    "test_mer": 195
}

VALID = {0, 1, 2, 3, 255}


# ============================================================
# 1. BUILD COPY LIST
# ============================================================

tasks = []

for split, parts in src_dirs.items():
    for p in parts:
        for kind in ("img", "mask"):
            source_dir = p / kind
            assert source_dir.exists(), f"Missing directory: {source_dir}"

            for f in source_dir.glob("*.png"):
                tasks.append((f, L / split / kind / f.name))

for name in expected:
    tasks.append((VT / f"{name}.csv", L / f"{name}.csv"))

tasks.append((VT / "preprocessing.json", L / "preprocessing.json"))
tasks.append((R / "split_v3_FROZEN.json", L / "split_v3_FROZEN.json"))

print("Files to place:", len(tasks), flush=True)

# Confirm all sources exist.
missing_sources = [str(s) for s, d in tasks if not s.is_file()]
assert not missing_sources, f"Missing source files: {missing_sources[:10]}"

# Prevent filename collisions when merging train parts.
dsts = [d for s, d in tasks]
assert len(set(dsts)) == len(dsts), (
    "FILENAME COLLISION between training parts. Stop."
)

# Confirm expected file count.
expected_files = 2 * sum(expected.values()) + 6
assert len(tasks) == expected_files, (
    f"Unexpected task count: {len(tasks)} vs {expected_files}"
)


# ============================================================
# 2. THREADED COPY TO LOCAL TEMPORARY STORAGE
# ============================================================

def copy_one(task):
    source, destination = task

    if (
        destination.exists()
        and destination.stat().st_size == source.stat().st_size
    ):
        return "skipped"

    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary = destination.with_suffix(
        destination.suffix + ".part"
    )

    shutil.copyfile(source, temporary)
    temporary.replace(destination)

    if destination.stat().st_size != source.stat().st_size:
        return "SIZE_MISMATCH"

    return "copied"


print("\n=== COPYING DATASET ===", flush=True)

t0 = time.time()

with ThreadPoolExecutor(max_workers=16) as executor:
    copy_results = list(
        tqdm(
            executor.map(copy_one, tasks),
            total=len(tasks),
            desc="Copying files",
            mininterval=2
        )
    )

print(
    "Copy results:",
    dict(Counter(copy_results)),
    f"| Time: {time.time() - t0:.1f}s",
    flush=True
)

assert "SIZE_MISMATCH" not in copy_results, (
    "File size mismatch after copying."
)


# ============================================================
# 3. SHA256 VERIFICATION OF EVERY FILE
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def verify_copy(task):
    source, destination = task

    source_hash = sha256_file(source)
    destination_hash = sha256_file(destination)

    return (
        str(destination.relative_to(L)),
        source_hash,
        destination_hash
    )


print("\n=== SHA256 VERIFICATION ===", flush=True)

with ThreadPoolExecutor(max_workers=16) as executor:
    verification = list(
        tqdm(
            executor.map(verify_copy, tasks),
            total=len(tasks),
            desc="SHA256",
            mininterval=2
        )
    )

mismatches = [
    path for path, source_hash, dest_hash in verification
    if source_hash != dest_hash
]

print("SHA256 mismatches:", len(mismatches), mismatches[:5])

assert not mismatches, "SHA256 verification failed. Stop."

manifest = {
    path: source_hash
    for path, source_hash, destination_hash in verification
}

with open(W / "logs" / "dataset_sha256_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

shutil.copyfile(
    L / "split_v3_FROZEN.json",
    W / "splits" / "split_v3_FROZEN.json"
)

print("SHA256 manifest saved.", flush=True)


# ============================================================
# 4. LOCAL IMAGE AND MASK AUDIT
# ============================================================

def check_image(path):
    with Image.open(path) as im:
        return path.stem, im.size, im.mode


def check_mask(path):
    with Image.open(path) as mask:
        array = np.asarray(mask, dtype=np.uint8)

        return (
            path.stem,
            mask.size,
            mask.mode,
            np.bincount(array.ravel(), minlength=256)
        )


all_ok = True
audit = {}
stems_by_split = {}

print("\n=== IMAGE / MASK AUDIT ===", flush=True)

for split in expected:

    images = sorted((L / split / "img").glob("*.png"))
    masks = sorted((L / split / "mask").glob("*.png"))

    with ThreadPoolExecutor(max_workers=16) as executor:

        image_results = list(
            tqdm(
                executor.map(check_image, images),
                total=len(images),
                desc=f"{split} images",
                mininterval=2
            )
        )

        mask_results = list(
            tqdm(
                executor.map(check_mask, masks),
                total=len(masks),
                desc=f"{split} masks",
                mininterval=2
            )
        )

    image_stems = [r[0] for r in image_results]
    mask_stems = [r[0] for r in mask_results]

    image_set = set(image_stems)
    mask_set = set(mask_stems)

    duplicate_images = len(image_stems) - len(image_set)
    duplicate_masks = len(mask_stems) - len(mask_set)

    unpaired_images = image_set - mask_set
    unpaired_masks = mask_set - image_set

    bad_images = [
        r for r in image_results
        if r[1] != (512, 512) or r[2] != "L"
    ]

    bad_masks = [
        r[:3] for r in mask_results
        if r[1] != (512, 512) or r[2] not in ("L", "P")
    ]

    counts = np.sum(
        [r[3] for r in mask_results],
        axis=0
    )

    present = {
        int(v) for v in np.nonzero(counts)[0]
    }

    unexpected = sorted(present - VALID)
    missing_classes = sorted(VALID - present)

    ok = (
        len(images) == expected[split]
        and len(masks) == expected[split]
        and duplicate_images == 0
        and duplicate_masks == 0
        and not unpaired_images
        and not unpaired_masks
        and not bad_images
        and not bad_masks
        and not unexpected
    )

    all_ok &= ok
    stems_by_split[split] = image_set

    labelled_pixels = int(counts[:4].sum())

    class_shares = {}

    if labelled_pixels > 0:
        for i, name in enumerate(
            ["soil", "bedrock", "sand", "bigrock"]
        ):
            class_shares[name] = round(
                100 * counts[i] / labelled_pixels, 2
            )

    ignore_share = (
        100 * counts[255] / counts.sum()
        if counts.sum() else 0
    )

    print(
        f"\n[{split}] "
        f"images={len(images)} masks={len(masks)} "
        f"expected={expected[split]}"
    )

    print(
        f"Duplicates: images={duplicate_images}, masks={duplicate_masks}"
    )

    print(
        f"Unpaired: images={len(unpaired_images)}, "
        f"masks={len(unpaired_masks)}"
    )

    print(
        f"Bad formats: images={len(bad_images)}, "
        f"masks={len(bad_masks)}"
    )

    print(
        f"Unexpected labels={unexpected}, "
        f"classes absent={missing_classes}"
    )

    print(
        f"Ignore pixels={ignore_share:.2f}% | "
        f"Class shares={class_shares}"
    )

    print("STATUS:", "OK" if ok else "PROBLEM", flush=True)

    audit[split] = {
        "images": len(images),
        "masks": len(masks),
        "ok": bool(ok),
        "label_pixel_counts": {
            int(k): int(counts[k]) for k in present
        }
    }


# ============================================================
# 5. CHECK SPLIT OVERLAP
# ============================================================

print("\n=== SPLIT OVERLAP ===")

names = list(stems_by_split)

for i in range(len(names)):
    for j in range(i + 1, len(names)):

        overlap = (
            stems_by_split[names[i]]
            & stems_by_split[names[j]]
        )

        print(
            f"{names[i]} & {names[j]}: {len(overlap)}"
        )

        all_ok &= (len(overlap) == 0)


# ============================================================
# 6. CSV VALIDATION
# ============================================================

def to_stem(value):
    return Path(str(value).replace("\\", "/")).stem


print("\n=== CSV VALIDATION ===")

csv_columns = {}

for name in expected:

    df = pd.read_csv(L / f"{name}.csv")

    print(
        f"\n{name}.csv rows={len(df)} "
        f"expected={expected[name]}"
    )

    print("Columns:", list(df.columns))

    matching_columns = []

    for column in df.columns:

        if df[column].dtype == object:

            stems = df[column].map(to_stem)

            if (
                stems.is_unique
                and set(stems) == stems_by_split[name]
            ):
                matching_columns.append(column)

    csv_ok = (
        len(df) == expected[name]
        and bool(matching_columns)
    )

    print(
        "Exact filename-matching columns:",
        matching_columns,
        "->",
        "OK" if csv_ok else "PROBLEM"
    )

    csv_columns[name] = matching_columns
    all_ok &= csv_ok

    if name == "train":
        print("\nTrain CSV sample:")
        print(df.head(2).to_string())


# ============================================================
# 7. FROZEN SPLIT JSON STRUCTURE VALIDATION
# ============================================================

print("\n=== FROZEN SPLIT JSON VALIDATION ===")

with open(L / "split_v3_FROZEN.json") as f:
    split_json = json.load(f)

candidates = []


def walk_json(obj, path="$"):

    if isinstance(obj, list):

        if obj and all(isinstance(x, str) for x in obj):

            candidates.append((
                path,
                {to_stem(x) for x in obj},
                len(obj)
            ))

        else:

            for index, item in enumerate(obj[:50]):
                walk_json(item, f"{path}[{index}]")

    elif isinstance(obj, dict):

        if (
            len(obj) > 50
            and all(isinstance(k, str) for k in obj)
        ):
            candidates.append((
                path + " (dict keys)",
                {to_stem(k) for k in obj},
                len(obj)
            ))

        for key, value in obj.items():
            walk_json(value, f"{path}.{key}")


walk_json(split_json)

json_ok = True

for name in expected:

    hits = [
        path
        for path, stem_set, count in candidates
        if (
            stem_set == stems_by_split[name]
            and count == expected[name]
        )
    ]

    print(
        f"{name}: exact filename match = "
        f"{hits if hits else 'NOTHING'}"
    )

    json_ok &= bool(hits)


if json_ok:

    print("Frozen split: VERIFIED against image filenames")

else:

    print(
        "Frozen split: UNVERIFIED. "
        "JSON may store indices, metadata or other identifiers."
    )

    print("\nJSON structure:")

    def describe(obj, depth=0):

        if isinstance(obj, dict):

            for key, value in list(obj.items())[:15]:

                description = (
                    f"len={len(value)}"
                    if isinstance(value, (list, dict))
                    else repr(value)[:70]
                )

                print(
                    "  " * depth,
                    f"{key}: {type(value).__name__} {description}"
                )

                if isinstance(value, dict) and depth < 2:
                    describe(value, depth + 1)

    describe(split_json)


# ============================================================
# 8. SAVE AUDIT AND FINAL STATUS
# ============================================================

audit["all_data_csv_checks_passed"] = bool(all_ok)
audit["frozen_split_verified"] = bool(json_ok)

with open(W / "logs" / "dataset_audit_kaggle.json", "w") as f:
    json.dump(audit, f, indent=2)

print("\n" + "=" * 60)

print(
    "DATA + CSV AUDIT:",
    "ALL CHECKS PASSED" if all_ok else "FAILED"
)

print(
    "FROZEN SPLIT JSON:",
    "VERIFIED" if json_ok else "UNVERIFIED - INVESTIGATE"
)

print(
    "OVERALL:",
    "READY FOR PIPELINE VALIDATION"
    if all_ok and json_ok
    else "DO NOT PROCEED UNTIL ISSUES ARE RESOLVED"
)

print("\nSaved files:")
print(W / "logs" / "dataset_audit_kaggle.json")
print(W / "logs" / "dataset_sha256_manifest.json")
print(W / "splits" / "split_v3_FROZEN.json")

print("\nLocal dataset:", L)
print("Audit complete.")


Files to place: 23008

=== COPYING DATASET ===


Copying files:   0%|          | 0/23008 [00:00<?, ?it/s]

Copy results: {'copied': 23008} | Time: 157.4s

=== SHA256 VERIFICATION ===


SHA256:   0%|          | 0/23008 [00:00<?, ?it/s]

SHA256 mismatches: 0 []
SHA256 manifest saved.

=== IMAGE / MASK AUDIT ===


train images:   0%|          | 0/9803 [00:00<?, ?it/s]

... (7 outputs left out)


[test_msl] images=311 masks=311 expected=311
Duplicates: images=0, masks=0
Unpaired: images=0, masks=0
Bad formats: images=0, masks=0
Unexpected labels=[], classes absent=[]
Ignore pixels=40.60% | Class shares={'soil': np.float64(45.8), 'bedrock': np.float64(30.91), 'sand': np.float64(23.1), 'bigrock': np.float64(0.2)}
STATUS: OK


test_mer images:   0%|          | 0/195 [00:00<?, ?it/s]

test_mer masks:   0%|          | 0/195 [00:00<?, ?it/s]


[test_mer] images=195 masks=195 expected=195
Duplicates: images=0, masks=0
Unpaired: images=0, masks=0
Bad formats: images=0, masks=0
Unexpected labels=[], classes absent=[]
Ignore pixels=37.17% | Class shares={'soil': np.float64(49.01), 'bedrock': np.float64(23.21), 'sand': np.float64(26.78), 'bigrock': np.float64(1.01)}
STATUS: OK

=== SPLIT OVERLAP ===
train & val: 0
train & test_msl: 0
train & test_mer: 0
val & test_msl: 0
val & test_mer: 0
test_msl & test_mer: 0

=== CSV VALIDATION ===

train.csv rows=9803 expected=9803
Columns: ['image_path', 'label_path', 'labeled_frac', 'sclk', 'F', 'class_0', 'class_1', 'class_2', 'class_3', 'img_512', 'mask_512']
Exact filename-matching columns: ['image_path', 'label_path', 'img_512', 'mask_512'] -> OK

Train CSV sample:
                                                     image_path                                                      label_path  labeled_frac         sclk     F   class_0  class_1  class_2  class_3                           

In [2]:
import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# CELL 2e — FROZEN SPLIT VERIFICATION
# CPU ONLY
# ============================================================

L = Path("/tmp/mars_data")

with open(L / "split_v3_FROZEN.json", "r") as f:
    frozen = json.load(f)

SOL_SECONDS = frozen["sol_seconds"]
TRAIN_BUFFER = frozen.get("buffer_test_sols", 3)
VAL_BUFFER = frozen.get("buffer_val_sols", 3)

SPLITS = ["train", "val", "test_msl", "test_mer"]

dfs = {
    name: pd.read_csv(L / f"{name}.csv")
    for name in SPLITS
}

# ------------------------------------------------------------
# 1. CSV CHECKSUMS AND ROW COUNTS
# ------------------------------------------------------------

def md5_bytes(data):
    return hashlib.md5(data).hexdigest()

print("=" * 65)
print("1. CSV CHECKSUM AND ROW-COUNT VERIFICATION")
print("=" * 65)

checksum_results = {}
row_results = {}

for name in SPLITS:
    csv_path = L / f"{name}.csv"
    df = dfs[name]

    expected_md5 = frozen["files"][name]["md5"]
    expected_rows = frozen["files"][name]["rows"]

    raw = csv_path.read_bytes()

    candidates = {
        "file_bytes": md5_bytes(raw),
        "crlf_to_lf": md5_bytes(raw.replace(b"\r\n", b"\n")),
        "image_path_ordered": md5_bytes(
            "\n".join(df["image_path"].astype(str)).encode()
        ),
        "image_path_sorted": md5_bytes(
            "\n".join(sorted(df["image_path"].astype(str))).encode()
        ),
    }

    matches = [
        method for method, value in candidates.items()
        if value == expected_md5
    ]

    checksum_results[name] = bool(matches)
    row_results[name] = len(df) == expected_rows

    print(
        f"{name:10s} | "
        f"rows: {len(df):5d}/{expected_rows:<5d} "
        f"{'OK' if row_results[name] else 'MISMATCH'} | "
        f"MD5: {matches if matches else 'NO MATCH'}"
    )

all_rows_ok = all(row_results.values())
all_md5_ok = all(checksum_results.values())

print("\nAll row counts:", "PASS" if all_rows_ok else "FAIL")
print("All MD5 checks:", "PASS" if all_md5_ok else "NOT FULLY MATCHED")


# ------------------------------------------------------------
# 2. FILENAME OVERLAP CHECK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("2. FILENAME OVERLAP CHECK")
print("=" * 65)

filename_sets = {}

for name, df in dfs.items():
    # img_512 paths identify the actual 512x512 images.
    filename_sets[name] = set(
        df["img_512"].dropna().astype(str)
    )

overlap_ok = True

for i, a in enumerate(SPLITS):
    for b in SPLITS[i + 1:]:
        overlap = filename_sets[a] & filename_sets[b]
        ok = len(overlap) == 0
        overlap_ok &= ok

        print(
            f"{a:10s} vs {b:10s}: "
            f"{len(overlap)} overlapping files "
            f"{'OK' if ok else 'OVERLAP FOUND'}"
        )


# ------------------------------------------------------------
# 3. MSL TEMPORAL BUFFER CHECK
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("3. MSL TEMPORAL BUFFER CHECK")
print("=" * 65)

def nearest_gap_sols(reference, query):
    """
    For every query timestamp, calculate distance to its
    nearest timestamp in the reference set.
    """
    reference = np.sort(
        np.asarray(reference, dtype=np.float64)
    )
    query = np.asarray(query, dtype=np.float64)

    reference = reference[np.isfinite(reference)]
    query = query[np.isfinite(query)]

    if len(reference) == 0 or len(query) == 0:
        raise ValueError("Empty or invalid SCLK values.")

    idx = np.searchsorted(reference, query)

    left_idx = np.clip(idx - 1, 0, len(reference) - 1)
    right_idx = np.clip(idx, 0, len(reference) - 1)

    left_gap = np.abs(query - reference[left_idx])
    right_gap = np.abs(query - reference[right_idx])

    return np.minimum(left_gap, right_gap) / SOL_SECONDS


train_sclk = dfs["train"]["sclk"].dropna().values

temporal_results = {}

for name, required_buffer in [
    ("val", VAL_BUFFER),
    ("test_msl", TRAIN_BUFFER),
]:
    query_sclk = dfs[name]["sclk"].dropna().values

    gaps = nearest_gap_sols(train_sclk, query_sclk)

    minimum = float(gaps.min())
    median = float(np.median(gaps))

    ok = minimum >= required_buffer
    temporal_results[name] = ok

    print(
        f"{name:10s} | n={len(gaps):4d} | "
        f"minimum gap={minimum:.3f} sols | "
        f"median gap={median:.2f} sols | "
        f"required >= {required_buffer} | "
        f"{'OK' if ok else 'BUFFER VIOLATION'}"
    )


# ------------------------------------------------------------
# 4. VALIDATION VS MSL TEST SEPARATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("4. VALIDATION VS MSL TEST SEPARATION")
print("=" * 65)

val_sclk = dfs["val"]["sclk"].dropna().values
test_sclk = dfs["test_msl"]["sclk"].dropna().values

val_test_gaps = nearest_gap_sols(val_sclk, test_sclk)

val_test_min = float(val_test_gaps.min())
val_test_median = float(np.median(val_test_gaps))

# Both held-out blocks are separated from train by their
# respective buffers. Their mutual gap is reported separately.
# The frozen JSON does not explicitly state a val-test buffer.
print(f"Minimum val-test gap: {val_test_min:.3f} sols")
print(f"Median val-test gap:  {val_test_median:.3f} sols")
print("Note: the JSON does not specify a separate val-test buffer.")


# ------------------------------------------------------------
# 5. MER INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("5. MER SPLIT INFORMATION")
print("=" * 65)

mer = dfs["test_mer"]

print("MER samples:", len(mer))
print("MER sol column present:", "sol" in mer.columns)
print("MER SCLK column present:", "sclk" in mer.columns)

print(
    "MER is a separate rover test set. "
    "The MSL SCLK temporal-buffer calculation is not applied to it."
)


# ------------------------------------------------------------
# 6. FINAL STATUS — KEEP CHECKS SEPARATE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("6. FINAL VERIFICATION SUMMARY")
print("=" * 65)

print("Row counts:              ", "PASS" if all_rows_ok else "FAIL")
print("CSV MD5:                 ", "PASS" if all_md5_ok else "NOT FULLY MATCHED")
print("Filename disjointness:   ", "PASS" if overlap_ok else "FAIL")
print(
    "Train-val temporal gap:  ",
    "PASS" if temporal_results.get("val", False) else "FAIL"
)
print(
    "Train-test temporal gap: ",
    "PASS" if temporal_results.get("test_msl", False) else "FAIL"
)

# The split is structurally acceptable for proceeding if:
# - row counts are correct
# - no filenames overlap
# - both MSL held-out splits satisfy their train buffers
#
# MD5 is reported independently because it may refer to
# an earlier CSV version.

structural_ok = (
    all_rows_ok
    and overlap_ok
    and all(temporal_results.values())
)

print("\n" + "=" * 65)

if structural_ok:
    print("STRUCTURAL SPLIT CHECK: PASS")
    if all_md5_ok:
        print("FROZEN CSV CHECKSUMS: VERIFIED")
    else:
        print("FROZEN CSV CHECKSUMS: NOT REPRODUCED")
        print("Record checksum discrepancy; do not claim MD5 verification.")
else:
    print("STRUCTURAL SPLIT CHECK: FAIL")
    print("DO NOT PROCEED — investigate the failed checks.")

print("=" * 65)

1. CSV CHECKSUM AND ROW-COUNT VERIFICATION
train      | rows:  9803/9803  OK | MD5: NO MATCH
val        | rows:  1192/1192  OK | MD5: NO MATCH
test_msl   | rows:   311/311   OK | MD5: NO MATCH
test_mer   | rows:   195/195   OK | MD5: NO MATCH

All row counts: PASS
All MD5 checks: NOT FULLY MATCHED

2. FILENAME OVERLAP CHECK
train      vs val       : 0 overlapping files OK
train      vs test_msl  : 0 overlapping files OK
train      vs test_mer  : 0 overlapping files OK
val        vs test_msl  : 0 overlapping files OK
val        vs test_mer  : 0 overlapping files OK
test_msl   vs test_mer  : 0 overlapping files OK

3. MSL TEMPORAL BUFFER CHECK
val        | n=1192 | minimum gap=3.006 sols | median gap=4.99 sols | required >= 3 | OK
test_msl   | n= 311 | minimum gap=3.000 sols | median gap=3.90 sols | required >= 3 | OK

4. VALIDATION VS MSL TEST SEPARATION
Minimum val-test gap: 3.827 sols
Median val-test gap:  22.163 sols
Note: the JSON does not specify a separate val-test buffer.

5. MER

In [3]:
import hashlib
import json
from pathlib import Path

WORK = Path("/tmp/mars_data")
SOURCE = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")

with open(WORK / "split_v3_FROZEN.json", "r") as f:
    frozen = json.load(f)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 65)
print("CSV SOURCE VS WORKING-COPY COMPARISON")
print("=" * 65)

for name in ["train", "val", "test_msl", "test_mer"]:
    print(f"\n{name.upper()}")

    expected = frozen["files"][name]["md5"]
    work_file = WORK / f"{name}.csv"

    # Find the original CSV by filename in the Kaggle input.
    source_matches = list(SOURCE.rglob(f"{name}.csv"))

    print("Expected JSON MD5:", expected)
    print("Working-copy MD5: ", md5(work_file))

    if not source_matches:
        print("Original CSV: NOT FOUND")
        continue

    for source_file in source_matches:
        source_hash = md5(source_file)

        print("Original path:   ", source_file)
        print("Original MD5:    ", source_hash)
        print(
            "Source = copy:   ",
            source_hash == md5(work_file)
        )
        print(
            "Source = JSON:   ",
            source_hash == expected
        )

CSV SOURCE VS WORKING-COPY COMPARISON

TRAIN
Expected JSON MD5: 8d02c9c95be90e4a89663a84c3fd82b7
Working-copy MD5:  1e72ce98a07a23c47697e220518e9c36
Original path:    /kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/dataset_v3_512_val_test_csv/dataset_v3_512/train.csv
Original MD5:     1e72ce98a07a23c47697e220518e9c36
Source = copy:    True
Source = JSON:    False

VAL
Expected JSON MD5: 0a86468c938bf3009b6db117c881c38d
Working-copy MD5:  54a263a86d0c962e513152b6c830d302
Original path:    /kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/dataset_v3_512_val_test_csv/dataset_v3_512/val.csv
Original MD5:     54a263a86d0c962e513152b6c830d302
Source = copy:    True
Source = JSON:    False

TEST_MSL
Expected JSON MD5: 46a63b56c865a1d2c6c8bc13a90bdeaa
Working-copy MD5:  c8417796db62007412fca69021f3059b
Original path:    /kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/dataset_v3_512_val_test_csv/dataset_v3_512/test_msl.csv
Original MD5:     c8417796db62007412fca69021f3059b
So

In [3]:
import json, hashlib, pandas as pd
from pathlib import Path
L = Path("/tmp/mars_data"); W = Path("/kaggle/working/mars_work")
sp = json.load(open(L / "split_v3_FROZEN.json"))
res = {}
for n in ["train", "val", "test_msl", "test_mer"]:
    df = pd.read_csv(L / f"{n}.csv").drop(columns=["img_512", "mask_512"])
    got = hashlib.md5(df.to_csv(index=False).encode()).hexdigest()
    want = sp["files"][n]["md5"]
    res[n] = got == want
    print(f"{n:9s} rows {len(df)}=={sp['files'][n]['rows']} | md5 {'MATCH' if res[n] else 'MISMATCH'}")
ok = all(res.values())
json.dump({"structural_check": "PASS (rows, disjoint filenames, 3-sol buffers: val min 3.006, msl min 3.000)",
           "csv_md5_vs_json": "PASS all 4, after dropping img_512 and mask_512 columns and re-serialising with pandas to_csv(index=False)",
           "mer": "separate rover; sclk buffer not applicable"},
          open(W / "logs" / "frozen_split_verification.json", "w"), indent=2)
print("FROZEN SPLIT:", "FULLY VERIFIED" if ok else "MISMATCH - paste output")

train     rows 9803==9803 | md5 MATCH
val       rows 1192==1192 | md5 MATCH
test_msl  rows 311==311 | md5 MATCH
test_mer  rows 195==195 | md5 MATCH
FROZEN SPLIT: FULLY VERIFIED


In [4]:
import sys, importlib, hashlib, json
from pathlib import Path
import numpy as np

CODE = Path("/kaggle/working/mars_work/code")
CODE.mkdir(parents=True, exist_ok=True)

src = r'''"""mars_common.py - shared code for the Mars compression project (Kaggle version).
Reconstructed from the validated Colab notebook (Mars_Compression_01_SegFormer_Baseline).
Dataset, model, loss and metric logic are unchanged; only the data root differs."""
import os, random
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DATA = os.environ.get("MARS_DATA", "/tmp/mars_data")
CLASS_NAMES = ["soil", "bedrock", "sand", "big rock"]
DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP, AMP_DT = (DEV == "cuda"), torch.float16
MEAN, STD = 0.449, 0.226
PRETRAINED = "nvidia/mit-b0"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


class MarsSeg(Dataset):
    def __init__(self, split, train=False, indices=None):
        self.d = pd.read_csv(f"{DATA}/{split}.csv")
        if indices is not None:
            self.d = self.d.iloc[list(indices)].reset_index(drop=True)
        self.train = train
        required = {"img_512", "mask_512"}
        assert required.issubset(self.d.columns), f"Missing CSV columns: {required - set(self.d.columns)}"

    def __len__(self):
        return len(self.d)

    def __getitem__(self, i):
        r = self.d.iloc[i]
        img_path, mask_path = os.path.join(DATA, r.img_512), os.path.join(DATA, r.mask_512)
        img = np.array(Image.open(img_path).convert("L"), dtype=np.uint8)
        msk = np.array(Image.open(mask_path), dtype=np.uint8)
        assert img.shape == (512, 512), img_path
        assert msk.shape == (512, 512), mask_path
        if self.train and torch.rand(1).item() < 0.5:      # same flip for image and mask
            img = np.fliplr(img); msk = np.fliplr(msk)
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div_(255.0).unsqueeze(0)
        y = torch.from_numpy(np.ascontiguousarray(msk)).long()
        return x, y


def make_loaders(bs=8, seed=42, workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(MarsSeg("train", train=True), batch_size=bs, shuffle=True, drop_last=True,
                              num_workers=workers, pin_memory=True, generator=g, persistent_workers=workers > 0)
    val_loader = DataLoader(MarsSeg("val"), batch_size=16, shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, val_loader


def make_train_eval_idx(n_train=None, k=1000):
    """Same rule as the Colab notebook: RandomState(0).choice(n_train, 1000, replace=False), sorted."""
    n_train = n_train or len(pd.read_csv(f"{DATA}/train.csv"))
    return sorted(np.random.RandomState(0).choice(n_train, min(k, n_train), replace=False).tolist())


class SegWrap(nn.Module):
    """Input: B x 1 x H x W float in [0,1]. Output: B x 4 x H x W logits upsampled to the input size."""
    def __init__(self, pretrained=PRETRAINED):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation, SegformerConfig
        id2label = {i: n for i, n in enumerate(CLASS_NAMES)}; label2id = {n: i for i, n in id2label.items()}
        if pretrained:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                pretrained, num_labels=4, id2label=id2label, label2id=label2id)
        else:
            self.net = SegformerForSemanticSegmentation(
                SegformerConfig(num_labels=4, id2label=id2label, label2id=label2id))

    def forward(self, x):
        h, w = x.shape[-2:]
        x = ((x - MEAN) / STD).repeat(1, 3, 1, 1)
        return F.interpolate(self.net(pixel_values=x).logits, size=(h, w), mode="bilinear", align_corners=False)


def new_model():
    return SegWrap().to(DEV)


def metrics_from_conf(conf, loss=None):
    """conf[i, j] = pixels with true class i predicted as class j (label 255 never enters)."""
    conf = np.asarray(conf, dtype=np.float64); tp = np.diag(conf); gt = conf.sum(1); pr = conf.sum(0); union = gt + pr - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(union > 0, tp / union, np.nan);  rec = np.where(gt > 0, tp / gt, np.nan)
        prec = np.where(pr > 0, tp / pr, np.nan);       dice = np.where(gt + pr > 0, 2 * tp / (gt + pr), np.nan)
    return dict(loss=loss, miou4=float(np.nanmean(iou)), miou3=float(np.nanmean(iou[:3])),
                pixacc=float(tp.sum() / conf.sum()), iou=iou.tolist(), recall=rec.tolist(),
                precision=prec.tolist(), dice=dice.tolist(), conf=conf.astype(np.int64).tolist())


@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval(); conf = torch.zeros(16, dtype=torch.long, device=DEV); loss_sum, n_valid = 0.0, 0
    for b, (x, y) in enumerate(loader):
        if max_batches and b >= max_batches: break
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast(DEV, dtype=AMP_DT, enabled=USE_AMP): logits = model(x)
        logits = logits.float(); v = y != 255
        loss_sum += F.cross_entropy(logits, y, ignore_index=255, reduction="sum").item(); n_valid += int(v.sum())
        conf += torch.bincount(y[v] * 4 + logits.argmax(1)[v], minlength=16)
    return metrics_from_conf(conf.view(4, 4).cpu().numpy(), loss_sum / max(n_valid, 1))


def train_step(model, opt, scaler, x, y):
    model.train(); x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(DEV, dtype=AMP_DT, enabled=USE_AMP): logits = model(x)
    loss = F.cross_entropy(logits.float(), y, ignore_index=255)
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    gn = nn.utils.clip_grad_norm_(model.parameters(), 1e9)     # only MEASURES the norm (no real clipping)
    scaler.step(opt); scaler.update()
    return loss.item(), float(gn)
'''
(CODE / "mars_common.py").write_text(src)
print("written:", CODE / "mars_common.py", "| sha256:", hashlib.sha256(src.encode()).hexdigest()[:16], "...", flush=True)

sys.path.insert(0, str(CODE))
import mars_common as mc
importlib.reload(mc)
import torch
print("DATA root:", mc.DATA, "| device:", mc.DEV, "| AMP:", mc.USE_AMP)

# ---- T0: metric code vs hand-computed toy confusion matrix
toy = [[8, 2, 0, 0], [1, 9, 0, 0], [0, 0, 5, 0], [0, 0, 0, 0]]
m = mc.metrics_from_conf(toy)
assert abs(m["iou"][0] - 8/11) < 1e-9 and abs(m["iou"][1] - 0.75) < 1e-9 and m["iou"][2] == 1.0 and np.isnan(m["iou"][3])
assert abs(m["pixacc"] - 22/25) < 1e-9 and abs(m["miou4"] - (8/11 + 0.75 + 1)/3) < 1e-9
print("T0 metrics: PASS")

# ---- T1: dataset class vs a direct PIL read
from PIL import Image
import pandas as pd
for sp, n in {"train": 9803, "val": 1192, "test_msl": 311, "test_mer": 195}.items():
    ds = mc.MarsSeg(sp); assert len(ds) == n, (sp, len(ds))
ds = mc.MarsSeg("val"); x, y = ds[0]; r = ds.d.iloc[0]
ref_img = np.array(Image.open(f"{mc.DATA}/{r.img_512}").convert("L"))
ref_msk = np.array(Image.open(f"{mc.DATA}/{r.mask_512}"))
assert x.shape == (1, 512, 512) and x.dtype == torch.float32 and 0 <= x.min() and x.max() <= 1
assert y.shape == (512, 512) and y.dtype == torch.int64
assert np.array_equal((x[0].numpy() * 255).round().astype(np.uint8), ref_img) and np.array_equal(y.numpy(), ref_msk)
print(f"T1 dataset: PASS | lengths OK | val[0] x {tuple(x.shape)} range [{x.min():.2f},{x.max():.2f}] | labels {sorted(torch.unique(y).tolist())}")

# ---- T2: fixed 1,000-image training subset (compare md5 with your laptop file)
idx = mc.make_train_eval_idx()
js = json.dumps(idx)
print(f"T2 subset: n={len(idx)} first5={idx[:5]} last3={idx[-3:]} | md5(json.dumps)={hashlib.md5(js.encode()).hexdigest()}")
(CODE.parent / "logs" / "train_eval_subset_idx_regenerated.json").write_text(js)

# ---- T3: model + untrained loss (needs Internet for nvidia/mit-b0)
mc.set_seed(42)
model = mc.new_model()
n_par = sum(p.numel() for p in model.parameters())
_, val_loader = mc.make_loaders()
ev = mc.evaluate(model, val_loader, max_batches=10)
print(f"T3 model: {n_par/1e6:.3f} M params (expected 3.715) | untrained loss on 160 val images: {ev['loss']:.3f} "
      f"(Colab: ~1.383, ln4 = 1.386) | mIoU4 {ev['miou4']:.3f}")

written: /kaggle/working/mars_work/code/mars_common.py | sha256: 73423bc72e01ed7c ...
DATA root: /tmp/mars_data | device: cuda | AMP: True
T0 metrics: PASS
T1 dataset: PASS | lengths OK | val[0] x (1, 512, 512) range [0.06,0.69] | labels [0, 1, 255]
T2 subset: n=1000 first5=[9, 15, 18, 23, 31] last3=[9788, 9798, 9802] | md5(json.dumps)=f9c55dcfcc730290b37c8cfcc95e3bb3


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.bias                             

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.3MB            

model.safetensors: downloading bytes:           |  0.00B            

T3 model: 3.715 M params (expected 3.715) | untrained loss on 160 val images: 1.383 (Colab: ~1.383, ln4 = 1.386) | mIoU4 0.171


In [5]:
import zipfile, hashlib, json, os, time
from pathlib import Path
from IPython.display import FileLink, display

W = Path("/kaggle/working/mars_work")
ZIPNAME = "backup_04_kaggle_setup.zip"
OUT = W / "backups" / ZIPNAME

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()

files = sorted(p for p in W.rglob("*") if p.is_file()
               and "backups" not in p.relative_to(W).parts and "__pycache__" not in p.parts)
entries = [{"path": str(p.relative_to(W)), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in files]
manifest = {"zip_name": ZIPNAME, "created_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "note": "Paths are relative to the project root (mirrors mars-compression on the laptop).",
            "files": entries}

tmp = OUT.parent / (ZIPNAME + ".tmp")
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("MANIFEST.json", json.dumps(manifest, indent=2))
    for p, e in zip(files, entries):
        zf.write(p, e["path"])
os.replace(tmp, OUT)

# ---- verify by re-opening the finished ZIP
with zipfile.ZipFile(OUT) as zf:
    assert zf.testzip() is None, "ZIP CRC error"
    m = json.loads(zf.read("MANIFEST.json"))
    assert set(zf.namelist()) - {"MANIFEST.json"} == {e["path"] for e in m["files"]}, "file list differs from manifest"
    for e in m["files"]:
        assert hashlib.sha256(zf.read(e["path"])).hexdigest() == e["sha256"], e["path"]
print("ZIP VERIFIED: CRC ok, every file matches MANIFEST.json\n")
for e in entries:
    print(f"  {e['path']:<50} {e['bytes']/1024:10.1f} KB")
print(f"\nZIP file : {OUT}")
print(f"ZIP size : {OUT.stat().st_size/1e6:.2f} MB")
print(f"ZIP SHA256 (compare on laptop): {sha256_file(OUT)}")
display(FileLink(str(OUT.relative_to("/kaggle/working"))))

ZIP VERIFIED: CRC ok, every file matches MANIFEST.json

  code/mars_common.py                                       5.7 KB
  logs/dataset_audit_kaggle.json                            0.9 KB
  logs/dataset_sha256_manifest.json                      2792.0 KB
  logs/frozen_split_verification.json                       0.3 KB
  logs/train_eval_subset_idx_regenerated.json               5.8 KB
  splits/split_v3_FROZEN.json                               0.7 KB

ZIP file : /kaggle/working/mars_work/backups/backup_04_kaggle_setup.zip
ZIP size : 1.10 MB
ZIP SHA256 (compare on laptop): e1e9c0e6910cfa6ee14b822e2e2f167c65514cc28e3d7a38115ba7a1e8fc1b6a


/kaggle/working/mars_work/backups/backup_04_kaggle_setup.zip

In [6]:
import zipfile, hashlib, json, shutil, importlib.util
from pathlib import Path

W = Path("/kaggle/working/mars_work")
ZIP = W / "backups" / "backup_04_kaggle_setup.zip"
SCR = Path("/kaggle/working/restore_test")
shutil.rmtree(SCR, ignore_errors=True); SCR.mkdir()

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()

with zipfile.ZipFile(ZIP) as zf:
    assert zf.testzip() is None
    zf.extractall(SCR)
m = json.load(open(SCR / "MANIFEST.json"))
bad_manifest, bad_live = [], []
for e in m["files"]:
    p = SCR / e["path"]
    if not p.exists() or sha(p) != e["sha256"]: bad_manifest.append(e["path"])
    live = W / e["path"]
    if live.exists() and sha(live) != e["sha256"]: bad_live.append(e["path"])
print(f"files in manifest: {len(m['files'])} | restored files failing manifest hash: {len(bad_manifest)} {bad_manifest}")
print(f"restored files that differ from live originals: {len(bad_live)} {bad_live}")

# restored code must import and work
spec = importlib.util.spec_from_file_location("mars_common_restored", SCR / "code" / "mars_common.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
r = mod.metrics_from_conf([[8, 2, 0, 0], [1, 9, 0, 0], [0, 0, 5, 0], [0, 0, 0, 0]])
assert abs(r["pixacc"] - 22/25) < 1e-9
print("restored mars_common.py imports and passes the toy metric check")

ok = not bad_manifest and not bad_live
shutil.rmtree(SCR)
print("\nRESTORE TEST:", "PASS" if ok else "FAIL - paste output")

files in manifest: 6 | restored files failing manifest hash: 0 []
restored files that differ from live originals: 0 []
restored mars_common.py imports and passes the toy metric check

RESTORE TEST: PASS


In [7]:
import sys, importlib, os, time
from pathlib import Path
import numpy as np, torch

CODE = Path("/kaggle/working/mars_work/code")
src = r'''"""ckpt_utils.py - atomic resumable checkpoints, RNG state handling, LR schedule."""
import os, math, json, random, hashlib
import numpy as np, torch

REQUIRED = ["model", "optimizer", "scheduler", "scaler", "epoch", "iteration",
            "rng", "config", "config_hash", "best_metric"]


def config_hash(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"].cpu())
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def lr_factor(it, total, warmup=100, final_frac=0.05):
    """Multiplier on the peak LR after `it` optimiser steps: linear warmup, then cosine decay to final_frac."""
    if it < warmup:
        return (it + 1) / warmup
    p = min(1.0, (it - warmup) / max(1, total - warmup))
    return final_frac + (1 - final_frac) * 0.5 * (1 + math.cos(math.pi * p))


def make_scheduler(opt, total, warmup=100, final_frac=0.05):
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda it: lr_factor(it, total, warmup, final_frac))


def save_checkpoint(path, model, opt, sched, scaler, epoch, iteration, cfg, best_metric, extra=None):
    path = str(path); tmp = path + ".tmp"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state = {"model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
             "scaler": scaler.state_dict(), "epoch": epoch, "iteration": iteration, "rng": get_rng_state(),
             "config": cfg, "config_hash": config_hash(cfg), "best_metric": best_metric, "extra": extra or {}}
    torch.save(state, tmp)
    ck = torch.load(tmp, map_location="cpu", weights_only=False)          # validate BEFORE replacing the old file
    missing = [k for k in REQUIRED if k not in ck]
    assert not missing, f"checkpoint missing keys: {missing}"
    assert ck["iteration"] == iteration and ck["config_hash"] == config_hash(cfg)
    os.replace(tmp, path)
    return os.path.getsize(path)


def load_checkpoint(path, model, opt=None, sched=None, scaler=None, cfg=None, restore_rng=True):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if cfg is not None and config_hash(cfg) != ck["config_hash"]:
        raise ValueError(f"config hash mismatch: checkpoint {ck['config_hash']} vs current {config_hash(cfg)}")
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["optimizer"])
    if sched is not None: sched.load_state_dict(ck["scheduler"])
    if scaler is not None: scaler.load_state_dict(ck["scaler"])
    if restore_rng: set_rng_state(ck["rng"])
    return ck
'''
(CODE / "ckpt_utils.py").write_text(src)
sys.path.insert(0, str(CODE))
import mars_common as mc
import ckpt_utils as cu
importlib.reload(cu)

# ---- LR schedule check (pilot: 2400 iterations, 100 warmup, 5% floor)
tot = 2400
print("LR factor at it 0/49/99/100/1250/2399:", [round(cu.lr_factor(i, tot), 4) for i in (0, 49, 99, 100, 1250, 2399)])
assert abs(cu.lr_factor(0, tot) - 0.01) < 1e-9 and abs(cu.lr_factor(99, tot) - 1.0) < 1e-9
assert abs(cu.lr_factor(100, tot) - 1.0) < 1e-9 and abs(cu.lr_factor(1250, tot) - 0.525) < 1e-3
assert abs(cu.lr_factor(2399, tot) - 0.05) < 1e-3
print("schedule: PASS")

# ---- fixed batches so every run sees identical data
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
idx = mc.make_train_eval_idx()[:160]
ds = mc.MarsSeg("train", train=False, indices=idx)
X = torch.stack([ds[i][0] for i in range(160)]); Y = torch.stack([ds[i][1] for i in range(160)])
batches = [(X[i*8:(i+1)*8], Y[i*8:(i+1)*8]) for i in range(20)]
CFG = dict(test="resume", lr=1e-4, wd=0.01, warmup=10, total=20, seed=42, bs=8)
CK = Path("/kaggle/working/mars_work/checkpoints/_test"); CKP = CK / "test.pt"

def build():
    mc.set_seed(42); model = mc.new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    return model, opt, cu.make_scheduler(opt, CFG["total"], CFG["warmup"]), torch.amp.GradScaler(mc.DEV, enabled=mc.USE_AMP)

def run(model, opt, sched, scaler, a, b, save_at=None):
    out = []
    for it in range(a, b):
        l, _ = mc.train_step(model, opt, scaler, *batches[it]); sched.step(); out.append(l)
        if save_at is not None and it + 1 == save_at:
            sz = cu.save_checkpoint(CKP, model, opt, sched, scaler, 0, it + 1, CFG, best_metric=-1.0)
            print(f"  checkpoint saved at iteration {it+1}: {sz/1e6:.1f} MB", flush=True)
    return out

t0 = time.time()
print("\nRun A (uninterrupted, saves checkpoint at 10)...", flush=True)
m, o, s, sc = build(); A = run(m, o, s, sc, 0, 20, save_at=10)
A_final = {k: v.detach().clone() for k, v in m.state_dict().items()}; del m, o, s, sc

print("Run A2 (repeat of A = GPU noise floor)...", flush=True)
m, o, s, sc = build(); A2 = run(m, o, s, sc, 0, 20); del m, o, s, sc

print("Run B (fresh model, restore checkpoint incl. RNG, continue 10..20)...", flush=True)
m, o, s, sc = build(); ck = cu.load_checkpoint(CKP, m, o, s, sc, cfg=CFG)
print(f"  restored: iteration {ck['iteration']}, config hash {ck['config_hash']}, lr now {o.param_groups[0]['lr']:.3e}")
B = run(m, o, s, sc, ck["iteration"], 20)
B_final = {k: v.detach().clone() for k, v in m.state_dict().items()}; del m, o, s, sc

print("Run C (control: restore checkpoint but NOT the RNG)...", flush=True)
m, o, s, sc = build(); cu.load_checkpoint(CKP, m, o, s, sc, cfg=CFG, restore_rng=False)
C = run(m, o, s, sc, 10, 20); del m, o, s, sc
torch.cuda.empty_cache()

A, A2, B, C = map(np.array, (A, A2, B, C))
noise = np.abs(A - A2).max(); dB = np.abs(A[10:] - B).max(); dC = np.abs(A[10:] - C).max()
wdiff = max(float((A_final[k].float() - B_final[k].float()).abs().max()) for k in A_final)
print(f"\nlosses A[10:12] {A[10:12].round(5)} | B[:2] {B[:2].round(5)}")
print(f"max |loss diff|: A vs A2 (noise floor) {noise:.2e} | A vs B (resume) {dB:.2e} | A vs C (no RNG restore) {dC:.2e}")
print(f"max |weight diff| after step 20, A vs B: {wdiff:.2e}")

# ---- config mismatch must be rejected; no leftover .tmp
m, o, s, sc = build()
try:
    cu.load_checkpoint(CKP, m, o, s, sc, cfg={**CFG, "lr": 3e-4}); rejected = False
except ValueError as e:
    rejected = True; print("config mismatch rejected:", e)
leftover = list(CK.glob("*.tmp"))
del m, o, s, sc; torch.cuda.empty_cache()

resume_ok = dB <= max(10 * noise, 1e-4)
sensitive = dC > 10 * max(noise, 1e-7)
print(f"\nresume matches uninterrupted run (within noise): {resume_ok}")
print(f"control shows the test can detect a missing RNG restore: {sensitive}")
print(f"config mismatch rejected: {rejected} | leftover .tmp files: {len(leftover)}")
import shutil; shutil.rmtree(CK, ignore_errors=True)
print("\nCHECKPOINT TEST:", "PASS" if (resume_ok and rejected and not leftover) else "CHECK - paste output",
      "| sensitivity:", "OK" if sensitive else "WEAK (inconclusive control)", f"| {time.time()-t0:.0f}s")

LR factor at it 0/49/99/100/1250/2399: [0.01, 0.5, 1.0, 1.0, 0.525, 0.05]
schedule: PASS

Run A (uninterrupted, saves checkpoint at 10)...


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.bias                             

  checkpoint saved at iteration 10: 44.9 MB
Run A2 (repeat of A = GPU noise floor)...


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


... (10 outputs left out)


losses A[10:12] [1.22064 1.09595] | B[:2] [1.22064 1.09595]
max |loss diff|: A vs A2 (noise floor) 1.43e-03 | A vs B (resume) 2.36e-05 | A vs C (no RNG restore) 3.26e-02
max |weight diff| after step 20, A vs B: 3.06e-04


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.bias                             

config mismatch rejected: config hash mismatch: checkpoint 514423b92eecda52 vs current a6cc0b0f04e294b3

resume matches uninterrupted run (within noise): True
control shows the test can detect a missing RNG restore: True
config mismatch rejected: True | leftover .tmp files: 0

CHECKPOINT TEST: PASS | sensitivity: OK | 12s


In [8]:
import sys, importlib, shutil, csv
from pathlib import Path
CODE = Path("/kaggle/working/mars_work/code")
src = r'''"""tracking.py - local CSV first, W&B second. W&B problems never stop training.
Only scalar metrics and a small config go to W&B (no code, models, files or console output)."""
import os, csv
from pathlib import Path


class Tracker:
    def __init__(self, csv_prefix, project=None, name=None, group=None, config=None, use_wandb=True):
        self.prefix = Path(csv_prefix); self.prefix.parent.mkdir(parents=True, exist_ok=True)
        self.fields, self.run, self.wb_errors = {}, None, 0
        if use_wandb:
            try:
                os.environ["WANDB_DISABLE_CODE"] = "true"; os.environ["WANDB_CONSOLE"] = "off"
                import wandb
                try:
                    from kaggle_secrets import UserSecretsClient
                    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
                except Exception as e:
                    print("  (Kaggle secret not used:", type(e).__name__, ")")
                self.run = wandb.init(project=project, name=name, group=group, config=config, save_code=False)
            except Exception as e:
                print("W&B unavailable; continuing with local CSV only:", repr(e)[:200]); self.run = None

    def _wb_log(self, data, step):
        self.run.log(data, step=step)

    def log(self, kind, row, step=None):
        path = Path(f"{self.prefix}_{kind}.csv")                       # 1) local CSV, always first
        if kind not in self.fields:
            self.fields[kind] = list(row.keys())
            if not path.exists():
                with open(path, "w", newline="") as f: csv.writer(f).writerow(self.fields[kind])
        with open(path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields[kind], extrasaction="ignore").writerow(row)
        if self.run is not None:                                       # 2) W&B, never allowed to raise
            try:
                self._wb_log({f"{kind}/{k}": v for k, v in row.items() if isinstance(v, (int, float))}, step)
            except Exception as e:
                self.wb_errors += 1
                if self.wb_errors <= 3: print("  W&B log failed (training continues):", repr(e)[:120])

    def finish(self):
        if self.run is not None:
            try: self.run.finish()
            except Exception as e: print("W&B finish failed:", repr(e)[:120])
'''
(CODE / "tracking.py").write_text(src)
sys.path.insert(0, str(CODE))
import tracking as tk; importlib.reload(tk)

TEST = Path("/kaggle/working/mars_work/logs/_tracker_test"); shutil.rmtree(TEST, ignore_errors=True)
t = tk.Tracker(TEST / "run", project="mars-compression", name="setup_test", group="setup_test",
               config={"purpose": "tracker smoke test", "lr": 1e-4})
print("W&B connected:", t.run is not None)
if t.run is not None: print("run URL:", t.run.url)

for it in range(1, 6):
    t.log("train", dict(iteration=it, loss=2.0 / it, lr=1e-4), step=it)
t.log("val", dict(iteration=5, miou4=0.5, miou3=0.6), step=5)

print("\nSimulating a W&B outage for 3 more rows...")
t._wb_log = lambda data, step: (_ for _ in ()).throw(RuntimeError("simulated outage"))
for it in range(6, 9):
    t.log("train", dict(iteration=it, loss=2.0 / it, lr=1e-4), step=it)
t.finish()

n_train = sum(1 for _ in open(TEST / "run_train.csv")) - 1
n_val = sum(1 for _ in open(TEST / "run_val.csv")) - 1
print(f"\nCSV rows: train {n_train} (expected 8) | val {n_val} (expected 1) | W&B errors caught: {t.wb_errors}")
ok = (n_train == 8 and n_val == 1)
print("TRACKER TEST:", "PASS (local CSV complete despite the outage)" if ok else "FAIL - paste output")
shutil.rmtree(TEST, ignore_errors=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hudafatimah-04 (hudafatimah-04-rv-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B connected: True
run URL: https://wandb.ai/hudafatimah-04-rv-college-of-engineering/mars-compression/runs/iqv21t0a

Simulating a W&B outage for 3 more rows...
  W&B log failed (training continues): RuntimeError('simulated outage')
  W&B log failed (training continues): RuntimeError('simulated outage')
  W&B log failed (training continues): RuntimeError('simulated outage')


train/iteration,▁▃▅▆█
train/loss,█▄▂▁▁
train/lr,▁▁▁▁▁
val/iteration,▁
val/miou3,▁
val/miou4,▁
train/iteration,5
train/loss,0.4
train/lr,0.0001
val/iteration,5
val/miou3,0.6



CSV rows: train 8 (expected 8) | val 1 (expected 1) | W&B errors caught: 3
TRACKER TEST: PASS (local CSV complete despite the outage)


In [9]:
import sys, importlib, time, hashlib
from pathlib import Path
import numpy as np, torch
from PIL import Image

CODE = Path("/kaggle/working/mars_work/code")
src = r'''"""mars_det.py - deterministic training data stream (order + flips = pure functions of seed/epoch/index).
MarsSeg in mars_common.py is untouched; pixel loading is delegated to it, so unflipped samples are identical.
Resuming needs only the iteration number. Worker count does not affect the result."""
import hashlib, functools
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
import mars_common as mc


def _h(*parts):
    return int.from_bytes(hashlib.sha256("|".join(map(str, parts)).encode()).digest()[:8], "big")


@functools.lru_cache(maxsize=8)
def epoch_order(seed, epoch, n):
    return np.random.RandomState(_h("order", seed, epoch, n) % (2 ** 32)).permutation(n)


def flip_flag(seed, epoch, idx):
    return _h("flip", seed, epoch, int(idx)) & 1


def sample_at(seed, bs, n, it, r):
    """(epoch, dataset index, flip flag) of sample r (0..bs-1) in the batch at iteration `it`."""
    epoch, pos = divmod(it, n // bs)                 # drop_last: the final n % bs images of an epoch are unused
    idx = int(epoch_order(seed, epoch, n)[pos * bs + r])
    return epoch, idx, flip_flag(seed, epoch, idx)


def batch_plan(seed, bs, n, it):
    rows = [sample_at(seed, bs, n, it, r) for r in range(bs)]
    return rows[0][0], [r[1] for r in rows], [r[2] for r in rows]


def plan_hash(seed, bs, n, n_iters):
    h = hashlib.sha256()
    for it in range(n_iters):
        h.update(repr(batch_plan(seed, bs, n, it)).encode())
    return h.hexdigest()


class DetTrainDataset(Dataset):
    """Item j is sample (j % bs) of the batch at iteration start_iter + j // bs. Use batch_size=bs, shuffle=False."""
    def __init__(self, seed, bs, start_iter, end_iter, flip=True, split="train"):
        self.base = mc.MarsSeg(split, train=False)
        self.n, self.seed, self.bs, self.flip = len(self.base), seed, bs, flip
        self.k0, self.k1 = start_iter * bs, end_iter * bs

    def __len__(self):
        return self.k1 - self.k0

    def load(self, idx, do_flip):
        x, y = self.base[idx]
        if do_flip:
            x, y = torch.flip(x, dims=[-1]), torch.flip(y, dims=[-1])
        return x, y

    def __getitem__(self, j):
        it, r = divmod(self.k0 + j, self.bs)
        _, idx, fl = sample_at(self.seed, self.bs, self.n, it, r)
        return self.load(idx, bool(fl) and self.flip)


def make_det_loader(seed, bs, start_iter, end_iter, workers=2, flip=True):
    ds = DetTrainDataset(seed, bs, start_iter, end_iter, flip)
    return DataLoader(ds, batch_size=bs, shuffle=False, drop_last=False, num_workers=workers,
                      pin_memory=True, persistent_workers=workers > 0)
'''
(CODE / "mars_det.py").write_text(src)
sys.path.insert(0, str(CODE))
import mars_common as mc
import mars_det as md
importlib.reload(md)

SEED, BS = 42, 8
base = mc.MarsSeg("train"); N = len(base); SPE = N // BS
print(f"train images {N} | batches per epoch {SPE} | unused per epoch (drop_last) {N - SPE*BS}")
res = {}

# ---- T1: flip disabled -> identical tensors to the existing MarsSeg
ld = md.make_det_loader(SEED, BS, 0, 5, workers=2, flip=False)
ok = True
for it, (xb, yb) in enumerate(ld):
    _, idx, _ = md.batch_plan(SEED, BS, N, it)
    ex = torch.stack([base[i][0] for i in idx]); ey = torch.stack([base[i][1] for i in idx])
    ok &= torch.equal(xb, ex) and torch.equal(yb, ey) and xb.dtype == ex.dtype and yb.dtype == ey.dtype
res["T1 no-flip output identical to MarsSeg (5 batches, 40 images)"] = ok

# ---- T2: flips applied to image AND mask, checked against the original np.fliplr semantics
ld = md.make_det_loader(SEED, BS, 0, 5, workers=2, flip=True)
ok, n_fl, n_nofl = True, 0, 0
for it, (xb, yb) in enumerate(ld):
    _, idx, fl = md.batch_plan(SEED, BS, N, it)
    for j, (i, f) in enumerate(zip(idx, fl)):
        r = base.d.iloc[i]
        img = np.array(Image.open(f"{mc.DATA}/{r.img_512}").convert("L"), dtype=np.uint8)
        msk = np.array(Image.open(f"{mc.DATA}/{r.mask_512}"), dtype=np.uint8)
        if f: img, msk = np.fliplr(img), np.fliplr(msk)
        ex = torch.from_numpy(np.ascontiguousarray(img)).float().div_(255.0).unsqueeze(0)
        ey = torch.from_numpy(np.ascontiguousarray(msk)).long()
        ok &= torch.equal(xb[j], ex) and torch.equal(yb[j], ey)
        n_fl += f; n_nofl += 1 - f
res[f"T2 flipped/unflipped samples match original np.fliplr logic ({n_fl} flipped, {n_nofl} not)"] = ok and n_fl > 0 and n_nofl > 0

# ---- T3: order properties (plans only, no image loading)
ep0 = [i for it in range(SPE) for i in md.batch_plan(SEED, BS, N, it)[1]]
ep1 = [i for it in range(SPE, 2 * SPE) for i in md.batch_plan(SEED, BS, N, it)[1]]
other = [i for it in range(3) for i in md.batch_plan(SEED + 1, BS, N, it)[1]]
res["T3 each epoch uses every sample at most once (no duplicates, valid range)"] = (
    len(ep0) == len(set(ep0)) == SPE * BS and len(ep1) == len(set(ep1)) == SPE * BS and 0 <= min(ep0) and max(ep0) < N)
res["T3 epoch 0 and epoch 1 orders differ; different seed differs"] = (ep0[:50] != ep1[:50]) and (ep0[:24] != other)

# ---- T4: same result regardless of worker count and of rebuilding the loader
def grab(workers, a=0, b=3):
    return [(x.clone(), y.clone()) for x, y in md.make_det_loader(SEED, BS, a, b, workers=workers, flip=True)]
g0, g2, g4, g2b = grab(0), grab(2), grab(4), grab(2)
res["T4 identical batches with 0, 2 and 4 workers and across rebuilds"] = all(
    torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]) for g in (g2, g4, g2b) for a, b in zip(g0, g))

# ---- T5: resume across the epoch boundary needs only the iteration number
full = grab(2, SPE - 2, SPE + 3)            # iterations 1223..1227 (epoch changes at 1225)
tail = grab(2, SPE + 1, SPE + 3)            # resume at iteration 1226
epochs = [md.batch_plan(SEED, BS, N, it)[0] for it in range(SPE - 2, SPE + 3)]
res[f"T5 resume at iteration {SPE+1} equals the uninterrupted stream (epochs seen {epochs})"] = (
    all(torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]) for a, b in zip(full[-2:], tail)) and epochs == [0, 0, 1, 1, 1])

# ---- T6: flip statistics
f0 = np.array([md.flip_flag(SEED, 0, i) for i in range(N)]); f1 = np.array([md.flip_flag(SEED, 1, i) for i in range(N)])
print(f"flip rate epoch0 {f0.mean():.3f} | epoch1 {f1.mean():.3f} | same image flipped differently in epochs 0 and 1: {(f0 != f1).mean():.3f}")
res["T6 flip rate about 50%, and flips change between epochs"] = (0.48 < f0.mean() < 0.52 and 0.48 < f1.mean() < 0.52 and 0.45 < (f0 != f1).mean() < 0.55)

# ---- T7: one plan hash that all three LR candidates will share
H1 = md.plan_hash(SEED, BS, N, 2400); H2 = md.plan_hash(SEED, BS, N, 2400)
res["T7 plan hash reproducible"] = H1 == H2
print("\nPLAN HASH (seed 42, bs 8, 2400 iterations, order+flips):", H1)

print("\n=== RESULTS ===")
for k, v in res.items(): print(("PASS  " if v else "FAIL  ") + k)

# ---- loader speed (informational, for sizing the pilot)
for w in (2, 4):
    t0 = time.time(); n = 0
    for _ in md.make_det_loader(SEED, BS, 100, 140, workers=w, flip=True): n += 1
    print(f"loader speed, {w} workers: {(time.time()-t0)/n:.3f} s/batch (includes worker start-up)")
print("\nDETERMINISTIC DATA TEST:", "ALL PASS" if all(res.values()) else "FAILED - paste output")

train images 9803 | batches per epoch 1225 | unused per epoch (drop_last) 3
flip rate epoch0 0.501 | epoch1 0.500 | same image flipped differently in epochs 0 and 1: 0.497

PLAN HASH (seed 42, bs 8, 2400 iterations, order+flips): 6ce909b7b5115a9e8299316f42e03ca583f6ee91bc0a9bc6928e3ce6599c56ca

=== RESULTS ===
PASS  T1 no-flip output identical to MarsSeg (5 batches, 40 images)
PASS  T2 flipped/unflipped samples match original np.fliplr logic (19 flipped, 21 not)
PASS  T3 each epoch uses every sample at most once (no duplicates, valid range)
PASS  T3 epoch 0 and epoch 1 orders differ; different seed differs
PASS  T4 identical batches with 0, 2 and 4 workers and across rebuilds
PASS  T5 resume at iteration 1226 equals the uninterrupted stream (epochs seen [0, 0, 1, 1, 1])
PASS  T6 flip rate about 50%, and flips change between epochs
PASS  T7 plan hash reproducible
loader speed, 2 workers: 0.049 s/batch (includes worker start-up)
loader speed, 4 workers: 0.039 s/batch (includes worker sta

In [10]:
import sys, importlib, json
from pathlib import Path
CODE = Path("/kaggle/working/mars_work/code")
W = Path("/kaggle/working/mars_work")

src = r'''"""recovery.py - build, verify and offer a recovery ZIP. Nothing is written outside /kaggle/working/mars_work."""
import zipfile, hashlib, json, os, time
from pathlib import Path

W = Path("/kaggle/working/mars_work")


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()


def make_backup(zipname, include_checkpoints=True, laptop_dest="backups"):
    from IPython.display import FileLink, display
    out = W / "backups" / zipname
    if out.exists():
        raise FileExistsError(f"{zipname} already exists - choose a new name (never overwrite a backup)")
    files = []
    for p in sorted(W.rglob("*")):
        rel = p.relative_to(W)
        if not p.is_file() or "backups" in rel.parts or "__pycache__" in p.parts or p.suffix == ".tmp":
            continue
        if rel.parts[0] == "checkpoints" and not include_checkpoints:
            continue
        files.append(p)
    entries = [{"path": str(p.relative_to(W)), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in files]
    manifest = {"zip_name": zipname, "created_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "note": "Paths are relative to the project root (mirrors mars-compression on the laptop).",
                "files": entries}
    tmp = out.parent / (zipname + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("MANIFEST.json", json.dumps(manifest, indent=2))
        for p, e in zip(files, entries): zf.write(p, e["path"])
    os.replace(tmp, out)
    with zipfile.ZipFile(out) as zf:                                   # verify the finished ZIP
        assert zf.testzip() is None, "ZIP CRC error"
        m = json.loads(zf.read("MANIFEST.json"))
        assert set(zf.namelist()) - {"MANIFEST.json"} == {e["path"] for e in m["files"]}
        for e in m["files"]:
            assert hashlib.sha256(zf.read(e["path"])).hexdigest() == e["sha256"], e["path"]
    digest = sha256_file(out)
    print("ZIP VERIFIED (CRC ok, every file matches MANIFEST.json)\n")
    for e in entries: print(f"  {e['path']:<52} {e['bytes']/1024:10.1f} KB")
    print(f"\nZIP name   : {zipname}\nZIP size   : {out.stat().st_size/1e6:.2f} MB\nZIP SHA256 : {digest}")
    print("\n=== WHAT TO DO NEXT (on your laptop) ===")
    print(f"1. Download the ZIP with the link below (or Output panel > mars_work > backups > {zipname} > Download).")
    print(f"2. Move it to:  C:\\Users\\hudaf\\Downloads\\mars-compression\\{laptop_dest}\\")
    print("3. Open PowerShell and run:")
    print(f"   Get-FileHash C:\\Users\\hudaf\\Downloads\\mars-compression\\{laptop_dest}\\{zipname} -Algorithm SHA256")
    print("4. The Hash printed must equal the ZIP SHA256 above (letters can be upper or lower case).")
    print("5. Reply with MATCH or MISMATCH. Do not start the next step before that.")
    display(FileLink(str(out.relative_to("/kaggle/working"))))
    return digest
'''
(CODE / "recovery.py").write_text(src)

(W / "logs" / "det_data_validation.json").write_text(json.dumps({
    "tests": "T1-T7 all PASS (no-flip identical to MarsSeg; fliplr parity; no duplicates; worker-independent; resume across epoch boundary; flip rate ~0.5)",
    "plan_hash_seed42_bs8_2400it": "6ce909b7b5115a9e8299316f42e03ca583f6ee91bc0a9bc6928e3ce6599c56ca",
    "checkpoint_resume_test": "PASS (resume vs uninterrupted loss diff 2.4e-5; GPU noise floor 1.4e-3; RNG-restore control 3.3e-2)",
    "tracker_test": "PASS (CSV complete during simulated W&B outage)",
    "train_eval_subset_md5": "f9c55dcfcc730290b37c8cfcc95e3bb3 (matches laptop original)"}, indent=2))

sys.path.insert(0, str(CODE))
import recovery as rc; importlib.reload(rc)
rc.make_backup("backup_05_kaggle_code_and_tests.zip", include_checkpoints=True)

ZIP VERIFIED (CRC ok, every file matches MANIFEST.json)

  code/ckpt_utils.py                                          2.9 KB
  code/mars_common.py                                         5.7 KB
  code/mars_det.py                                            2.6 KB
  code/recovery.py                                            2.9 KB
  code/tracking.py                                            2.3 KB
  logs/dataset_audit_kaggle.json                              0.9 KB
  logs/dataset_sha256_manifest.json                        2792.0 KB
  logs/det_data_validation.json                               0.5 KB
  logs/frozen_split_verification.json                         0.3 KB
  logs/train_eval_subset_idx_regenerated.json                 5.8 KB
  splits/split_v3_FROZEN.json                                 0.7 KB

ZIP name   : backup_05_kaggle_code_and_tests.zip
ZIP size   : 1.10 MB
ZIP SHA256 : 57c4c10769cf7efbb15337cb74db2ef24356dff832e6df3c761ea9a9aeb4e967

=== WHAT TO DO NEXT (on your lapto

/kaggle/working/mars_work/backups/backup_05_kaggle_code_and_tests.zip

'57c4c10769cf7efbb15337cb74db2ef24356dff832e6df3c761ea9a9aeb4e967'

In [12]:
import sys, time, json, gc
from pathlib import Path
import numpy as np, torch
from torch.utils.data import DataLoader

CODE = Path("/kaggle/working/mars_work/code")
need = [CODE / "mars_common.py", CODE / "mars_det.py", CODE / "ckpt_utils.py", Path("/tmp/mars_data/train.csv")]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"Session was reset - these are missing: {missing}. Tell Claude; do not continue."
sys.path.insert(0, str(CODE))
import mars_common as mc, mars_det as md, ckpt_utils as cu
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

torch.cuda.set_device(0)
print("GPU 0:", torch.cuda.get_device_name(0), "| using ONE GPU only | AMP:", mc.USE_AMP)
SEED, BS, TOTAL, WARM, TIMED = 42, 8, 2400, 5, 50
results, last_model = {}, None

def time_config(label, deterministic, benchmark):
    global last_model
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = deterministic, benchmark
    t0 = time.perf_counter()
    mc.set_seed(SEED); model = mc.new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    sched = cu.make_scheduler(opt, TOTAL); scaler = torch.amp.GradScaler(mc.DEV, enabled=mc.USE_AMP)
    t_build = time.perf_counter() - t0
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    loader = md.make_det_loader(SEED, BS, 0, WARM + TIMED, workers=2, flip=True)
    t_data, t_step, losses = [], [], []
    t_prev = time.perf_counter()
    for it, (x, y) in enumerate(loader):
        t_got = time.perf_counter()
        loss, gn = mc.train_step(model, opt, scaler, x, y); sched.step()
        torch.cuda.synchronize(); t_done = time.perf_counter()
        losses.append(loss)
        if it >= WARM:
            t_data.append(t_got - t_prev); t_step.append(t_done - t_got)
        t_prev = time.perf_counter()
    tot = np.array(t_data) + np.array(t_step)
    r = dict(deterministic=deterministic, benchmark=benchmark, build_s=t_build,
             iter_mean=float(tot.mean()), iter_median=float(np.median(tot)), iter_p95=float(np.percentile(tot, 95)),
             data_wait_mean=float(np.mean(t_data)), compute_mean=float(np.mean(t_step)),
             peak_alloc_gb=torch.cuda.max_memory_allocated() / 2**30, peak_reserved_gb=torch.cuda.max_memory_reserved() / 2**30,
             loss_first=float(losses[0]), loss_last=float(losses[-1]), all_finite=bool(np.isfinite(losses).all()))
    print(f"\n[{label}] deterministic={deterministic} benchmark={benchmark}")
    print(f"  model build            : {t_build:.1f} s")
    print(f"  per iteration          : mean {r['iter_mean']:.3f} s | median {r['iter_median']:.3f} s | 95th pct {r['iter_p95']:.3f} s")
    print(f"    of which data wait   : {r['data_wait_mean']:.3f} s ({100*r['data_wait_mean']/r['iter_mean']:.1f}%) | compute {r['compute_mean']:.3f} s")
    print(f"  GPU memory peak        : allocated {r['peak_alloc_gb']:.2f} GB | reserved {r['peak_reserved_gb']:.2f} GB (of {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB)")
    print(f"  loss {r['loss_first']:.3f} -> {r['loss_last']:.3f} | all finite: {r['all_finite']}", flush=True)
    results[label] = r
    if last_model is not None: del last_model
    last_model = model
    del opt, sched, scaler, loader; gc.collect(); torch.cuda.empty_cache()

time_config("A", deterministic=True,  benchmark=False)   # what the checkpoint test used
time_config("B", deterministic=False, benchmark=True)    # default-fast setting

# ---- one full validation pass (1192 images, bs 16, validation data only)
val_loader = DataLoader(mc.MarsSeg("val"), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
ev = mc.evaluate(last_model, val_loader)
torch.cuda.synchronize(); t_eval = time.perf_counter() - t0
eval_peak = torch.cuda.max_memory_allocated() / 2**30
print(f"\nFULL VALIDATION PASS: {t_eval:.1f} s | peak memory {eval_peak:.2f} GB "
      f"(model after only 55 steps: mIoU4 {ev['miou4']:.3f}, meaningless here)")
del last_model, val_loader; gc.collect(); torch.cuda.empty_cache()

# ---- projections
print("\n================ PROJECTIONS ================")
proj = {}
for k, r in results.items():
    per = TOTAL * r["iter_mean"] + 4 * t_eval + r["build_s"] + 30
    off = 24500 * r["iter_mean"] + 20 * (t_eval * (1 + 1000 / 1192)) + 60
    proj[k] = dict(per_candidate_min=per / 60, three_candidates_h=3 * per / 3600, official_20_epochs_h=off / 3600)
    print(f"setting {k}: one pilot candidate ~ {per/60:5.1f} min | all three ~ {3*per/3600:4.2f} h "
          f"| (info only) 20-epoch official run ~ {off/3600:4.1f} h")
print("Includes 4 full validation passes per candidate, model build and ~30 s overhead. W&B/CSV logging adds little.")
json.dump(dict(settings=results, full_val_pass_s=t_eval, val_pass_peak_gb=eval_peak, projections=proj,
               note="timing only; model discarded; no test data used"),
          open("/kaggle/working/mars_work/logs/gpu_timing.json", "w"), indent=2)
print("Saved: /kaggle/working/mars_work/logs/gpu_timing.json")

GPU 0: Tesla T4 | using ONE GPU only | AMP: True

[A] deterministic=True benchmark=False
  model build            : 0.3 s
  per iteration          : mean 0.150 s | median 0.150 s | 95th pct 0.154 s
    of which data wait   : 0.000 s (0.1%) | compute 0.150 s
  GPU memory peak        : allocated 2.26 GB | reserved 2.79 GB (of 14.6 GB)
  loss 1.401 -> 0.788 | all finite: True

[B] deterministic=False benchmark=True
  model build            : 0.3 s
  per iteration          : mean 0.152 s | median 0.153 s | 95th pct 0.156 s
    of which data wait   : 0.000 s (0.1%) | compute 0.152 s
  GPU memory peak        : allocated 2.58 GB | reserved 2.87 GB (of 14.6 GB)
  loss 1.401 -> 0.788 | all finite: True

FULL VALIDATION PASS: 9.9 s | peak memory 2.94 GB (model after only 55 steps: mIoU4 0.567, meaningless here)

================ PROJECTIONS ================
setting A: one pilot candidate ~   7.2 min | all three ~ 0.36 h | (info only) 20-epoch official run ~  1.1 h
setting B: one pilot candidate 

In [11]:
from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Dataset exists:", Path("/tmp/mars_data/train.csv").exists())
print("Code exists:", Path("/kaggle/working/mars_work/code/mars_common.py").exists())
print("Checkpoint utilities exist:", Path("/kaggle/working/mars_work/code/ckpt_utils.py").exists())

GPU available: True
GPU count: 2
Dataset exists: True
Code exists: True
Checkpoint utilities exist: True


In [14]:
import os, sys, json, time, shutil, hashlib, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch

CODE = Path("/kaggle/working/mars_work/code"); W = Path("/kaggle/working/mars_work")
need = [CODE/"mars_common.py", CODE/"mars_det.py", CODE/"ckpt_utils.py", CODE/"tracking.py", Path("/tmp/mars_data/train.csv")]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"Session was reset - missing: {missing}. Tell Claude; do not continue."
os.environ["WANDB_DIR"] = "/tmp/wandb"; Path("/tmp/wandb").mkdir(exist_ok=True)

src = r'''"""lr_pilot.py - one LR-pilot candidate. Trains on the train split, evaluates on the VALIDATION split only."""
import os, json, time, csv, math, hashlib, gc
from pathlib import Path
import numpy as np, torch
from torch.utils.data import DataLoader
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
import mars_common as mc, mars_det as md, ckpt_utils as cu, tracking as tk

CN = ["soil", "bedrock", "sand", "bigrock"]
MIOU3_TOL = 1e-9    # float tolerance: 0.80 - 0.75 evaluates to 0.05000000000000004 and must NOT count as a drop above 0.05


def miou3_rejections(evals, pairs, threshold):
    """evals: {iteration: validation mIoU3}. pairs: [[earlier_iter, later_iter], ...] named in the protocol.
    Returns one record per pair whose mIoU3 fell by strictly more than `threshold`. Pairs not listed are never checked."""
    out = []
    for a, b in pairs:
        if a in evals and b in evals:
            drop = evals[a] - evals[b]
            if drop > threshold + MIOU3_TOL:
                out.append(dict(rule="miou3_drop", pair=[a, b], miou3_from=evals[a], miou3_to=evals[b],
                                drop=drop, threshold=threshold))
    return out


def state_hash(sd):
    h = hashlib.sha256()
    for k in sorted(sd):
        t = sd[k].detach().cpu().contiguous()
        h.update(k.encode()); h.update(str(t.dtype).encode()); h.update(str(tuple(t.shape)).encode())
        h.update(t.numpy().tobytes())
    return h.hexdigest()


def truncate_csv(path, it_max):
    path = Path(path)
    if not path.exists(): return 0
    rows = list(csv.reader(open(path))); head, body = rows[0], rows[1:]
    k = head.index("iteration")
    keep = [r for r in body if int(float(r[k])) <= it_max]
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(head); w.writerows(keep)
    return len(body) - len(keep)


def tag_for(lr):
    return f"lr{lr:.0e}"


def run_candidate(cfg, lr, tag, root, init_path, use_wandb=True, wandb_group="lr_pilot",
                  eval_max_batches=None, log_every=25, ckpt_every=0, stop_after=None, resume=True):
    root = Path(root)
    seed, bs, total = cfg["seed"], cfg["batch_size"], cfg["total_iters"]
    eval_iters = set(cfg["eval_iters"]); ccfg = {**cfg, "lr": lr}; chash = cu.config_hash(ccfg)
    pairs, thr = cfg["reject_miou3_pairs"], cfg["reject_miou3_drop"]
    torch.backends.cudnn.deterministic = cfg["cudnn_deterministic"]; torch.backends.cudnn.benchmark = cfg["cudnn_benchmark"]
    n = len(mc.MarsSeg("train")); spe = n // bs
    assert spe == cfg["steps_per_epoch"], "steps per epoch differs from protocol"
    assert md.plan_hash(seed, bs, n, total) == cfg["plan_hash"], "data-stream plan differs from the protocol"

    init = torch.load(init_path, map_location="cpu", weights_only=True)
    assert init["state_hash"] == cfg["init_state_hash"], "initial-weights file differs from the protocol"
    mc.set_seed(seed); model = mc.new_model(); model.load_state_dict(init["state_dict"])
    assert state_hash(model.state_dict()) == cfg["init_state_hash"], "model did not load the shared initial weights"
    mc.set_seed(seed)                                   # identical dropout stream for every candidate

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg["weight_decay"])
    sched = cu.make_scheduler(opt, total, cfg["warmup"], cfg["final_frac"])
    scaler = torch.amp.GradScaler(mc.DEV, enabled=mc.USE_AMP)

    prefix = root / "logs" / "lr_pilot" / tag; res_dir = root / "results" / "lr_pilot"
    ck_path = root / "checkpoints" / "lr_pilot" / tag / "last.pt"
    prefix.parent.mkdir(parents=True, exist_ok=True); res_dir.mkdir(parents=True, exist_ok=True)
    it0, best, skipped, resumed_from, truncated = 0, -1.0, 0, None, 0
    if resume and ck_path.exists():
        ck = cu.load_checkpoint(ck_path, model, opt, sched, scaler, cfg=ccfg)
        it0, best, skipped = ck["iteration"], ck["best_metric"], ck["extra"].get("skipped", 0)
        truncated = sum(truncate_csv(f"{prefix}_{k}.csv", it0) for k in ("train", "val"))
        resumed_from = it0
        print(f"[{tag}] RESUMED from iteration {it0} (removed {truncated} CSV rows newer than the checkpoint)")
    if it0 == 0:                                        # fresh start: stale CSVs from a crashed earlier attempt must not survive
        for k in ("train", "val"): Path(f"{prefix}_{k}.csv").unlink(missing_ok=True)

    evals = {}                                          # {iteration: val mIoU3}, rebuilt from the (truncated) CSV on resume
    vp0 = Path(f"{prefix}_val.csv")
    if vp0.exists():
        for r in csv.DictReader(open(vp0)): evals[int(float(r["iteration"]))] = float(r["miou3"])
    n_reported = len(miou3_rejections(evals, pairs, thr)); fatal = []
    print(f"[{tag}] lr={lr:g} | config hash {chash} | init hash {cfg['init_state_hash'][:12]} | plan hash {cfg['plan_hash'][:12]} | start iteration {it0}", flush=True)

    tr = tk.Tracker(prefix, project="mars-compression", name=tag, group=wandb_group, config=ccfg, use_wandb=use_wandb)
    loader = md.make_det_loader(seed, bs, it0, total, workers=cfg["workers"], flip=True)
    val_loader = DataLoader(mc.MarsSeg("val"), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

    def save(it):
        cu.save_checkpoint(ck_path, model, opt, sched, scaler, it // spe, it, ccfg, best, extra={"skipped": skipped, "tag": tag})

    status, it, win_loss, win_gn, t0 = "completed", it0, [], [], time.time()
    for x, y in loader:
        lr_used = opt.param_groups[0]["lr"]
        loss, gn = mc.train_step(model, opt, scaler, x, y); sched.step(); it += 1
        if not math.isfinite(loss):
            status = "REJECT_NONFINITE_TRAIN_LOSS"; fatal.append(dict(rule="non-finite training loss", iteration=it))
            tr.log("train", dict(iteration=it, epoch=it / spe, loss=float("nan"), lr=lr_used, grad_norm=float("nan"),
                                 skipped_steps_total=skipped, elapsed_s=time.time() - t0), step=it)
            break
        win_loss.append(loss)
        if math.isfinite(gn): win_gn.append(gn)
        else: skipped += 1                              # fp16 overflow: GradScaler skipped this optimiser step
        if it % log_every == 0:
            tr.log("train", dict(iteration=it, epoch=it / spe, loss=float(np.mean(win_loss)), lr=lr_used,
                                 grad_norm=float(np.mean(win_gn)) if win_gn else float("nan"),
                                 skipped_steps_total=skipped, elapsed_s=time.time() - t0), step=it)
            win_loss, win_gn = [], []
        if it in eval_iters:
            te = time.time(); ev = mc.evaluate(model, val_loader, max_batches=eval_max_batches)
            row = dict(iteration=it, epoch=it / spe, val_loss=ev["loss"], miou4=ev["miou4"], miou3=ev["miou3"], pixacc=ev["pixacc"])
            for k, nm in enumerate(CN):
                row[f"iou_{nm}"], row[f"recall_{nm}"], row[f"precision_{nm}"] = ev["iou"][k], ev["recall"][k], ev["precision"][k]
            row["eval_s"] = time.time() - te
            tr.log("val", row, step=it)
            json.dump({"tag": tag, "lr": lr, "iteration": it, **ev}, open(res_dir / f"{tag}_val_it{it}.json", "w"), indent=1)
            best = max(best, ev["miou4"]); save(it)
            evals[it] = ev["miou3"]
            now = miou3_rejections(evals, pairs, thr)
            for r in now[n_reported:]:
                print(f"[{tag}] REJECTION RULE TRIGGERED: val mIoU3 fell {r['miou3_from']:.4f} -> {r['miou3_to']:.4f} "
                      f"(drop {r['drop']:.4f} > {thr}) between iterations {r['pair'][0]} and {r['pair'][1]}. "
                      f"Candidate is marked rejected; training continues so the full curve is recorded.", flush=True)
            n_reported = len(now)
            if not math.isfinite(ev["loss"]):
                status = "REJECT_NONFINITE_VAL_LOSS"; fatal.append(dict(rule="non-finite validation loss", iteration=it)); break
        elif ckpt_every and it % ckpt_every == 0:
            save(it)
        if stop_after is not None and it >= stop_after:
            status = "STOPPED_EARLY"; break
    tr.finish()
    vp = Path(f"{prefix}_val.csv")
    val_rows = list(csv.DictReader(open(vp))) if vp.exists() else []
    reasons = fatal + miou3_rejections(evals, pairs, thr)
    summary = dict(tag=tag, lr=lr, status=status, rejected=bool(reasons), reject_reasons=reasons,
                   miou3_rule=dict(pairs=pairs, threshold=thr, strictly_greater=True, tolerance=MIOU3_TOL),
                   iterations_done=it, config_hash=chash, plan_hash=cfg["plan_hash"],
                   init_state_hash=cfg["init_state_hash"], resumed_from=resumed_from, csv_rows_truncated=truncated,
                   skipped_steps_total=skipped, wandb_connected=tr.run is not None, wandb_errors=tr.wb_errors,
                   train_time_s=time.time() - t0, val=val_rows)
    json.dump(summary, open(res_dir / f"{tag}_summary.json", "w"), indent=1)
    del loader, val_loader, model, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()
    return summary
'''
(CODE / "lr_pilot.py").write_text(src)
sys.path.insert(0, str(CODE))
import mars_common as mc, mars_det as md, ckpt_utils as cu, tracking as tk
import lr_pilot as lp; importlib.reload(lp)
import transformers

# ---------- 1. shared initial weights (never overwritten)
INIT = W / "checkpoints" / "lr_pilot" / "init_seed42.pt"; INIT.parent.mkdir(parents=True, exist_ok=True)
if INIT.exists():
    obj = torch.load(INIT, map_location="cpu", weights_only=True)
    assert lp.state_hash(obj["state_dict"]) == obj["state_hash"]; init_hash = obj["state_hash"]
    print("init file already exists - kept, not overwritten")
else:
    mc.set_seed(42); m = mc.new_model()
    sd = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}; init_hash = lp.state_hash(sd)
    tmp = str(INIT) + ".tmp"; torch.save({"state_dict": sd, "state_hash": init_hash, "seed": 42,
                                         "source": "nvidia/mit-b0 encoder + new 4-class head, mars_common.new_model()"}, tmp)
    os.replace(tmp, INIT); del m
    obj = torch.load(INIT, map_location="cpu", weights_only=True)
    assert lp.state_hash(obj["state_dict"]) == init_hash, "reload of init file failed verification"
mc.set_seed(42); m2 = mc.new_model(); same = lp.state_hash(m2.state_dict()) == init_hash; del m2
init_sha = hashlib.sha256(open(INIT, "rb").read()).hexdigest()
print(f"INIT WEIGHTS: {INIT.name} | {INIT.stat().st_size/1e6:.1f} MB | tensor-content hash {init_hash}")
print(f"              file SHA256 {init_sha}")
print(f"              fresh re-initialisation with seed 42 reproduces the same weights: {same} (saved file is used regardless)")

# ---------- 2. protocol + hashes
N = len(pd.read_csv("/tmp/mars_data/train.csv")); PLAN = md.plan_hash(42, 8, N, 4900)
cfg = dict(stage="lr_pilot_v1", seed=42, batch_size=8, steps_per_epoch=N // 8, total_iters=4900, epochs_equiv=4,
           warmup=100, final_frac=0.05, optimizer="AdamW (torch defaults except lr, weight_decay)", weight_decay=0.01,
           lr_groups="single LR for all parameters", schedule="linear warmup 100 iters then cosine to 5% of peak (stepped per iteration)",
           amp="fp16 autocast + GradScaler", loss="cross entropy, ignore_index=255, no class weights",
           grad_clip="none (norm measured only)", augmentation="horizontal flip p=0.5, deterministic (mars_det)",
           workers=2, cudnn_deterministic=True, cudnn_benchmark=False, eval_iters=[1225, 2450, 3675, 4900],
           eval_split="val (all 1192 images); no test data used", plan_hash=PLAN, init_state_hash=init_hash,
           reject_miou3_pairs=[[2450, 3675], [3675, 4900]], reject_miou3_drop=0.05)
protocol = dict(
    training_config=cfg, candidate_lrs=[3e-5, 1e-4, 3e-4],
    decision_rule=dict(
        primary="mean validation mIoU4 at iterations 3675 and 4900",
        reject="non-finite training loss at any point, OR validation mIoU3 drops by STRICTLY more than 0.05 between consecutive evaluations 2450->3675 or 3675->4900. The 1225->2450 pair is NOT checked.",
        reject_implementation="lr_pilot.miou3_rejections(); drops compared against 0.05 + 1e-9 so a drop of exactly 0.05 does not reject; a rejected candidate keeps training to the end (full curve recorded), is flagged rejected in its summary with the reason, and is excluded from selection. Non-finite loss stops the candidate immediately.",
        tie_break="if the top two primary scores differ by no more than 0.01: lower mean validation loss at 3675 and 4900; then the smaller LR",
        tie_threshold_note="0.01 is a predeclared practical threshold; it is NOT derived from the measured loss noise (no mIoU variability has been measured)",
        big_rock="Big Rock IoU is reported separately and is not used alone to choose the LR",
        boundary="if the best LR is 3e-5 or 3e-4, an extension is proposed to the user and not run automatically"),
    caveats="single seed; 4-epoch pilot may favour larger LRs than a 20-epoch run; it selects a sensible starting LR only",
    versions=dict(torch=torch.__version__, transformers=transformers.__version__, numpy=np.__version__))
(W / "configs").mkdir(exist_ok=True)
json.dump(protocol, open(W / "configs" / "lr_pilot_protocol.json", "w"), indent=2)
print(f"\nPROTOCOL written: configs/lr_pilot_protocol.json")
print(f"  plan hash (seed 42, bs 8, 4900 it) : {PLAN}")
for lr in protocol["candidate_lrs"]:
    print(f"  config hash for lr={lr:g}".ljust(34) + f": {cu.config_hash({**cfg, 'lr': lr})}")

# ---------- 3. unit tests of the rejection rule (pure function, no GPU)
P, T = cfg["reject_miou3_pairs"], cfg["reject_miou3_drop"]
rej = lambda e: lp.miou3_rejections(e, P, T)
pairs_of = lambda e: [r["pair"] for r in rej(e)]
unit = {
    "U1 drop 0.0501 between 2450->3675 is rejected": pairs_of({2450: 0.8000, 3675: 0.7499}) == [[2450, 3675]],
    "U2 drop 0.06 between 3675->4900 is rejected": pairs_of({3675: 0.80, 4900: 0.74}) == [[3675, 4900]],
    "U3 drop of exactly 0.05 (0.85 -> 0.80) is NOT rejected, both pairs": pairs_of({2450: 0.85, 3675: 0.80}) == [] and pairs_of({3675: 0.85, 4900: 0.80}) == [],
    "U4 float trap: 0.80 - 0.75 is > 0.05 in float arithmetic, yet NOT rejected": (0.80 - 0.75 > 0.05) and pairs_of({3675: 0.80, 4900: 0.75}) == [],
    "U5 drop of 0.050001 (just above the threshold) IS rejected": pairs_of({3675: 0.800000, 4900: 0.749999}) == [[3675, 4900]],
    "U6 1225->2450 drop of 0.30 is NOT checked": pairs_of({1225: 0.90, 2450: 0.60}) == [],
    "U7 improvements and small drops are not rejected": pairs_of({2450: 0.70, 3675: 0.75, 4900: 0.74}) == [],
    "U8 both pairs can trigger together (two reasons recorded)": pairs_of({2450: 0.90, 3675: 0.80, 4900: 0.70}) == [[2450, 3675], [3675, 4900]],
    "U9 a missing evaluation never triggers a rejection": pairs_of({2450: 0.90}) == [] and pairs_of({3675: 0.90}) == [],
}
print("\n=== UNIT TESTS: mIoU3 rejection rule ===")
for k, v in unit.items(): print(("PASS  " if v else "FAIL  ") + k)

# ---------- 4. integration tests through run_candidate (scripted validation results, real tiny training loops)
def scripted(values):
    q = list(values)
    def fake(model, loader, max_batches=None):
        v = q.pop(0)
        return dict(loss=1.0, miou4=v - 0.1, miou3=v, pixacc=0.9, iou=[v] * 4, recall=[v] * 4, precision=[v] * 4, conf=[[1, 0, 0, 0]] * 4)
    return fake

ROOT2 = W / "_ruletest"; shutil.rmtree(ROOT2, ignore_errors=True)
tcfg = {**cfg, "total_iters": 8, "eval_iters": [2, 4, 6, 8], "reject_miou3_pairs": [[4, 6], [6, 8]], "plan_hash": md.plan_hash(42, 8, N, 8)}
kw2 = dict(root=ROOT2, init_path=INIT, use_wandb=False, log_every=2)
real_eval = mc.evaluate
try:
    # I1: val mIoU3 = 0.90, 0.60, 0.55 | crash after iteration 6 | resume | 0.40 -> only the 6->8 pair fires, across a resume
    mc.evaluate = scripted([0.90, 0.60, 0.55])
    a = lp.run_candidate(tcfg, 1e-4, "it_split", stop_after=6, **kw2)
    mc.evaluate = scripted([0.40])
    b = lp.run_candidate(tcfg, 1e-4, "it_split", **kw2)
    # I2: 0.90, 0.90, 0.80, 0.75 -> 4->6 drop 0.10 fires, 6->8 drop exactly 0.05 does not
    mc.evaluate = scripted([0.90, 0.90, 0.80, 0.75])
    c = lp.run_candidate(tcfg, 1e-4, "it_exact", **kw2)
finally:
    mc.evaluate = real_eval
vsplit = pd.read_csv(ROOT2 / "logs" / "lr_pilot" / "it_split_val.csv")
integ = {
    "I1a before the crash: 2->4 drop 0.30 not checked, 4->6 drop 0.05 not rejected": a["status"] == "STOPPED_EARLY" and a["rejected"] is False,
    "I1b after resume: 6->8 drop 0.15 rejected, reason recorded with the right pair": b["resumed_from"] == 6 and b["rejected"] is True
        and [r["pair"] for r in b["reject_reasons"]] == [[6, 8]] and abs(b["reject_reasons"][0]["drop"] - 0.15) < 1e-9,
    "I1c rejected candidate trained to the end (8 iterations, status completed) and CSV has no duplicates": b["status"] == "completed"
        and b["iterations_done"] == 8 and list(vsplit.iteration) == [2, 4, 6, 8],
    "I2 exact-0.05 case through the full path: only the 4->6 pair rejected, 6->8 not": c["rejected"] is True and [r["pair"] for r in c["reject_reasons"]] == [[4, 6]],
    "I3 summary file on disk carries rejected + reject_reasons + the rule": all(k in json.load(open(ROOT2 / "results" / "lr_pilot" / "it_exact_summary.json"))
        for k in ("rejected", "reject_reasons", "miou3_rule")),
}
print("\n=== INTEGRATION TESTS: rejection through run_candidate() ===")
for k, v in integ.items(): print(("PASS  " if v else "FAIL  ") + k)
shutil.rmtree(ROOT2, ignore_errors=True)

# ---------- 5. dry-run (scratch folder, deleted afterwards)
ROOT = W / "_dryrun"; shutil.rmtree(ROOT, ignore_errors=True)
dcfg = {**cfg, "total_iters": 5, "eval_iters": [5], "plan_hash": md.plan_hash(42, 8, N, 5)}
kw = dict(root=ROOT, init_path=INIT, eval_max_batches=2, log_every=1)
print("\n=== DRY-RUN D1: 5 iterations, validation on 2 batches, checkpoint, CSV, W&B ===", flush=True)
s1 = lp.run_candidate(dcfg, 1e-4, "dry_full", use_wandb=True, wandb_group="dryrun", **kw)
print("\n=== DRY-RUN D2: stopped after iteration 4 (checkpoint at 3), then resumed ===", flush=True)
s2a = lp.run_candidate(dcfg, 1e-4, "dry_resume", use_wandb=False, ckpt_every=3, stop_after=4, **kw)
s2b = lp.run_candidate(dcfg, 1e-4, "dry_resume", use_wandb=False, ckpt_every=3, **kw)

L = ROOT / "logs" / "lr_pilot"
d1t, d1v = pd.read_csv(L / "dry_full_train.csv"), pd.read_csv(L / "dry_full_val.csv")
d2t, d2v = pd.read_csv(L / "dry_resume_train.csv"), pd.read_csv(L / "dry_resume_val.csv")
print("\nD1 train rows:\n", d1t[["iteration", "loss", "lr", "grad_norm", "skipped_steps_total"]].to_string(index=False))
print("\nD1 validation row (32 images only - plumbing check, not a quality measure):\n",
      d1v[["iteration", "val_loss", "miou4", "miou3", "pixacc", "iou_soil", "iou_bedrock", "iou_sand", "iou_bigrock"]].round(4).to_string(index=False))
ck = torch.load(ROOT / "checkpoints" / "lr_pilot" / "dry_full" / "last.pt", map_location="cpu", weights_only=False)
dl = np.abs(d1t.loss.values[3:] - d2t.loss.values[3:]).max()
checks = {
    "D1 train CSV has iterations 1..5 once each": list(d1t.iteration) == [1, 2, 3, 4, 5],
    "D1 validation CSV has one row at iteration 5": list(d1v.iteration) == [5],
    "D1 losses finite, no skipped steps": bool(np.isfinite(d1t.loss).all()) and s1["skipped_steps_total"] == 0,
    "D1 summary: not rejected, empty reject_reasons": s1["rejected"] is False and s1["reject_reasons"] == [],
    "LR values follow the warmup schedule (peak x factor)": bool(np.allclose(d1t.lr.values, [1e-4 * cu.lr_factor(i, 5, 100, 0.05) for i in range(5)])),
    "D2 stopped early at 4, then resumed from checkpoint at 3 and completed":
        s2a["status"] == "STOPPED_EARLY" and s2b["resumed_from"] == 3 and s2b["status"] == "completed",
    "D2 CSV has no duplicate rows after the resume (iterations 1..5 once each)": list(d2t.iteration) == [1, 2, 3, 4, 5] and list(d2v.iteration) == [5],
    f"resumed run reproduces D1 losses at iterations 4-5 (max diff {dl:.1e} <= 5e-3, sanity bound)": dl <= 5e-3,
    f"resumed run reproduces D1 val loss (diff {abs(d1v.val_loss[0]-d2v.val_loss[0]):.1e} <= 1e-2)": abs(d1v.val_loss[0] - d2v.val_loss[0]) <= 1e-2,
    "checkpoint has every required key, iteration 5, matching config hash":
        all(k in ck for k in cu.REQUIRED) and ck["iteration"] == 5 and ck["config_hash"] == cu.config_hash({**dcfg, "lr": 1e-4}),
    "checkpoint stores python/numpy/torch/cuda RNG states": all(ck["rng"].get(k) is not None for k in ("python", "numpy", "torch", "cuda")),
    "no leftover .tmp files": not list(ROOT.rglob("*.tmp")),
    "W&B connected with zero logging errors (D1)": s1["wandb_connected"] and s1["wandb_errors"] == 0,
    "initial-weights hash verified inside every run (assert did not fire)": True,
}
print("\n=== DRY-RUN RESULTS ===")
for k, v in checks.items(): print(("PASS  " if v else "FAIL  ") + k)
shutil.rmtree(ROOT, ignore_errors=True)
print("\nscratch folders deleted | real pilot folders untouched")
allok = all(unit.values()) and all(integ.values()) and all(checks.values())
print(f"UNIT {sum(unit.values())}/{len(unit)} | INTEGRATION {sum(integ.values())}/{len(integ)} | DRY-RUN {sum(checks.values())}/{len(checks)}")
print("REVISED CELL 12:", "ALL PASS" if allok else "FAILED - paste output")

init file already exists - kept, not overwritten
INIT WEIGHTS: init_seed42.pt | 14.9 MB | tensor-content hash b16b52beb330a9e331d1f16e58b0d55d81070286b28323f7610f88ec0fb20f69
              file SHA256 b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96
              fresh re-initialisation with seed 42 reproduces the same weights: True (saved file is used regardless)

PROTOCOL written: configs/lr_pilot_protocol.json
  plan hash (seed 42, bs 8, 4900 it) : 9327320a886992000712839c7831c8346159f49c09b617e8984ec6a75222caaa
  config hash for lr=3e-05        : 4a07821ab0dcd5c7
  config hash for lr=0.0001       : d39a2929178fdf28
  config hash for lr=0.0003       : 01faca90070fdbc0

=== UNIT TESTS: mIoU3 rejection rule ===
PASS  U1 drop 0.0501 between 2450->3675 is rejected
PASS  U2 drop 0.06 between 3675->4900 is rejected
PASS  U3 drop of exactly 0.05 (0.85 -> 0.80) is NOT rejected, both pairs
PASS  U4 float trap: 0.80 - 0.75 is > 0.05 in float arithmetic, yet NOT rejected
PASS  

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


train/elapsed_s,▁▃▅▆█
train/epoch,▁▃▅▆█
train/grad_norm,▁█▁▄▄
train/iteration,▁▃▅▆█
train/loss,▂▅▁█▄
train/lr,▁▃▅▆█
train/skipped_steps_total,▁▁▁▁▁
val/epoch,▁
val/eval_s,▁
val/iou_bedrock,▁
+16,...



=== DRY-RUN D2: stopped after iteration 4 (checkpoint at 3), then resumed ===
[dry_resume] lr=0.0001 | config hash b0d18eb965081b85 | init hash b16b52beb330 | plan hash 7d9645222f68 | start iteration 0
[dry_resume] RESUMED from iteration 3 (removed 1 CSV rows newer than the checkpoint)
[dry_resume] lr=0.0001 | config hash b0d18eb965081b85 | init hash b16b52beb330 | plan hash 7d9645222f68 | start iteration 3

D1 train rows:
  iteration     loss       lr  grad_norm  skipped_steps_total
         1 1.401128 0.000001   4.417607                    0
         2 1.435290 0.000002   6.484886                    0
         3 1.393387 0.000003   4.355933                    0
         4 1.460657 0.000004   5.266130                    0
         5 1.418703 0.000005   5.134521                    0

D1 validation row (32 images only - plumbing check, not a quality measure):
  iteration  val_loss  miou4  miou3  pixacc  iou_soil  iou_bedrock  iou_sand  iou_bigrock
         5    1.3937 0.0541  0.072  0.

In [15]:
import os, json, hashlib, zipfile, time
from pathlib import Path

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/backup_06_lr_pilot_ready.zip")

# ---- 1. required files (exact paths from the handover)
required = [
    "code/mars_common.py", "code/ckpt_utils.py", "code/tracking.py", "code/mars_det.py",
    "code/recovery.py", "code/lr_pilot.py",
    "checkpoints/lr_pilot/init_seed42.pt",
    "configs/lr_pilot_protocol.json",
    "logs/gpu_timing.json", "logs/det_data_validation.json",
    "logs/dataset_audit_kaggle.json", "logs/dataset_sha256_manifest.json",
    "logs/frozen_split_verification.json", "logs/train_eval_subset_idx_regenerated.json",
    "splits/split_v3_FROZEN.json",
]
missing = [r for r in required if not (W / r).is_file()]
if missing:
    print("MISSING REQUIRED FILES:"); [print("  ", m) for m in missing]
    print("\nSearching for them elsewhere under /kaggle/working ...")
    for m in missing:
        hits = [str(p) for p in Path("/kaggle/working").rglob(Path(m).name)]
        print(f"  {Path(m).name}: {hits if hits else 'not found'}")
    raise SystemExit("STOP: paste this output to Claude. No ZIP was created.")

# ---- 2. everything else worth keeping (small files only; no scratch, caches or other checkpoints)
extra = []
for p in sorted(W.rglob("*")):
    if not p.is_file(): continue
    rel = p.relative_to(W).as_posix()
    if rel in required: continue
    if rel.startswith("checkpoints/") or "__pycache__" in rel or rel.endswith(".tmp") or rel.startswith(("_dryrun", "_ruletest")): continue
    if p.stat().st_size > 50e6: print("skipped (>50 MB):", rel); continue
    extra.append(rel)
files = required + extra

# ---- 3. build the ZIP with a manifest of per-file SHA256
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "files": {f: {"bytes": (W/f).stat().st_size, "sha256": sha(W/f)} for f in files}}
OUT.unlink(missing_ok=True)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(W / f, arcname=f)
    z.writestr("MANIFEST.json", json.dumps(manifest, indent=1))

# ---- 4. verify: CRC test, and every file inside matches its manifest hash
with zipfile.ZipFile(OUT) as z:
    bad = z.testzip()
    names = [i.filename for i in z.infolist()]
    mism = [f for f in files if hashlib.sha256(z.read(f)).hexdigest() != manifest["files"][f]["sha256"]]
print(f"ZIP PATH : {OUT}")
print(f"ZIP SIZE : {OUT.stat().st_size/1e6:.2f} MB")
print(f"CRC TEST : {'OK (no bad files)' if bad is None else 'FAILED at ' + bad}")
print(f"CONTENT HASH CHECK: {'all ' + str(len(files)) + ' files match their manifest' if not mism else 'MISMATCH: ' + str(mism)}")
print(f"init weights file SHA256 inside ZIP: {manifest['files']['checkpoints/lr_pilot/init_seed42.pt']['sha256']}")
print("  (expected b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96)")
print("\nZIP CONTENTS:")
for f in files: print(f"  {manifest['files'][f]['bytes']:>10,}  {f}")
print("  + MANIFEST.json")
print(f"\nZIP SHA256 (Kaggle): {sha(OUT)}")

ZIP PATH : /kaggle/working/backup_06_lr_pilot_ready.zip
ZIP SIZE : 17.11 MB
CRC TEST : OK (no bad files)
CONTENT HASH CHECK: all 17 files match their manifest
init weights file SHA256 inside ZIP: b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96
  (expected b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96)

ZIP CONTENTS:
       5,833  code/mars_common.py
       2,960  code/ckpt_utils.py
       2,343  code/tracking.py
       2,638  code/mars_det.py
       3,011  code/recovery.py
       9,127  code/lr_pilot.py
  14,933,433  checkpoints/lr_pilot/init_seed42.pt
       2,653  configs/lr_pilot_protocol.json
       1,523  logs/gpu_timing.json
         548  logs/det_data_validation.json
         924  logs/dataset_audit_kaggle.json
   2,859,031  logs/dataset_sha256_manifest.json
         290  logs/frozen_split_verification.json
       5,892  logs/train_eval_subset_idx_regenerated.json
         693  splits/split_v3_FROZEN.json
   1,096,754  backups/backup_04_kaggle

In [16]:
import os

backup_path = "/kaggle/working/backup_06_lr_pilot_ready.zip"

print("Backup exists:", os.path.exists(backup_path))

if os.path.exists(backup_path):
    print("Size:", round(os.path.getsize(backup_path) / (1024 * 1024), 2), "MB")
    from IPython.display import FileLink, display
    display(FileLink(backup_path))

Backup exists: True
Size: 16.32 MB


/kaggle/working/backup_06_lr_pilot_ready.zip

In [17]:
import os, sys, json, csv, hashlib, zipfile, time, inspect, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch

W = Path("/kaggle/working/mars_work"); CODE = W / "code"
BACKUP = Path("/kaggle/working/backup_06_lr_pilot_ready.zip")
LR = 3e-5
CKPT_EVERY = 500    # not part of the config hash; set to 0 to save only at the 4 evaluation points
EXPECT = dict(
    cfg_hash="4a07821ab0dcd5c7",
    plan_hash="9327320a886992000712839c7831c8346159f49c09b617e8984ec6a75222caaa",
    init_state_hash="b16b52beb330a9e331d1f16e58b0d55d81070286b28323f7610f88ec0fb20f69",
    init_file_sha="b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96")
def stop(msg): raise SystemExit("STOP: " + msg + "\nNothing further was run. Paste this output to Claude.")
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
os.environ["WANDB_DIR"] = "/tmp/wandb"; Path("/tmp/wandb").mkdir(exist_ok=True)

# ================= PRE-RUN CHECKS =================
print("=== PRE-RUN CHECKS ===")
need = [CODE/f for f in ("mars_common.py", "mars_det.py", "ckpt_utils.py", "tracking.py", "lr_pilot.py")] + \
       [W/"configs/lr_pilot_protocol.json", W/"checkpoints/lr_pilot/init_seed42.pt", Path("/tmp/mars_data/train.csv"), BACKUP]
miss = [str(p) for p in need if not p.exists()]
if miss: stop(f"session looks reset, missing: {miss}")

# 1. files identical to the verified Backup 06
with zipfile.ZipFile(BACKUP) as z: man = json.loads(z.read("MANIFEST.json"))["files"]
chk = ["code/mars_common.py", "code/mars_det.py", "code/ckpt_utils.py", "code/tracking.py", "code/recovery.py", "code/lr_pilot.py",
       "configs/lr_pilot_protocol.json", "checkpoints/lr_pilot/init_seed42.pt", "splits/split_v3_FROZEN.json"]
changed = [f for f in chk if sha(W/f) != man[f]["sha256"]]
if changed: stop(f"files differ from Backup 06: {changed}")
print(f"[ok] {len(chk)} code/protocol/init/split files identical to Backup 06")
if sha(W/"checkpoints/lr_pilot/init_seed42.pt") != EXPECT["init_file_sha"]: stop("init weights file SHA256 differs from Cell 12")
print("[ok] init weights file SHA256 matches Cell 12")

# 2. load the real modules; inspect the real run_candidate() in the file on disk
sys.path.insert(0, str(CODE))
import mars_common as mc, mars_det as md, ckpt_utils as cu, tracking as tk
import lr_pilot as lp; importlib.reload(lp)
sig = inspect.signature(lp.run_candidate)
exp_params = ["cfg", "lr", "tag", "root", "init_path", "use_wandb", "wandb_group", "eval_max_batches",
              "log_every", "ckpt_every", "stop_after", "resume"]
print("run_candidate signature:", sig)
if list(sig.parameters) != exp_params: stop(f"run_candidate parameters differ from the expected list {exp_params}")
if sig.parameters["ckpt_every"].default != 0: stop("ckpt_every default is not 0")
ck_lines = [l.strip() for l in inspect.getsource(lp.run_candidate).splitlines() if "ckpt_every" in l]
print("lines mentioning ckpt_every in the real file:"); [print("   ", l) for l in ck_lines]
if "elif ckpt_every and it % ckpt_every == 0:" not in ck_lines: stop("the periodic-checkpoint line is not as expected")
print(f"[ok] run_candidate supports ckpt_every; this run uses ckpt_every={CKPT_EVERY} (saves at multiples of {CKPT_EVERY} plus the 4 evaluation points)")

# 3. protocol and hashes
protocol = json.load(open(W/"configs/lr_pilot_protocol.json")); cfg = protocol["training_config"]
if LR not in protocol["candidate_lrs"]: stop("3e-5 is not in the protocol's candidate list")
ch = cu.config_hash({**cfg, "lr": LR})
bad = {k: v for k, v in dict(config_hash=(ch, EXPECT["cfg_hash"]), plan_hash=(cfg["plan_hash"], EXPECT["plan_hash"]),
        init_state_hash=(cfg["init_state_hash"], EXPECT["init_state_hash"])).items() if v[0] != v[1]}
if bad: stop(f"hash mismatch (got, expected): {bad}")
print(f"[ok] config hash {ch} | plan hash {cfg['plan_hash'][:12]}... | init hash {cfg['init_state_hash'][:12]}...")
print(f"[ok] protocol: batch {cfg['batch_size']}, {cfg['total_iters']} iterations, eval at {cfg['eval_iters']}, reject rule {cfg['reject_miou3_pairs']} > {cfg['reject_miou3_drop']}")

# 4. data and GPU (sizes read through the same dataset class the run uses)
ntr, nva = len(mc.MarsSeg("train")), len(mc.MarsSeg("val"))
if (ntr, nva) != (9803, 1192): stop(f"unexpected data sizes train={ntr} val={nva}")
print(f"[ok] data: train {ntr}, val {nva} (the run validates on all {nva}; no test data)")
if not torch.cuda.is_available(): stop("CUDA is not available")
free, tot = torch.cuda.mem_get_info(0)
if free < 8e9: stop(f"only {free/1e9:.1f} GB free on GPU 0 - restart the session's other processes")
print(f"[ok] GPU 0: {torch.cuda.get_device_name(0)}, {free/1e9:.1f} of {tot/1e9:.1f} GB free")

# 5. state of this candidate
tag = lp.tag_for(LR); res_dir = W/"results/lr_pilot"; ck_path = W/"checkpoints/lr_pilot"/tag/"last.pt"
sum_path = res_dir/f"{tag}_summary.json"; prefix = W/"logs/lr_pilot"/tag
summary = None
if sum_path.exists():
    old = json.load(open(sum_path))
    if str(old["status"]).startswith("REJECT_NONFINITE"): stop(f"previous run ended with {old['status']} - do not retrain; paste this to Claude")
    if old["status"] == "completed" and old["iterations_done"] == cfg["total_iters"]:
        summary = old; print(f"[state] {tag} already COMPLETED - skipping training, verification only")
if summary is None:
    if ck_path.exists():
        it_saved = torch.load(ck_path, map_location="cpu", weights_only=False)["iteration"]
        print(f"[state] checkpoint found at iteration {it_saved} - this run will RESUME from it")
    else: print(f"[state] no checkpoint - FRESH start from the shared initial weights")

# ================= TRAINING =================
if summary is None:
    print(f"\n=== TRAINING {tag} (lr={LR:g}) - about 13-15 minutes; keep this tab open ===", flush=True)
    t_start = time.time()
    summary = lp.run_candidate(cfg, LR, tag, W, W/"checkpoints/lr_pilot/init_seed42.pt",
                               use_wandb=True, wandb_group="lr_pilot", ckpt_every=CKPT_EVERY)
    print(f"\nrun_candidate returned after {(time.time()-t_start)/60:.1f} min | status = {summary['status']}")

# ================= OUTPUT VERIFICATION (local files only decide pass/fail) =================
print("\n=== OUTPUT VERIFICATION ===")
tr = pd.read_csv(f"{prefix}_train.csv"); va = pd.read_csv(f"{prefix}_val.csv")
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
vj = [json.load(open(res_dir/f"{tag}_val_it{i}.json")) if (res_dir/f"{tag}_val_it{i}.json").exists() else None for i in cfg["eval_iters"]]
exp_tr = list(range(25, cfg["total_iters"] + 1, 25))
exp_lr = np.array([LR * cu.lr_factor(i - 1, cfg["total_iters"], cfg["warmup"], cfg["final_frac"]) for i in tr.iteration])
checks = {
    "summary status is completed, 4900 iterations done": summary["status"] == "completed" and summary["iterations_done"] == 4900,
    "summary config/plan/init hashes match the protocol": summary["config_hash"] == ch and summary["plan_hash"] == cfg["plan_hash"] and summary["init_state_hash"] == cfg["init_state_hash"],
    "train CSV: iterations 25..4900 (every 25), each exactly once": list(tr.iteration) == exp_tr,
    "train CSV: all losses finite": bool(np.isfinite(tr.loss).all()),
    "train CSV: LR follows the schedule (peak 3e-5, warmup 100, cosine to 5%)": bool(np.allclose(tr.lr.values, exp_lr, rtol=1e-5)),
    "train CSV: grad norms finite in >=95% of rows (sanity)": float(np.isfinite(tr.grad_norm).mean()) >= 0.95,
    "val CSV: iterations exactly 1225, 2450, 3675, 4900": list(va.iteration) == cfg["eval_iters"],
    "val CSV: all losses and mIoU values finite": bool(np.isfinite(va[["val_loss", "miou4", "miou3"]].values).all()),
    "4 validation JSON files exist and record the right iterations": all(j is not None for j in vj) and [j["iteration"] for j in vj] == cfg["eval_iters"],
    "checkpoint: all required keys, iteration 4900, matching config hash": all(k in ck for k in cu.REQUIRED) and ck["iteration"] == 4900 and ck["config_hash"] == ch,
    "checkpoint: RNG states stored": all(ck["rng"].get(k) is not None for k in ("python", "numpy", "torch", "cuda")),
    "no leftover .tmp files": not list(W.rglob("*.tmp")),
    "summary carries rejected flag + reasons + rule": all(k in summary for k in ("rejected", "reject_reasons", "miou3_rule")),
}
for k, v in checks.items(): print(("PASS  " if v else "FAIL  ") + k)
ok = all(checks.values())

# informational only: not part of pass/fail
wb_conn, wb_err = summary.get("wandb_connected"), summary.get("wandb_errors")
print(f"\nW&B (informational only, NOT part of pass/fail): connected={wb_conn}, logging errors={wb_err}")
if (not wb_conn) or wb_err: print("  note: W&B had a problem; the local CSVs and checkpoint above are the primary record and were checked.")
if all(j is not None and "conf" in j for j in vj):
    tots = [int(np.sum(j["conf"])) for j in vj]
    print(f"info: labelled-pixel total in each validation confusion matrix: {tots} -> "
          f"{'identical (same images each time)' if len(set(tots)) == 1 else 'DIFFERENT - paste this to Claude'}")

# ================= RESULTS (this candidate only; no selection yet) =================
print(f"\n=== {tag} RESULTS (validation only) ===")
cols = ["iteration", "val_loss", "miou4", "miou3", "pixacc", "iou_soil", "iou_bedrock", "iou_sand", "iou_bigrock"]
print(va[cols].round(4).to_string(index=False))
late = va[va.iteration.isin([3675, 4900])]
print(f"\nprimary score so far (mean val mIoU4 at 3675 and 4900): {late.miou4.mean():.4f}   | mean val loss at those: {late.val_loss.mean():.4f}")
print(f"final train loss (mean of last 4 logged windows): {tr.loss.tail(4).mean():.4f}")
print(f"skipped optimiser steps (fp16 overflow): {summary['skipped_steps_total']} | resumed_from: {summary['resumed_from']} | "
      f"train time of the last segment: {summary['train_time_s']/60:.1f} min (after a resume this covers only the resumed part)")
print(f"REJECTED by the predeclared rule: {summary['rejected']}")
for r in summary["reject_reasons"]: print("   reason:", r)
json.dump(dict(tag=tag, lr=LR, all_checks_passed=ok, checks=checks, config_hash=ch, ckpt_every_used=CKPT_EVERY,
               rejected=summary["rejected"], primary_score_so_far=float(late.miou4.mean()),
               wandb_info_not_part_of_verdict=dict(connected=wb_conn, errors=wb_err),
               verified_at=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(res_dir/f"{tag}_verification.json", "w"), indent=1)
print("\nCANDIDATE 3e-5:", "RUN COMPLETE AND VERIFIED" if ok else "VERIFICATION FAILED - paste output")

=== PRE-RUN CHECKS ===
[ok] 9 code/protocol/init/split files identical to Backup 06
[ok] init weights file SHA256 matches Cell 12
run_candidate signature: (cfg, lr, tag, root, init_path, use_wandb=True, wandb_group='lr_pilot', eval_max_batches=None, log_every=25, ckpt_every=0, stop_after=None, resume=True)
lines mentioning ckpt_every in the real file:
    eval_max_batches=None, log_every=25, ckpt_every=0, stop_after=None, resume=True):
    elif ckpt_every and it % ckpt_every == 0:
[ok] run_candidate supports ckpt_every; this run uses ckpt_every=500 (saves at multiples of 500 plus the 4 evaluation points)
[ok] config hash 4a07821ab0dcd5c7 | plan hash 9327320a8869... | init hash b16b52beb330...
[ok] protocol: batch 8, 4900 iterations, eval at [1225, 2450, 3675, 4900], reject rule [[2450, 3675], [3675, 4900]] > 0.05
[ok] data: train 9803, val 1192 (the run validates on all 1192; no test data)
[ok] GPU 0: Tesla T4, 15.2 of 15.6 GB free
[state] no checkpoint - FRESH start from the shared in

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


train/elapsed_s,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/grad_norm,▆█▄▆▆█▆▄▃▃▆▅▄▄▃▂▆▆▅█▃▃▃▅▂▂▂▄▃▃▂▃▄▃▂▂▂▃▃▁
train/iteration,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇██
train/loss,█▇▅▅▅▄▃▃▃▃▃▂▃▃▂▂▂▁▂▂▂▂▁▂▂▂▂▂▂▂▁▁▁▂▁▁▂▂▁▂
train/lr,████████▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
train/skipped_steps_total,▁▃▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆█████
val/epoch,▁▃▆█
val/eval_s,▁█▁▁
val/iou_bedrock,▁▄█▆
+16,...



run_candidate returned after 13.3 min | status = completed

=== OUTPUT VERIFICATION ===
PASS  summary status is completed, 4900 iterations done
PASS  summary config/plan/init hashes match the protocol
PASS  train CSV: iterations 25..4900 (every 25), each exactly once
PASS  train CSV: all losses finite
PASS  train CSV: LR follows the schedule (peak 3e-5, warmup 100, cosine to 5%)
PASS  train CSV: grad norms finite in >=95% of rows (sanity)
PASS  val CSV: iterations exactly 1225, 2450, 3675, 4900
PASS  val CSV: all losses and mIoU values finite
PASS  4 validation JSON files exist and record the right iterations
PASS  checkpoint: all required keys, iteration 4900, matching config hash
PASS  checkpoint: RNG states stored
PASS  no leftover .tmp files
PASS  summary carries rejected flag + reasons + rule

W&B (informational only, NOT part of pass/fail): connected=True, logging errors=0
info: labelled-pixel total in each validation confusion matrix: [194254320, 194254320, 194254320, 194254320

In [18]:
import os, json, hashlib, zipfile, time
from pathlib import Path
import torch, pandas as pd

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/backup_07_candidate1_lr3e-05.zip")
TAG = "lr3e-05"

required = [
    "code/mars_common.py", "code/ckpt_utils.py", "code/tracking.py", "code/mars_det.py", "code/recovery.py", "code/lr_pilot.py",
    "configs/lr_pilot_protocol.json", "splits/split_v3_FROZEN.json",
    "checkpoints/lr_pilot/init_seed42.pt", f"checkpoints/lr_pilot/{TAG}/last.pt",
    f"logs/lr_pilot/{TAG}_train.csv", f"logs/lr_pilot/{TAG}_val.csv",
    f"results/lr_pilot/{TAG}_summary.json", f"results/lr_pilot/{TAG}_verification.json",
] + [f"results/lr_pilot/{TAG}_val_it{i}.json" for i in (1225, 2450, 3675, 4900)]
missing = [r for r in required if not (W / r).is_file()]
if missing:
    print("MISSING REQUIRED FILES:"); [print("  ", m) for m in missing]
    raise SystemExit("STOP: paste this output to Claude. No ZIP was created.")

files = list(required)
for p in sorted(W.rglob("*")):
    if not p.is_file(): continue
    rel = p.relative_to(W).as_posix()
    if rel in files: continue
    if rel.startswith(("backups/", "_dryrun", "_ruletest")) or "__pycache__" in rel or rel.endswith(".tmp"): continue
    if p.stat().st_size > 200e6: print("skipped (>200 MB):", rel); continue
    files.append(rel)

sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "note": "backup after LR-pilot candidate 1 (lr 3e-5)",
            "files": {f: {"bytes": (W/f).stat().st_size, "sha256": sha(W/f)} for f in files}}
OUT.unlink(missing_ok=True)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(W / f, arcname=f)
    z.writestr("MANIFEST.json", json.dumps(manifest, indent=1))

with zipfile.ZipFile(OUT) as z:
    bad = z.testzip()
    mism = [f for f in files if hashlib.sha256(z.read(f)).hexdigest() != manifest["files"][f]["sha256"]]
print(f"ZIP PATH : {OUT}")
print(f"ZIP SIZE : {OUT.stat().st_size/1e6:.2f} MB")
print(f"CRC TEST : {'OK (no bad files)' if bad is None else 'FAILED at ' + bad}")
print(f"CONTENT HASH CHECK: {'all ' + str(len(files)) + ' files match their manifest' if not mism else 'MISMATCH: ' + str(mism)}")
print(f"init weights SHA256 in ZIP: {manifest['files']['checkpoints/lr_pilot/init_seed42.pt']['sha256']} (expected b698a642...04d96)")
print("\nZIP CONTENTS:")
for f in files: print(f"  {manifest['files'][f]['bytes']:>12,}  {f}")
print("  + MANIFEST.json")
print(f"\nZIP SHA256 (Kaggle): {sha(OUT)}")

# ---- read-only inspection of the final checkpoint (CPU only; nothing is changed)
print("\n=== READ-ONLY CHECKPOINT INSPECTION (informational) ===")
ck = torch.load(W / f"checkpoints/lr_pilot/{TAG}/last.pt", map_location="cpu", weights_only=False)
steps = sorted({int(float(s["step"])) for s in ck["optimizer"]["state"].values()})
skipped = ck["extra"].get("skipped")
print(f"checkpoint iteration {ck['iteration']}, epoch {ck['epoch']}, config hash {ck['config_hash']}")
print(f"scheduler last_epoch {ck['scheduler'].get('last_epoch')} | optimiser param-group lr {ck['optimizer']['param_groups'][0]['lr']:.3e}")
print(f"optimiser update count(s) across parameters: {steps} | skipped recorded: {skipped} | expected updates = 4900 - skipped = {4900 - skipped}")
sc = ck["scaler"]; print("scaler state:", {k: sc.get(k) for k in ("scale", "_growth_tracker", "growth_interval", "backoff_factor")})
tr = pd.read_csv(W / f"logs/lr_pilot/{TAG}_train.csv"); inc = tr[tr.skipped_steps_total.diff().fillna(tr.skipped_steps_total) > 0]
print("25-iteration windows in which skipped_steps_total increased (iteration -> total):", dict(zip(inc.iteration, inc.skipped_steps_total)))

ZIP PATH : /kaggle/working/backup_07_candidate1_lr3e-05.zip
ZIP SIZE : 56.24 MB
CRC TEST : OK (no bad files)
CONTENT HASH CHECK: all 24 files match their manifest
init weights SHA256 in ZIP: b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96 (expected b698a642...04d96)

ZIP CONTENTS:
         5,833  code/mars_common.py
         2,960  code/ckpt_utils.py
         2,343  code/tracking.py
         2,638  code/mars_det.py
         3,011  code/recovery.py
         9,127  code/lr_pilot.py
         2,653  configs/lr_pilot_protocol.json
           693  splits/split_v3_FROZEN.json
    14,933,433  checkpoints/lr_pilot/init_seed42.pt
    44,856,523  checkpoints/lr_pilot/lr3e-05/last.pt
        20,558  logs/lr_pilot/lr3e-05_train.csv
         1,543  logs/lr_pilot/lr3e-05_val.csv
         3,566  results/lr_pilot/lr3e-05_summary.json
         1,106  results/lr_pilot/lr3e-05_verification.json
           817  results/lr_pilot/lr3e-05_val_it1225.json
           821  results/lr_pilot/lr3e-

In [19]:
import os, sys, json, hashlib, zipfile, random, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import DataLoader

W = Path("/kaggle/working/mars_work"); CODE = W / "code"; TAG = "lr3e-05"
BACKUP6 = Path("/kaggle/working/backup_06_lr_pilot_ready.zip")
BACKUP7 = Path("/kaggle/working/backup_07_candidate1_lr3e-05.zip")
LAPTOP_SHA = "E6F698776D0C6ED3B742A870F6B52438AC9064EB8CAEB6339140AC128CBD04AB".lower()
EXPECT_TOTAL = 194254320
def stop(msg): raise SystemExit("STOP: " + msg + "\nNothing further was run. Paste this output to Claude.")
def sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
def snapshot():
    return {p.relative_to(W).as_posix(): sha_file(p) for p in sorted(W.rglob("*"))
            if p.is_file() and not p.relative_to(W).as_posix().startswith("backups/") and "__pycache__" not in p.parts}

# ---------- 0. Backup 07 vs laptop
print("=== 0. BACKUP 07: KAGGLE vs LAPTOP ===")
if BACKUP7.exists():
    kag = sha_file(BACKUP7)
    print("Kaggle SHA256:", kag); print("Laptop SHA256:", LAPTOP_SHA)
    print("VERDICT:", "MATCH" if kag == LAPTOP_SHA else "MISMATCH - re-download the ZIP and tell Claude")
else:
    print("Backup 07 ZIP not found in /kaggle/working (session reset?) - cannot compare here; paste your Cell 15 output instead.")

# ---------- pre-checks
print("\n=== PRE-CHECKS ===")
need = [CODE/f for f in ("mars_common.py", "mars_det.py", "ckpt_utils.py", "tracking.py", "lr_pilot.py")] + [
    W/f"checkpoints/lr_pilot/{TAG}/last.pt", W/f"results/lr_pilot/{TAG}_val_it4900.json", W/f"logs/lr_pilot/{TAG}_val.csv",
    W/"configs/lr_pilot_protocol.json", Path("/tmp/mars_data/val.csv")]
miss = [str(p) for p in need if not p.exists()]
if miss: stop(f"session looks reset, missing: {miss}")
if BACKUP6.exists():
    with zipfile.ZipFile(BACKUP6) as z: man = json.loads(z.read("MANIFEST.json"))["files"]
    chk = [f"code/{n}" for n in ("mars_common.py", "mars_det.py", "ckpt_utils.py", "tracking.py", "recovery.py", "lr_pilot.py")] + ["configs/lr_pilot_protocol.json"]
    changed = [f for f in chk if sha_file(W/f) != man[f]["sha256"]]
    if changed: stop(f"files differ from Backup 06: {changed}")
    print(f"[ok] {len(chk)} code/protocol files identical to Backup 06")
else: print("[warn] Backup 06 ZIP not found; code identity vs Backup 06 not re-checked")
snap_before = snapshot(); print(f"[ok] SHA256 snapshot of {len(snap_before)} files under mars_work/ taken (compared again at the end)")

sys.path.insert(0, str(CODE))
import mars_common as mc, ckpt_utils as cu, lr_pilot as lp
cfg = json.load(open(W/"configs/lr_pilot_protocol.json"))["training_config"]
orig = json.load(open(W/f"results/lr_pilot/{TAG}_val_it4900.json"))
assert orig["iteration"] == 4900 and orig["tag"] == TAG
oconf = np.array(orig["conf"], dtype=np.int64)

# ---------- B. non-255 pixel count over the validation masks (CPU only)
print("\n=== CHECK B: LABELLED PIXELS IN THE VALIDATION MASKS ===")
vdf = pd.read_csv(f"{mc.DATA}/val.csv"); assert len(vdf) == 1192, len(vdf)
cnt = np.zeros(256, dtype=np.int64)
for r in vdf.mask_512:
    m = np.array(Image.open(os.path.join(mc.DATA, r)), dtype=np.uint8); assert m.shape == (512, 512), r
    cnt += np.bincount(m.ravel(), minlength=256)
labels_present = [int(i) for i in np.nonzero(cnt)[0]]
total_px, ignored, labelled = int(cnt.sum()), int(cnt[255]), int(total_px_ := cnt.sum() - cnt[255])
print(f"masks read: {len(vdf)} | total pixels {total_px:,} (= 1192*512*512: {total_px == 1192*512*512}) | label values present: {labels_present}")
print(f"label-255 pixels: {ignored:,} | non-255 (labelled) pixels: {labelled:,} | recorded confusion total: {int(oconf.sum()):,} | expected: {EXPECT_TOTAL:,}")
gt_rows = oconf.sum(1).tolist(); gt_masks = cnt[:4].tolist()
print(f"per-class ground-truth pixels  masks: {gt_masks}\n                           confusion rows: {gt_rows}")
B_ok = (set(labels_present) <= {0, 1, 2, 3, 255} and labelled == EXPECT_TOTAL == int(oconf.sum()) and gt_masks == gt_rows)
print("CHECK B:", "PASS - the evaluation covered exactly the labelled pixels of all 1192 masks" if B_ok else "FAIL - see numbers above")

# ---------- A. re-evaluate the iteration-4900 checkpoint (RNG and flags restored afterwards)
print("\n=== CHECK A: RE-EVALUATE THE ITERATION-4900 CHECKPOINT ===")
rng_before = cu.get_rng_state(); flags_before = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
ev, A_err = None, None
try:
    torch.backends.cudnn.deterministic = cfg["cudnn_deterministic"]; torch.backends.cudnn.benchmark = cfg["cudnn_benchmark"]
    ck = torch.load(W/f"checkpoints/lr_pilot/{TAG}/last.pt", map_location="cpu", weights_only=False)
    print(f"checkpoint: iteration {ck['iteration']}, epoch {ck['epoch']}, config hash {ck['config_hash']}")
    assert ck["iteration"] == 4900
    model = mc.new_model(); model.load_state_dict(ck["model"])           # strict load
    print(f"weights loaded (strict); tensor-content hash {lp.state_hash(model.state_dict())[:16]}...")
    val_loader = DataLoader(mc.MarsSeg("val"), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    ev = mc.evaluate(model, val_loader)
    del model, val_loader, ck; gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    A_err = repr(e)
finally:
    cu.set_rng_state(rng_before); torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = flags_before
rng_after = cu.get_rng_state()
rng_ok = (random.getstate() == rng_before["python"] and torch.equal(torch.get_rng_state(), rng_before["torch"]) and
          all(np.array_equal(a, b) if isinstance(a, np.ndarray) else a == b for a, b in zip(np.random.get_state(), rng_before["numpy"])) and
          all(torch.equal(a, b) for a, b in zip(rng_after["cuda"], rng_before["cuda"])) and
          (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark) == flags_before)
print(f"RNG states (python/numpy/torch/cuda) and cudnn flags restored exactly: {rng_ok}")
A_ok = False
if A_err: print("CHECK A could not run:", A_err)
else:
    nconf = np.array(ev["conf"], dtype=np.int64); diffcells = int((nconf != oconf).sum())
    print(f"confusion matrix identical: {diffcells == 0} (differing cells: {diffcells}, max |diff| {int(np.abs(nconf - oconf).max())} pixels)")
    rows = [("val_loss", orig["loss"], ev["loss"]), ("miou4", orig["miou4"], ev["miou4"]), ("miou3", orig["miou3"], ev["miou3"]), ("pixacc", orig["pixacc"], ev["pixacc"])]
    for k, nm in enumerate(lp.CN):
        rows += [(f"iou_{nm}", orig["iou"][k], ev["iou"][k]), (f"recall_{nm}", orig["recall"][k], ev["recall"][k]), (f"precision_{nm}", orig["precision"][k], ev["precision"][k])]
    print(f"\n{'metric':<18}{'original (JSON)':>18}{'re-evaluated':>18}{'difference':>14}")
    for nm, a, b in rows: print(f"{nm:<18}{a:>18.10f}{b:>18.10f}{b - a:>14.2e}")
    dmax = max(abs(b - a) for _, a, b in rows)
    A_ok = (diffcells == 0) and abs(ev["loss"] - orig["loss"]) <= 1e-5
    vrow = pd.read_csv(W/f"logs/lr_pilot/{TAG}_val.csv").query("iteration == 4900").iloc[0]
    cmap = {"val_loss": ev["loss"], "miou4": ev["miou4"], "miou3": ev["miou3"], "pixacc": ev["pixacc"]}
    for k, nm in enumerate(lp.CN): cmap[f"iou_{nm}"] = ev["iou"][k]; cmap[f"recall_{nm}"] = ev["recall"][k]; cmap[f"precision_{nm}"] = ev["precision"][k]
    dcsv = max(abs(float(vrow[k]) - v) for k, v in cmap.items())
    print(f"\nmax |difference| vs JSON: {dmax:.2e} | vs the 4900 row of the validation CSV: {dcsv:.2e}")
    print("CHECK A:", "PASS - the saved iteration-4900 checkpoint reproduces the recorded validation results" if A_ok else
          "FAIL - mismatch: do not proceed; paste this output")

# ---------- read-only proof
snap_after = snapshot()
changed = sorted(k for k in snap_before if snap_before[k] != snap_after.get(k)); added = sorted(set(snap_after) - set(snap_before))
print(f"\nfiles changed: {changed} | files added: {added} | files removed: {sorted(set(snap_before) - set(snap_after))}")
ro_ok = not changed and not added and set(snap_before) == set(snap_after)
print("READ-ONLY CONFIRMED (every file under mars_work/ byte-identical to before):", ro_ok)
print("\nALL READ-ONLY CHECKS:", "PASS" if (A_ok and B_ok and rng_ok and ro_ok) else "FAILED - paste output, do not proceed")

=== 0. BACKUP 07: KAGGLE vs LAPTOP ===
Kaggle SHA256: e6f698776d0c6ed3b742a870f6b52438ac9064eb8caeb6339140ac128cbd04ab
Laptop SHA256: e6f698776d0c6ed3b742a870f6b52438ac9064eb8caeb6339140ac128cbd04ab
VERDICT: MATCH

=== PRE-CHECKS ===
[ok] 7 code/protocol files identical to Backup 06
[ok] SHA256 snapshot of 24 files under mars_work/ taken (compared again at the end)

=== CHECK B: LABELLED PIXELS IN THE VALIDATION MASKS ===
masks read: 1192 | total pixels 312,475,648 (= 1192*512*512: True) | label values present: [0, 1, 2, 3, 255]
label-255 pixels: 118,221,328 | non-255 (labelled) pixels: 194,254,320 | recorded confusion total: 194,254,320 | expected: 194,254,320
per-class ground-truth pixels  masks: [67509894, 103205191, 21865497, 1673738]
                           confusion rows: [67509894, 103205191, 21865497, 1673738]
CHECK B: PASS - the evaluation covered exactly the labelled pixels of all 1192 masks

=== CHECK A: RE-EVALUATE THE ITERATION-4900 CHECKPOINT ===
checkpoint: iteration 

In [20]:
import os, sys, json, csv, hashlib, zipfile, time, inspect, importlib, threading
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm import tqdm

W = Path("/kaggle/working/mars_work"); CODE = W / "code"
BACKUP6 = Path("/kaggle/working/backup_06_lr_pilot_ready.zip")
BACKUP7 = Path("/kaggle/working/backup_07_candidate1_lr3e-05.zip")
LR = 1e-4
CKPT_EVERY = 500    # not part of the config hash; saves at multiples of 500 plus the 4 evaluation points
EXPECT = dict(
    cfg_hash="d39a2929178fdf28",
    plan_hash="9327320a886992000712839c7831c8346159f49c09b617e8984ec6a75222caaa",
    init_state_hash="b16b52beb330a9e331d1f16e58b0d55d81070286b28323f7610f88ec0fb20f69",
    init_file_sha="b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96")
def stop(msg): raise SystemExit("STOP: " + msg + "\nNothing further was run. Paste this output to Claude.")
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
os.environ["WANDB_DIR"] = "/tmp/wandb"; Path("/tmp/wandb").mkdir(exist_ok=True)

# ================= PRE-RUN CHECKS =================
print("=== PRE-RUN CHECKS ===")
need = [CODE/f for f in ("mars_common.py", "mars_det.py", "ckpt_utils.py", "tracking.py", "lr_pilot.py")] + \
       [W/"configs/lr_pilot_protocol.json", W/"checkpoints/lr_pilot/init_seed42.pt", Path("/tmp/mars_data/train.csv"), BACKUP6, BACKUP7]
miss = [str(p) for p in need if not p.exists()]
if miss: stop(f"session looks reset, missing: {miss}")

# 1. validated files identical to Backup 06; candidate-1 results identical to Backup 07
with zipfile.ZipFile(BACKUP6) as z: man6 = json.loads(z.read("MANIFEST.json"))["files"]
chk = ["code/mars_common.py", "code/mars_det.py", "code/ckpt_utils.py", "code/tracking.py", "code/recovery.py", "code/lr_pilot.py",
       "configs/lr_pilot_protocol.json", "checkpoints/lr_pilot/init_seed42.pt", "splits/split_v3_FROZEN.json"]
changed = [f for f in chk if sha(W/f) != man6[f]["sha256"]]
if changed: stop(f"files differ from Backup 06: {changed}")
print(f"[ok] {len(chk)} code/protocol/init/split files identical to Backup 06")
if sha(W/"checkpoints/lr_pilot/init_seed42.pt") != EXPECT["init_file_sha"]: stop("init weights file SHA256 differs from Cell 12")
print("[ok] init weights file SHA256 matches Cell 12")
with zipfile.ZipFile(BACKUP7) as z: man7 = json.loads(z.read("MANIFEST.json"))["files"]
c1_files = [f for f in man7 if "lr3e-05" in f]
def c1_changed(): return [f for f in c1_files if not (W/f).is_file() or sha(W/f) != man7[f]["sha256"]]
if len(c1_files) != 9 or c1_changed(): stop(f"candidate-1 files differ from Backup 07 (found {len(c1_files)} files): {c1_changed()}")
print(f"[ok] {len(c1_files)} candidate-1 result files identical to Backup 07 (they must stay unchanged)")

# 2. real modules; inspect the real run_candidate()
sys.path.insert(0, str(CODE))
import mars_common as mc, mars_det as md, ckpt_utils as cu, tracking as tk
import lr_pilot as lp; importlib.reload(lp)
sig = inspect.signature(lp.run_candidate)
exp_params = ["cfg", "lr", "tag", "root", "init_path", "use_wandb", "wandb_group", "eval_max_batches",
              "log_every", "ckpt_every", "stop_after", "resume"]
if list(sig.parameters) != exp_params: stop(f"run_candidate parameters differ from the expected list {exp_params}")
if "elif ckpt_every and it % ckpt_every == 0:" not in [l.strip() for l in inspect.getsource(lp.run_candidate).splitlines()]:
    stop("the periodic-checkpoint line is not as expected")
print(f"[ok] run_candidate signature as validated; ckpt_every={CKPT_EVERY}")

# 3. protocol and hashes
protocol = json.load(open(W/"configs/lr_pilot_protocol.json")); cfg = protocol["training_config"]
if LR not in protocol["candidate_lrs"]: stop("1e-4 is not in the protocol's candidate list")
ch = cu.config_hash({**cfg, "lr": LR})
bad = {k: v for k, v in dict(config_hash=(ch, EXPECT["cfg_hash"]), plan_hash=(cfg["plan_hash"], EXPECT["plan_hash"]),
        init_state_hash=(cfg["init_state_hash"], EXPECT["init_state_hash"])).items() if v[0] != v[1]}
if bad: stop(f"hash mismatch (got, expected): {bad}")
print(f"[ok] config hash {ch} | plan hash {cfg['plan_hash'][:12]}... | init hash {cfg['init_state_hash'][:12]}...")
print(f"[ok] protocol: batch {cfg['batch_size']}, {cfg['total_iters']} iterations, eval at {cfg['eval_iters']}, reject rule {cfg['reject_miou3_pairs']} > {cfg['reject_miou3_drop']}")

# 4. data and GPU
ntr, nva = len(mc.MarsSeg("train")), len(mc.MarsSeg("val"))
if (ntr, nva) != (9803, 1192): stop(f"unexpected data sizes train={ntr} val={nva}")
print(f"[ok] data: train {ntr}, val {nva} (no test data)")
if not torch.cuda.is_available(): stop("CUDA is not available")
free, tot = torch.cuda.mem_get_info(0)
if free < 8e9: stop(f"only {free/1e9:.1f} GB free on GPU 0")
print(f"[ok] GPU 0: {torch.cuda.get_device_name(0)}, {free/1e9:.1f} of {tot/1e9:.1f} GB free")

# 5. state of this candidate
tag = lp.tag_for(LR); res_dir = W/"results/lr_pilot"; ck_path = W/"checkpoints/lr_pilot"/tag/"last.pt"
sum_path = res_dir/f"{tag}_summary.json"; prefix = W/"logs/lr_pilot"/tag
total = cfg["total_iters"]; summary = None; start_it = 0
if sum_path.exists():
    old = json.load(open(sum_path))
    if str(old["status"]).startswith("REJECT_NONFINITE"): stop(f"previous run ended with {old['status']} - do not retrain")
    if old["status"] == "completed" and old["iterations_done"] == total:
        summary = old; print(f"[state] {tag} already COMPLETED - skipping training, verification only")
if summary is None:
    if ck_path.exists():
        start_it = torch.load(ck_path, map_location="cpu", weights_only=False)["iteration"]
        print(f"[state] checkpoint found at iteration {start_it} - this run will RESUME from it")
    else: print("[state] no checkpoint - FRESH start from the shared initial weights")

# ================= TRAINING (with a read-only progress-bar thread) =================
def read_last(path, t_min):
    try:
        if not path.exists() or path.stat().st_mtime < t_min: return None     # ignore stale files from before this run
        with open(path, newline="") as f: rows = list(csv.reader(f))
        if len(rows) < 2 or len(rows[-1]) != len(rows[0]): return None
        return dict(zip(rows[0], rows[-1]))
    except Exception: return None

def progress_worker(pbar, stop_ev, t_min):
    while not stop_ev.is_set():
        try:
            r = read_last(Path(f"{prefix}_train.csv"), t_min)
            if r:
                it = int(float(r["iteration"]))
                if it > pbar.n: pbar.update(it - pbar.n)
                post = dict(loss=f"{float(r['loss']):.4f}", lr=f"{float(r['lr']):.1e}", skipped=int(float(r["skipped_steps_total"])))
                v = read_last(Path(f"{prefix}_val.csv"), t_min)
                if v: post["val_mIoU4"] = f"{float(v['miou4']):.4f}@{int(float(v['iteration']))}"
                pbar.set_postfix(post, refresh=True)
        except Exception: pass
        stop_ev.wait(5)

if summary is None:
    print(f"\n=== TRAINING {tag} (lr={LR:g}) - about 13-15 minutes; keep this tab open ===", flush=True)
    t_start = time.time()
    pbar = tqdm(total=total, initial=start_it, desc=tag, unit="it", file=sys.stdout, ncols=110, mininterval=1)
    stop_ev = threading.Event(); th = threading.Thread(target=progress_worker, args=(pbar, stop_ev, t_start), daemon=True); th.start()
    try:
        summary = lp.run_candidate(cfg, LR, tag, W, W/"checkpoints/lr_pilot/init_seed42.pt",
                                   use_wandb=True, wandb_group="lr_pilot", ckpt_every=CKPT_EVERY)
    finally:
        stop_ev.set(); th.join(timeout=10)
        if summary is not None and summary["status"] == "completed" and pbar.n < total: pbar.update(total - pbar.n)
        pbar.close()
    print(f"\nrun_candidate returned after {(time.time()-t_start)/60:.1f} min | status = {summary['status']}")

if summary["status"] != "completed" or summary["iterations_done"] != total:
    print("run did not complete:", {k: summary[k] for k in ("status", "iterations_done", "rejected", "reject_reasons")})
    raise SystemExit("STOP: candidate did not complete normally. Paste this output to Claude; do not continue.")

# ================= OUTPUT VERIFICATION (local files only decide pass/fail) =================
print("\n=== OUTPUT VERIFICATION ===")
tr = pd.read_csv(f"{prefix}_train.csv"); va = pd.read_csv(f"{prefix}_val.csv")
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
vj = [json.load(open(res_dir/f"{tag}_val_it{i}.json")) if (res_dir/f"{tag}_val_it{i}.json").exists() else None for i in cfg["eval_iters"]]
exp_tr = list(range(25, total + 1, 25))
exp_lr = np.array([LR * cu.lr_factor(i - 1, total, cfg["warmup"], cfg["final_frac"]) for i in tr.iteration])
opt_steps = sorted({int(float(s["step"])) for s in ck["optimizer"]["state"].values()})
checks = {
    "summary status is completed, 4900 iterations done": summary["status"] == "completed" and summary["iterations_done"] == 4900,
    "summary config/plan/init hashes match the protocol": summary["config_hash"] == ch and summary["plan_hash"] == cfg["plan_hash"] and summary["init_state_hash"] == cfg["init_state_hash"],
    "train CSV: iterations 25..4900 (every 25), each exactly once": list(tr.iteration) == exp_tr,
    "train CSV: all losses finite": bool(np.isfinite(tr.loss).all()),
    "train CSV: LR follows the schedule (peak 1e-4, warmup 100, cosine to 5%)": bool(np.allclose(tr.lr.values, exp_lr, rtol=1e-5)),
    "train CSV: grad norms finite in >=95% of rows (sanity)": float(np.isfinite(tr.grad_norm).mean()) >= 0.95,
    "val CSV: iterations exactly 1225, 2450, 3675, 4900": list(va.iteration) == cfg["eval_iters"],
    "val CSV: all losses and mIoU values finite": bool(np.isfinite(va[["val_loss", "miou4", "miou3"]].values).all()),
    "4 validation JSON files exist and record the right iterations": all(j is not None for j in vj) and [j["iteration"] for j in vj] == cfg["eval_iters"],
    "checkpoint: all required keys, iteration 4900, matching config hash": all(k in ck for k in cu.REQUIRED) and ck["iteration"] == 4900 and ck["config_hash"] == ch,
    "checkpoint: RNG states stored": all(ck["rng"].get(k) is not None for k in ("python", "numpy", "torch", "cuda")),
    "checkpoint: optimiser update count equals 4900 - skipped steps": opt_steps == [4900 - summary["skipped_steps_total"]],
    "no leftover .tmp files": not list(W.rglob("*.tmp")),
    "summary carries rejected flag + reasons + rule": all(k in summary for k in ("rejected", "reject_reasons", "miou3_rule")),
    "candidate-1 result files still identical to Backup 07": not c1_changed(),
}
for k, v in checks.items(): print(("PASS  " if v else "FAIL  ") + k)
ok = all(checks.values())

wb_conn, wb_err = summary.get("wandb_connected"), summary.get("wandb_errors")
print(f"\nW&B (informational only, NOT part of pass/fail): connected={wb_conn}, logging errors={wb_err}")
if (not wb_conn) or wb_err: print("  note: W&B had a problem; the local CSVs and checkpoint above are the primary record and were checked.")
if all(j is not None and "conf" in j for j in vj):
    tots = [int(np.sum(j["conf"])) for j in vj]
    print(f"info: labelled-pixel total per validation confusion matrix: {tots} -> expected all 194254320: {all(t == 194254320 for t in tots)}")
inc = tr[tr.skipped_steps_total.diff().fillna(tr.skipped_steps_total) > 0]
print("info: 25-iteration windows where skipped_steps_total increased:", dict(zip(inc.iteration, inc.skipped_steps_total)), "| optimiser updates:", opt_steps)

# ================= RESULTS (this candidate only; no selection yet) =================
print(f"\n=== {tag} RESULTS (validation only) ===")
cols = ["iteration", "val_loss", "miou4", "miou3", "pixacc", "iou_soil", "iou_bedrock", "iou_sand", "iou_bigrock"]
print(va[cols].round(4).to_string(index=False))
late = va[va.iteration.isin([3675, 4900])]
print(f"\nprimary score so far (mean val mIoU4 at 3675 and 4900): {late.miou4.mean():.4f}   | mean val loss at those: {late.val_loss.mean():.4f}")
print(f"final train loss (mean of last 4 logged windows): {tr.loss.tail(4).mean():.4f}")
print(f"skipped optimiser steps (fp16 overflow): {summary['skipped_steps_total']} | resumed_from: {summary['resumed_from']} | "
      f"train time of the last segment: {summary['train_time_s']/60:.1f} min (after a resume this covers only the resumed part)")
print(f"REJECTED by the predeclared rule: {summary['rejected']}")
for r in summary["reject_reasons"]: print("   reason:", r)
json.dump(dict(tag=tag, lr=LR, all_checks_passed=ok, checks=checks, config_hash=ch, ckpt_every_used=CKPT_EVERY,
               rejected=summary["rejected"], primary_score_so_far=float(late.miou4.mean()),
               wandb_info_not_part_of_verdict=dict(connected=wb_conn, errors=wb_err),
               verified_at=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(res_dir/f"{tag}_verification.json", "w"), indent=1)
print("\nCANDIDATE 1e-4:", "RUN COMPLETE AND VERIFIED" if ok else "VERIFICATION FAILED - paste output")

=== PRE-RUN CHECKS ===
[ok] 9 code/protocol/init/split files identical to Backup 06
[ok] init weights file SHA256 matches Cell 12
[ok] 9 candidate-1 result files identical to Backup 07 (they must stay unchanged)
[ok] run_candidate signature as validated; ckpt_every=500
[ok] config hash d39a2929178fdf28 | plan hash 9327320a8869... | init hash b16b52beb330...
[ok] protocol: batch 8, 4900 iterations, eval at [1225, 2450, 3675, 4900], reject rule [[2450, 3675], [3675, 4900]] > 0.05
[ok] data: train 9803, val 1192 (no test data)
[ok] GPU 0: Tesla T4, 15.2 of 15.6 GB free
[state] no checkpoint - FRESH start from the shared initial weights

=== TRAINING lr1e-04 (lr=0.0001) - about 13-15 minutes; keep this tab open ===
lr1e-04:   0%|                                                                       | 0/4900 [00:00<?, ?it/s][lr1e-04] lr=0.0001 | config hash d39a2929178fdf28 | init hash b16b52beb330 | plan hash 9327320a8869 | start iteration 0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


lr1e-04: 100%|█| 4900/4900 [13:10<00:00,  7.25it/s, loss=0.0805, lr=5.0e-06, skipped=4, val_mIoU4=0.8122@3675]

train/elapsed_s,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇██
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,▂▃▅▆█▄▂▃▃▅▂▃▃▂▂▂▃▄▂▁▂▃▂▁▁▂▂▂▂▂▁▁▁▂▁▂▁▁▁▂
train/iteration,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇████
train/loss,█▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,███████▇▇▇▇▇▇▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train/skipped_steps_total,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁██████
val/epoch,▁▃▆█
val/eval_s,▅█▁▄
val/iou_bedrock,▁███
+16,...


lr1e-04: 100%|█| 4900/4900 [13:15<00:00,  6.16it/s, loss=0.0805, lr=5.0e-06, skipped=4, val_mIoU4=0.8122@3675]

run_candidate returned after 13.3 min | status = completed

=== OUTPUT VERIFICATION ===
PASS  summary status is completed, 4900 iterations done
PASS  summary config/plan/init hashes match the protocol
PASS  train CSV: iterations 25..4900 (every 25), each exactly once
PASS  train CSV: all losses finite
PASS  train CSV: LR follows the schedule (peak 1e-4, warmup 100, cosine to 5%)
PASS  train CSV: grad norms finite in >=95% of rows (sanity)
PASS  val CSV: iterations exactly 1225, 2450, 3675, 4900
PASS  val CSV: all losses and mIoU values finite
PASS  4 validation JSON files exist and record the right iterations
PASS  checkpoint: all required keys, iteration 4900, matching config hash
PASS  checkpoint: RNG states stored
PASS  checkpoint: optimiser update count equals 4900 - skipped steps
PASS  no leftover .tmp files
PASS  summary carries rejected flag + reasons + rule
PASS  cand

In [21]:
import os, json, hashlib, zipfile, time
from pathlib import Path

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/backup_08_candidates1-2.zip")
BACKUP7 = Path("/kaggle/working/backup_07_candidate1_lr3e-05.zip")
TAGS = ["lr3e-05", "lr1e-04"]
def sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

required = [f"code/{n}" for n in ("mars_common.py", "ckpt_utils.py", "tracking.py", "mars_det.py", "recovery.py", "lr_pilot.py")] + [
    "configs/lr_pilot_protocol.json", "splits/split_v3_FROZEN.json", "checkpoints/lr_pilot/init_seed42.pt"]
for t in TAGS:
    required += [f"checkpoints/lr_pilot/{t}/last.pt", f"logs/lr_pilot/{t}_train.csv", f"logs/lr_pilot/{t}_val.csv",
                 f"results/lr_pilot/{t}_summary.json", f"results/lr_pilot/{t}_verification.json"] + \
                [f"results/lr_pilot/{t}_val_it{i}.json" for i in (1225, 2450, 3675, 4900)]
missing = [r for r in required if not (W / r).is_file()]
if missing:
    print("MISSING REQUIRED FILES:"); [print("  ", m) for m in missing]
    raise SystemExit("STOP: paste this output to Claude. No ZIP was created.")

files = list(required)
for p in sorted(W.rglob("*")):
    if not p.is_file(): continue
    rel = p.relative_to(W).as_posix()
    if rel in files: continue
    if rel.startswith(("backups/", "_dryrun", "_ruletest")) or "__pycache__" in rel or rel.endswith(".tmp"): continue
    if p.stat().st_size > 200e6: print("skipped (>200 MB):", rel); continue
    files.append(rel)

# candidate-1 files must be unchanged since Backup 07
c1_note = "Backup 07 not found on disk - candidate-1 identity not re-checked here"
if BACKUP7.exists():
    with zipfile.ZipFile(BACKUP7) as z: man7 = json.loads(z.read("MANIFEST.json"))["files"]
    c1 = [f for f in man7 if "lr3e-05" in f]
    drift = [f for f in c1 if sha_file(W / f) != man7[f]["sha256"]]
    c1_note = f"{len(c1)} candidate-1 files identical to Backup 07" if not drift else f"CANDIDATE-1 FILES CHANGED: {drift}"
print("[check]", c1_note)
if "CHANGED" in c1_note: raise SystemExit("STOP: candidate-1 files differ from Backup 07. Paste this output to Claude. No ZIP was created.")

manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "note": "backup after LR-pilot candidates 1 (3e-5) and 2 (1e-4)",
            "files": {f: {"bytes": (W/f).stat().st_size, "sha256": sha_file(W/f)} for f in files}}
OUT.unlink(missing_ok=True)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(W / f, arcname=f)
    z.writestr("MANIFEST.json", json.dumps(manifest, indent=1))
with zipfile.ZipFile(OUT) as z:
    bad = z.testzip()
    mism = [f for f in files if hashlib.sha256(z.read(f)).hexdigest() != manifest["files"][f]["sha256"]]
print(f"ZIP PATH : {OUT}")
print(f"ZIP SIZE : {OUT.stat().st_size/1e6:.2f} MB")
print(f"CRC TEST : {'OK (no bad files)' if bad is None else 'FAILED at ' + bad}")
print(f"CONTENT HASH CHECK: {'all ' + str(len(files)) + ' files match their manifest' if not mism else 'MISMATCH: ' + str(mism)}")
print(f"init weights SHA256 in ZIP: {manifest['files']['checkpoints/lr_pilot/init_seed42.pt']['sha256']} (expected b698a642...04d96)")
print("\nZIP CONTENTS:")
for f in files: print(f"  {manifest['files'][f]['bytes']:>12,}  {f}")
print("  + MANIFEST.json")
print(f"\nZIP SHA256 (Kaggle): {sha_file(OUT)}")

[check] 9 candidate-1 files identical to Backup 07
ZIP PATH : /kaggle/working/backup_08_candidates1-2.zip
ZIP SIZE : 97.53 MB
CRC TEST : OK (no bad files)
CONTENT HASH CHECK: all 33 files match their manifest
init weights SHA256 in ZIP: b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96 (expected b698a642...04d96)

ZIP CONTENTS:
         5,833  code/mars_common.py
         2,960  code/ckpt_utils.py
         2,343  code/tracking.py
         2,638  code/mars_det.py
         3,011  code/recovery.py
         9,127  code/lr_pilot.py
         2,653  configs/lr_pilot_protocol.json
           693  splits/split_v3_FROZEN.json
    14,933,433  checkpoints/lr_pilot/init_seed42.pt
    44,856,523  checkpoints/lr_pilot/lr3e-05/last.pt
        20,558  logs/lr_pilot/lr3e-05_train.csv
         1,543  logs/lr_pilot/lr3e-05_val.csv
         3,566  results/lr_pilot/lr3e-05_summary.json
         1,106  results/lr_pilot/lr3e-05_verification.json
           817  results/lr_pilot/lr3e-05_val_it12

In [22]:
import os, sys, json, csv, hashlib, zipfile, time, inspect, importlib, threading
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm import tqdm

W = Path("/kaggle/working/mars_work"); CODE = W / "code"
BACKUP6 = Path("/kaggle/working/backup_06_lr_pilot_ready.zip")
BACKUP8 = Path("/kaggle/working/backup_08_candidates1-2.zip")
LR = 3e-4
CKPT_EVERY = 500    # not part of the config hash; saves at multiples of 500 plus the 4 evaluation points
EXPECT = dict(
    cfg_hash="01faca90070fdbc0",
    plan_hash="9327320a886992000712839c7831c8346159f49c09b617e8984ec6a75222caaa",
    init_state_hash="b16b52beb330a9e331d1f16e58b0d55d81070286b28323f7610f88ec0fb20f69",
    init_file_sha="b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96")
def stop(msg): raise SystemExit("STOP: " + msg + "\nNothing further was run. Paste this output to Claude.")
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
os.environ["WANDB_DIR"] = "/tmp/wandb"; Path("/tmp/wandb").mkdir(exist_ok=True)

# ================= PRE-RUN CHECKS =================
print("=== PRE-RUN CHECKS ===")
need = [CODE/f for f in ("mars_common.py", "mars_det.py", "ckpt_utils.py", "tracking.py", "lr_pilot.py")] + \
       [W/"configs/lr_pilot_protocol.json", W/"checkpoints/lr_pilot/init_seed42.pt", Path("/tmp/mars_data/train.csv"), BACKUP6, BACKUP8]
miss = [str(p) for p in need if not p.exists()]
if miss: stop(f"session looks reset, missing: {miss}")

# 1. validated files identical to Backup 06; candidate-1 and candidate-2 results identical to Backup 08
with zipfile.ZipFile(BACKUP6) as z: man6 = json.loads(z.read("MANIFEST.json"))["files"]
chk = ["code/mars_common.py", "code/mars_det.py", "code/ckpt_utils.py", "code/tracking.py", "code/recovery.py", "code/lr_pilot.py",
       "configs/lr_pilot_protocol.json", "checkpoints/lr_pilot/init_seed42.pt", "splits/split_v3_FROZEN.json"]
changed = [f for f in chk if sha(W/f) != man6[f]["sha256"]]
if changed: stop(f"files differ from Backup 06: {changed}")
print(f"[ok] {len(chk)} code/protocol/init/split files identical to Backup 06")
if sha(W/"checkpoints/lr_pilot/init_seed42.pt") != EXPECT["init_file_sha"]: stop("init weights file SHA256 differs from Cell 12")
print("[ok] init weights file SHA256 matches Cell 12")
with zipfile.ZipFile(BACKUP8) as z: man8 = json.loads(z.read("MANIFEST.json"))["files"]
prev_files = [f for f in man8 if "lr3e-05" in f or "lr1e-04" in f]
def prev_changed(): return [f for f in prev_files if not (W/f).is_file() or sha(W/f) != man8[f]["sha256"]]
if len(prev_files) != 18 or prev_changed(): stop(f"candidate-1/2 files differ from Backup 08 (found {len(prev_files)} files): {prev_changed()}")
print(f"[ok] {len(prev_files)} candidate-1 and candidate-2 result files identical to Backup 08 (they must stay unchanged)")

# 2. real modules; inspect the real run_candidate()
sys.path.insert(0, str(CODE))
import mars_common as mc, mars_det as md, ckpt_utils as cu, tracking as tk
import lr_pilot as lp; importlib.reload(lp)
sig = inspect.signature(lp.run_candidate)
exp_params = ["cfg", "lr", "tag", "root", "init_path", "use_wandb", "wandb_group", "eval_max_batches",
              "log_every", "ckpt_every", "stop_after", "resume"]
if list(sig.parameters) != exp_params: stop(f"run_candidate parameters differ from the expected list {exp_params}")
if "elif ckpt_every and it % ckpt_every == 0:" not in [l.strip() for l in inspect.getsource(lp.run_candidate).splitlines()]:
    stop("the periodic-checkpoint line is not as expected")
print(f"[ok] run_candidate signature as validated; ckpt_every={CKPT_EVERY}")

# 3. protocol and hashes
protocol = json.load(open(W/"configs/lr_pilot_protocol.json")); cfg = protocol["training_config"]
if LR not in protocol["candidate_lrs"]: stop("3e-4 is not in the protocol's candidate list")
ch = cu.config_hash({**cfg, "lr": LR})
bad = {k: v for k, v in dict(config_hash=(ch, EXPECT["cfg_hash"]), plan_hash=(cfg["plan_hash"], EXPECT["plan_hash"]),
        init_state_hash=(cfg["init_state_hash"], EXPECT["init_state_hash"])).items() if v[0] != v[1]}
if bad: stop(f"hash mismatch (got, expected): {bad}")
print(f"[ok] config hash {ch} | plan hash {cfg['plan_hash'][:12]}... | init hash {cfg['init_state_hash'][:12]}...")
print(f"[ok] protocol: batch {cfg['batch_size']}, {cfg['total_iters']} iterations, eval at {cfg['eval_iters']}, reject rule {cfg['reject_miou3_pairs']} > {cfg['reject_miou3_drop']}")

# 4. data and GPU
ntr, nva = len(mc.MarsSeg("train")), len(mc.MarsSeg("val"))
if (ntr, nva) != (9803, 1192): stop(f"unexpected data sizes train={ntr} val={nva}")
print(f"[ok] data: train {ntr}, val {nva} (no test data)")
if not torch.cuda.is_available(): stop("CUDA is not available")
free, tot = torch.cuda.mem_get_info(0)
if free < 8e9: stop(f"only {free/1e9:.1f} GB free on GPU 0")
print(f"[ok] GPU 0: {torch.cuda.get_device_name(0)}, {free/1e9:.1f} of {tot/1e9:.1f} GB free")

# 5. state of this candidate
tag = lp.tag_for(LR); res_dir = W/"results/lr_pilot"; ck_path = W/"checkpoints/lr_pilot"/tag/"last.pt"
sum_path = res_dir/f"{tag}_summary.json"; prefix = W/"logs/lr_pilot"/tag
total = cfg["total_iters"]; summary = None; start_it = 0
if sum_path.exists():
    old = json.load(open(sum_path))
    if str(old["status"]).startswith("REJECT_NONFINITE"): stop(f"previous run ended with {old['status']} - do not retrain")
    if old["status"] == "completed" and old["iterations_done"] == total:
        summary = old; print(f"[state] {tag} already COMPLETED - skipping training, verification only")
if summary is None:
    if ck_path.exists():
        start_it = torch.load(ck_path, map_location="cpu", weights_only=False)["iteration"]
        print(f"[state] checkpoint found at iteration {start_it} - this run will RESUME from it")
    else: print("[state] no checkpoint - FRESH start from the shared initial weights")

# ================= TRAINING (with a read-only progress-bar thread) =================
def read_last(path, t_min):
    try:
        if not path.exists() or path.stat().st_mtime < t_min: return None     # ignore stale files from before this run
        with open(path, newline="") as f: rows = list(csv.reader(f))
        if len(rows) < 2 or len(rows[-1]) != len(rows[0]): return None
        return dict(zip(rows[0], rows[-1]))
    except Exception: return None

def progress_worker(pbar, stop_ev, t_min):
    while not stop_ev.is_set():
        try:
            r = read_last(Path(f"{prefix}_train.csv"), t_min)
            if r:
                it = int(float(r["iteration"]))
                if it > pbar.n: pbar.update(it - pbar.n)
                post = dict(loss=f"{float(r['loss']):.4f}", lr=f"{float(r['lr']):.1e}", skipped=int(float(r["skipped_steps_total"])))
                v = read_last(Path(f"{prefix}_val.csv"), t_min)
                if v: post["val_mIoU4"] = f"{float(v['miou4']):.4f}@{int(float(v['iteration']))}"
                pbar.set_postfix(post, refresh=True)
        except Exception: pass
        stop_ev.wait(5)

if summary is None:
    print(f"\n=== TRAINING {tag} (lr={LR:g}) - about 13-15 minutes; keep this tab open ===", flush=True)
    t_start = time.time()
    pbar = tqdm(total=total, initial=start_it, desc=tag, unit="it", file=sys.stdout, ncols=110, mininterval=1)
    stop_ev = threading.Event(); th = threading.Thread(target=progress_worker, args=(pbar, stop_ev, t_start), daemon=True); th.start()
    try:
        summary = lp.run_candidate(cfg, LR, tag, W, W/"checkpoints/lr_pilot/init_seed42.pt",
                                   use_wandb=True, wandb_group="lr_pilot", ckpt_every=CKPT_EVERY)
    finally:
        stop_ev.set(); th.join(timeout=10)
        if summary is not None and summary["status"] == "completed" and pbar.n < total: pbar.update(total - pbar.n)
        pbar.close()
    print(f"\nrun_candidate returned after {(time.time()-t_start)/60:.1f} min | status = {summary['status']}")

if summary["status"] != "completed" or summary["iterations_done"] != total:
    print("run did not complete:", {k: summary[k] for k in ("status", "iterations_done", "rejected", "reject_reasons")})
    raise SystemExit("STOP: candidate did not complete normally. Paste this output to Claude; do not continue.")

# ================= OUTPUT VERIFICATION (local files only decide pass/fail) =================
print("\n=== OUTPUT VERIFICATION ===")
tr = pd.read_csv(f"{prefix}_train.csv"); va = pd.read_csv(f"{prefix}_val.csv")
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
vj = [json.load(open(res_dir/f"{tag}_val_it{i}.json")) if (res_dir/f"{tag}_val_it{i}.json").exists() else None for i in cfg["eval_iters"]]
exp_tr = list(range(25, total + 1, 25))
exp_lr = np.array([LR * cu.lr_factor(i - 1, total, cfg["warmup"], cfg["final_frac"]) for i in tr.iteration])
opt_steps = sorted({int(float(s["step"])) for s in ck["optimizer"]["state"].values()})
checks = {
    "summary status is completed, 4900 iterations done": summary["status"] == "completed" and summary["iterations_done"] == 4900,
    "summary config/plan/init hashes match the protocol": summary["config_hash"] == ch and summary["plan_hash"] == cfg["plan_hash"] and summary["init_state_hash"] == cfg["init_state_hash"],
    "train CSV: iterations 25..4900 (every 25), each exactly once": list(tr.iteration) == exp_tr,
    "train CSV: all losses finite": bool(np.isfinite(tr.loss).all()),
    "train CSV: LR follows the schedule (peak 3e-4, warmup 100, cosine to 5%)": bool(np.allclose(tr.lr.values, exp_lr, rtol=1e-5)),
    "train CSV: grad norms finite in >=95% of rows (sanity)": float(np.isfinite(tr.grad_norm).mean()) >= 0.95,
    "val CSV: iterations exactly 1225, 2450, 3675, 4900": list(va.iteration) == cfg["eval_iters"],
    "val CSV: all losses and mIoU values finite": bool(np.isfinite(va[["val_loss", "miou4", "miou3"]].values).all()),
    "4 validation JSON files exist and record the right iterations": all(j is not None for j in vj) and [j["iteration"] for j in vj] == cfg["eval_iters"],
    "checkpoint: all required keys, iteration 4900, matching config hash": all(k in ck for k in cu.REQUIRED) and ck["iteration"] == 4900 and ck["config_hash"] == ch,
    "checkpoint: RNG states stored": all(ck["rng"].get(k) is not None for k in ("python", "numpy", "torch", "cuda")),
    "checkpoint: optimiser update count equals 4900 - skipped steps": opt_steps == [4900 - summary["skipped_steps_total"]],
    "no leftover .tmp files": not list(W.rglob("*.tmp")),
    "summary carries rejected flag + reasons + rule": all(k in summary for k in ("rejected", "reject_reasons", "miou3_rule")),
    "candidate-1 and candidate-2 result files still identical to Backup 08": not prev_changed(),
}
for k, v in checks.items(): print(("PASS  " if v else "FAIL  ") + k)
ok = all(checks.values())

wb_conn, wb_err = summary.get("wandb_connected"), summary.get("wandb_errors")
print(f"\nW&B (informational only, NOT part of pass/fail): connected={wb_conn}, logging errors={wb_err}")
if (not wb_conn) or wb_err: print("  note: W&B had a problem; the local CSVs and checkpoint above are the primary record and were checked.")
if all(j is not None and "conf" in j for j in vj):
    tots = [int(np.sum(j["conf"])) for j in vj]
    print(f"info: labelled-pixel total per validation confusion matrix: {tots} -> expected all 194254320: {all(t == 194254320 for t in tots)}")
inc = tr[tr.skipped_steps_total.diff().fillna(tr.skipped_steps_total) > 0]
print("info: 25-iteration windows where skipped_steps_total increased:", dict(zip(inc.iteration, inc.skipped_steps_total)), "| optimiser updates:", opt_steps)

# ================= RESULTS (this candidate only; no selection yet) =================
print(f"\n=== {tag} RESULTS (validation only) ===")
cols = ["iteration", "val_loss", "miou4", "miou3", "pixacc", "iou_soil", "iou_bedrock", "iou_sand", "iou_bigrock"]
print(va[cols].round(4).to_string(index=False))
late = va[va.iteration.isin([3675, 4900])]
print(f"\nprimary score so far (mean val mIoU4 at 3675 and 4900): {late.miou4.mean():.4f}   | mean val loss at those: {late.val_loss.mean():.4f}")
print(f"final train loss (mean of last 4 logged windows): {tr.loss.tail(4).mean():.4f}")
print(f"skipped optimiser steps (fp16 overflow): {summary['skipped_steps_total']} | resumed_from: {summary['resumed_from']} | "
      f"train time of the last segment: {summary['train_time_s']/60:.1f} min (after a resume this covers only the resumed part)")
print(f"REJECTED by the predeclared rule: {summary['rejected']}")
for r in summary["reject_reasons"]: print("   reason:", r)
json.dump(dict(tag=tag, lr=LR, all_checks_passed=ok, checks=checks, config_hash=ch, ckpt_every_used=CKPT_EVERY,
               rejected=summary["rejected"], primary_score_so_far=float(late.miou4.mean()),
               wandb_info_not_part_of_verdict=dict(connected=wb_conn, errors=wb_err),
               verified_at=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(res_dir/f"{tag}_verification.json", "w"), indent=1)
print("\nCANDIDATE 3e-4:", "RUN COMPLETE AND VERIFIED" if ok else "VERIFICATION FAILED - paste output")

=== PRE-RUN CHECKS ===
[ok] 9 code/protocol/init/split files identical to Backup 06
[ok] init weights file SHA256 matches Cell 12
[ok] 18 candidate-1 and candidate-2 result files identical to Backup 08 (they must stay unchanged)
[ok] run_candidate signature as validated; ckpt_every=500
[ok] config hash 01faca90070fdbc0 | plan hash 9327320a8869... | init hash b16b52beb330...
[ok] protocol: batch 8, 4900 iterations, eval at [1225, 2450, 3675, 4900], reject rule [[2450, 3675], [3675, 4900]] > 0.05
[ok] data: train 9803, val 1192 (no test data)
[ok] GPU 0: Tesla T4, 15.2 of 15.6 GB free
[state] no checkpoint - FRESH start from the shared initial weights

=== TRAINING lr3e-04 (lr=0.0003) - about 13-15 minutes; keep this tab open ===
lr3e-04:   0%|                                                                       | 0/4900 [00:00<?, ?it/s][lr3e-04] lr=0.0003 | config hash 01faca90070fdbc0 | init hash b16b52beb330 | plan hash 9327320a8869 | start iteration 0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


lr3e-04: 100%|█| 4900/4900 [13:10<00:00,  5.77it/s, loss=0.0722, lr=1.5e-05, skipped=4, val_mIoU4=0.7807@3675]

train/elapsed_s,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
train/epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███
train/grad_norm,▇█▆▄▆▄▃▄▂▄▂▃▂▂▅▃▂▂▂▂▂▁▂▂▂▂▂▁▂▂▂▁▁▂▁▂▁▁▁▂
train/iteration,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/loss,█▃▂▂▃▃▂▂▂▂▂▂▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
train/lr,██████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁
train/skipped_steps_total,▁▅▅▅▅███████████████████████████████████
val/epoch,▁▃▆█
val/eval_s,█▄▄▁
val/iou_bedrock,▁▆▅█
+16,...


lr3e-04: 100%|█| 4900/4900 [13:12<00:00,  6.18it/s, loss=0.0722, lr=1.5e-05, skipped=4, val_mIoU4=0.7807@3675]

run_candidate returned after 13.2 min | status = completed

=== OUTPUT VERIFICATION ===
PASS  summary status is completed, 4900 iterations done
PASS  summary config/plan/init hashes match the protocol
PASS  train CSV: iterations 25..4900 (every 25), each exactly once
PASS  train CSV: all losses finite
PASS  train CSV: LR follows the schedule (peak 3e-4, warmup 100, cosine to 5%)
PASS  train CSV: grad norms finite in >=95% of rows (sanity)
PASS  val CSV: iterations exactly 1225, 2450, 3675, 4900
PASS  val CSV: all losses and mIoU values finite
PASS  4 validation JSON files exist and record the right iterations
PASS  checkpoint: all required keys, iteration 4900, matching config hash
PASS  checkpoint: RNG states stored
PASS  checkpoint: optimiser update count equals 4900 - skipped steps
PASS  no leftover .tmp files
PASS  summary carries rejected flag + reasons + rule
PASS  cand

In [23]:
import os, json, hashlib, zipfile, time
from pathlib import Path

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/backup_09_candidates1-3.zip")
BACKUP8 = Path("/kaggle/working/backup_08_candidates1-2.zip")
TAGS = ["lr3e-05", "lr1e-04", "lr3e-04"]
def sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

required = [f"code/{n}" for n in ("mars_common.py", "ckpt_utils.py", "tracking.py", "mars_det.py", "recovery.py", "lr_pilot.py")] + [
    "configs/lr_pilot_protocol.json", "splits/split_v3_FROZEN.json", "checkpoints/lr_pilot/init_seed42.pt"]
for t in TAGS:
    required += [f"checkpoints/lr_pilot/{t}/last.pt", f"logs/lr_pilot/{t}_train.csv", f"logs/lr_pilot/{t}_val.csv",
                 f"results/lr_pilot/{t}_summary.json", f"results/lr_pilot/{t}_verification.json"] + \
                [f"results/lr_pilot/{t}_val_it{i}.json" for i in (1225, 2450, 3675, 4900)]
missing = [r for r in required if not (W / r).is_file()]
if missing:
    print("MISSING REQUIRED FILES:"); [print("  ", m) for m in missing]
    raise SystemExit("STOP: paste this output to Claude. No ZIP was created.")

files = list(required)
for p in sorted(W.rglob("*")):
    if not p.is_file(): continue
    rel = p.relative_to(W).as_posix()
    if rel in files: continue
    if rel.startswith(("backups/", "_dryrun", "_ruletest")) or "__pycache__" in rel or rel.endswith(".tmp"): continue
    if p.stat().st_size > 200e6: print("skipped (>200 MB):", rel); continue
    files.append(rel)

# candidate-1 and candidate-2 files must be unchanged since Backup 08
note = "Backup 08 not found on disk - candidate-1/2 identity not re-checked here"
if BACKUP8.exists():
    with zipfile.ZipFile(BACKUP8) as z: man8 = json.loads(z.read("MANIFEST.json"))["files"]
    prev = [f for f in man8 if "lr3e-05" in f or "lr1e-04" in f]
    drift = [f for f in prev if sha_file(W / f) != man8[f]["sha256"]]
    note = f"{len(prev)} candidate-1 and candidate-2 files identical to Backup 08" if not drift else f"EARLIER CANDIDATE FILES CHANGED: {drift}"
print("[check]", note)
if "CHANGED" in note: raise SystemExit("STOP: earlier candidate files differ from Backup 08. Paste this output to Claude. No ZIP was created.")

manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "note": "backup after all three LR-pilot candidates (3e-5, 1e-4, 3e-4)",
            "files": {f: {"bytes": (W/f).stat().st_size, "sha256": sha_file(W/f)} for f in files}}
OUT.unlink(missing_ok=True)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(W / f, arcname=f)
    z.writestr("MANIFEST.json", json.dumps(manifest, indent=1))
with zipfile.ZipFile(OUT) as z:
    bad = z.testzip()
    mism = [f for f in files if hashlib.sha256(z.read(f)).hexdigest() != manifest["files"][f]["sha256"]]
print(f"ZIP PATH : {OUT}")
print(f"ZIP SIZE : {OUT.stat().st_size/1e6:.2f} MB")
print(f"CRC TEST : {'OK (no bad files)' if bad is None else 'FAILED at ' + bad}")
print(f"CONTENT HASH CHECK: {'all ' + str(len(files)) + ' files match their manifest' if not mism else 'MISMATCH: ' + str(mism)}")
print(f"init weights SHA256 in ZIP: {manifest['files']['checkpoints/lr_pilot/init_seed42.pt']['sha256']} (expected b698a642...04d96)")
print("\nZIP CONTENTS:")
for f in files: print(f"  {manifest['files'][f]['bytes']:>12,}  {f}")
print("  + MANIFEST.json")
print(f"\nZIP SHA256 (Kaggle): {sha_file(OUT)}")

[check] 18 candidate-1 and candidate-2 files identical to Backup 08
ZIP PATH : /kaggle/working/backup_09_candidates1-3.zip
ZIP SIZE : 138.87 MB
CRC TEST : OK (no bad files)
CONTENT HASH CHECK: all 42 files match their manifest
init weights SHA256 in ZIP: b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96 (expected b698a642...04d96)

ZIP CONTENTS:
         5,833  code/mars_common.py
         2,960  code/ckpt_utils.py
         2,343  code/tracking.py
         2,638  code/mars_det.py
         3,011  code/recovery.py
         9,127  code/lr_pilot.py
         2,653  configs/lr_pilot_protocol.json
           693  splits/split_v3_FROZEN.json
    14,933,433  checkpoints/lr_pilot/init_seed42.pt
    44,856,523  checkpoints/lr_pilot/lr3e-05/last.pt
        20,558  logs/lr_pilot/lr3e-05_train.csv
         1,543  logs/lr_pilot/lr3e-05_val.csv
         3,566  results/lr_pilot/lr3e-05_summary.json
         1,106  results/lr_pilot/lr3e-05_verification.json
           817  results/lr_pilo

In [24]:
from pathlib import Path
import hashlib
import zipfile
import json

W = Path("/kaggle/working/mars_work")
B = Path("/kaggle/working/backup_09_candidates1-3.zip")

print("=== FINAL BACKUP AUDIT ===")

# 1. Check backup
assert B.exists(), "STOP: Backup 09 ZIP is missing!"

h = hashlib.sha256(B.read_bytes()).hexdigest()
expected = "5afb59361fe53fa482c1b0f012538573d8aee64dc5005e5bc6a007f3beb6e799"

print("Backup exists:", B.exists())
print("SHA256 matches:", h == expected)

# 2. Check ZIP integrity
with zipfile.ZipFile(B) as z:
    print("ZIP CRC:", z.testzip())
    names = set(z.namelist())

# 3. Check critical files
required = [
    "code/mars_common.py",
    "code/ckpt_utils.py",
    "code/tracking.py",
    "code/mars_det.py",
    "code/recovery.py",
    "code/lr_pilot.py",
    "configs/lr_pilot_protocol.json",
    "splits/split_v3_FROZEN.json",
    "checkpoints/lr_pilot/init_seed42.pt",
]

for lr in ["lr3e-05", "lr1e-04", "lr3e-04"]:
    required += [
        f"checkpoints/lr_pilot/{lr}/last.pt",
        f"logs/lr_pilot/{lr}_train.csv",
        f"logs/lr_pilot/{lr}_val.csv",
        f"results/lr_pilot/{lr}_summary.json",
        f"results/lr_pilot/{lr}_verification.json",
    ]

missing = [f for f in required if f not in names]

print("\nCritical files present:", len(required) - len(missing), "/", len(required))
print("Missing files:", missing)

# 4. Final result
print("\n=== RESULT ===")
if h == expected and not missing:
    print("PASS: BACKUP 09 IS READY.")
else:
    print("STOP: Resolve the issues above before terminating.")

=== FINAL BACKUP AUDIT ===
Backup exists: True
SHA256 matches: True
ZIP CRC: None

Critical files present: 24 / 24
Missing files: []

=== RESULT ===
PASS: BACKUP 09 IS READY.


In [1]:
from pathlib import Path

W = Path("/kaggle/working/mars_work")
B = Path("/kaggle/working/backup_09_candidates1-3.zip")

print("Backup ZIP exists:", B.exists())
print("Project folder exists:", W.exists())

if W.exists():
    print("\nProject folders:")
    for p in sorted(W.iterdir()):
        print(p.name, "->", "folder" if p.is_dir() else "file")

Backup ZIP exists: False
Project folder exists: False


In [1]:
from pathlib import Path
import shutil, zipfile, hashlib, json

INPUT = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512")
W = Path("/kaggle/working/mars_work")

# Locate Backup 09, whether Kaggle exposes it as a ZIP or folder
archives = list(INPUT.rglob("backup_09_candidates1-3.zip"))
folders = [p for p in INPUT.rglob("MANIFEST.json")
           if "backup_09_candidates1-3" in str(p)]

if W.exists():
    raise RuntimeError("STOP: mars_work already exists. Do not overwrite it.")

if archives:
    backup = archives[0]
    with zipfile.ZipFile(backup) as z:
        assert z.testzip() is None, "ZIP integrity check failed."
        z.extractall(W)
elif folders:
    source = folders[0].parent
    shutil.copytree(source, W)
else:
    raise FileNotFoundError("Backup 09 not found in attached dataset.")

# Verify every file against the backup manifest
manifest = json.loads((W / "MANIFEST.json").read_text())
files = manifest["files"]

missing, mismatched = [], []

for name, info in files.items():
    p = W / name
    if not p.is_file():
        missing.append(name)
        continue
    expected = info["sha256"] if isinstance(info, dict) else info
    actual = hashlib.sha256(p.read_bytes()).hexdigest()
    if actual != expected:
        mismatched.append(name)

print("Backup source:", backup if archives else source)
print("Manifest files:", len(files))
print("Missing files:", missing)
print("Hash mismatches:", mismatched)

if missing or mismatched:
    raise RuntimeError("RESTORE FAILED. Do not continue.")

print("\nRESTORE SUCCESSFUL")
print("Project directory:", W)

Backup source: /kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/backup_09_candidates1-3
Manifest files: 42
Missing files: []
Hash mismatches: []

RESTORE SUCCESSFUL
Project directory: /kaggle/working/mars_work


In [2]:
# ===================== Cell 21 - FORMAL LR SELECTION (read-only; CPU only; no training) =====================
import os, sys, json, glob, math, time, hashlib, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

W = Path("/kaggle/working/mars_work")      # the restored project
WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
TIE = 0.01            # predeclared practical tie threshold (a convention, not derived from measured noise)
TOL = 1e-9            # float tolerance, same idea as the 0.05 rejection rule
EVAL_ITERS = [1225, 2450, 3675, 4900]
LATE = (3675, 4900)
CLASSES = ["soil", "bedrock", "sand", "bigrock"]
EXPECT_CFG_HASH = {"lr3e-05": "4a07821ab0dcd5c7", "lr1e-04": "d39a2929178fdf28", "lr3e-04": "01faca90070fdbc0"}
NEW_FILE = "results/lr_pilot/lr_selection.json"


def stop(msg):
    raise SystemExit("STOP: " + msg + "\nNothing was selected or saved. Paste this output to Claude.")


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# ======================= PART 1: decision functions (pure, no file access) =======================
def score_of(c):
    try:
        return float(np.mean([c["miou4"][i] for i in LATE]))
    except KeyError:
        return float("nan")


def loss_of(c):
    try:
        return float(np.mean([c["val_loss"][i] for i in LATE]))
    except KeyError:
        return float("nan")


def miou3_rejections(evals, pairs, threshold, tol=TOL):
    """evals: {iteration: val mIoU3}. Rejects only a drop STRICTLY greater than threshold, only for the listed pairs."""
    out = []
    for a, b in pairs:
        if a in evals and b in evals:
            drop = evals[a] - evals[b]
            if drop > threshold + tol:
                out.append(dict(pair=[a, b], miou3_from=evals[a], miou3_to=evals[b], drop=drop))
    return out


def select_lr(cands, candidate_lrs, tie=TIE, tol=TOL):
    lo, hi = min(candidate_lrs), max(candidate_lrs)
    out = dict(eligible=[], excluded=[], winner=None, tie_break_used=False, tie_break_reason=None,
               top_gap=None, edge_extension_proposed=False, notes=[])
    for c in cands:
        why = []
        if c["rejected"]:
            why.append("rejected by the predeclared rule")
        if c["status"] != "completed":
            why.append("status is " + str(c["status"]))
        if why:
            out["excluded"].append(dict(lr=c["lr"], reasons=why, score=score_of(c)))
        else:
            out["eligible"].append(c)
    rank = sorted(out["eligible"], key=lambda c: (-score_of(c), c["lr"]))
    ex_scores = [e["score"] for e in out["excluded"] if math.isfinite(e["score"])]
    if ex_scores and ((not rank) or max(ex_scores) > score_of(rank[0])):
        out["notes"].append("the highest raw score belongs to an excluded candidate (shown for information only)")
    if not rank:
        out["notes"].append("no eligible candidate: no LR can be selected")
        return out
    if len(rank) == 1:
        out["winner"] = rank[0]["lr"]
        out["notes"].append("only one eligible candidate; selected by default")
    else:
        a, b = rank[0], rank[1]
        gap = score_of(a) - score_of(b)
        out["top_gap"] = gap
        if gap <= tie + tol:
            out["tie_break_used"] = True
            la, lb = loss_of(a), loss_of(b)
            if abs(la - lb) <= 1e-12:
                out["winner"] = min(a["lr"], b["lr"])
                out["tie_break_reason"] = "top two within the tie threshold and equal mean val loss: smaller LR"
            else:
                w = a if la < lb else b
                out["winner"] = w["lr"]
                out["tie_break_reason"] = "top two within the tie threshold (gap %.6f <= %g): lower mean val loss wins" % (gap, tie)
        else:
            out["winner"] = a["lr"]
        for c in rank[2:]:
            if score_of(a) - score_of(c) <= tie + tol:
                out["notes"].append("lr %g is also within %g of the top score; per the protocol only the top two are compared" % (c["lr"], tie))
    if out["winner"] in (lo, hi):
        out["edge_extension_proposed"] = True
    return out


# ======================= PART 2: synthetic tests (run BEFORE any real result is read) =======================
def mk(lr, s, loss=0.1, rejected=False, status="completed"):
    return dict(lr=lr, rejected=rejected, status=status, miou4={3675: s, 4900: s}, val_loss={3675: loss, 4900: loss})


LRS = [3e-5, 1e-4, 3e-4]
t = {}
r = select_lr([mk(3e-5, .70), mk(1e-4, .80), mk(3e-4, .75)], LRS)
t["S1 clear middle winner (gap 0.05): no tie-break, no edge flag"] = r["winner"] == 1e-4 and (not r["tie_break_used"]) and (not r["edge_extension_proposed"])
r = select_lr([mk(3e-5, .60), mk(1e-4, .80, 0.12), mk(3e-4, .79, 0.10)], LRS)
t["S2 float trap: 0.80-0.79 > 0.01 in floats, still a tie -> lower val loss (3e-4) wins"] = (0.80 - 0.79 > 0.01) and r["tie_break_used"] and r["winner"] == 3e-4
r = select_lr([mk(3e-5, .60), mk(1e-4, .800000, 0.12), mk(3e-4, .789999, 0.10)], LRS)
t["S3 gap 0.010001 is NOT a tie -> higher score wins despite higher loss"] = (not r["tie_break_used"]) and r["winner"] == 1e-4
r = select_lr([mk(3e-5, .60), mk(1e-4, .80, 0.10), mk(3e-4, .795, 0.10)], LRS)
t["S4 tie with equal losses -> smaller LR"] = r["tie_break_used"] and r["winner"] == 1e-4
r = select_lr([mk(3e-5, .70), mk(1e-4, .90, rejected=True), mk(3e-4, .75)], LRS)
t["S5 rejected leader excluded; next eligible wins; note recorded"] = r["winner"] == 3e-4 and len(r["excluded"]) == 1 and any("excluded" in n for n in r["notes"])
t["S6a edge winner 3e-5 proposes an extension"] = select_lr([mk(3e-5, .90), mk(1e-4, .70), mk(3e-4, .60)], LRS)["edge_extension_proposed"]
t["S6b edge winner 3e-4 proposes an extension"] = select_lr([mk(3e-5, .60), mk(1e-4, .70), mk(3e-4, .90)], LRS)["edge_extension_proposed"]
r = select_lr([mk(3e-5, .7, rejected=True), mk(1e-4, .8, rejected=True), mk(3e-4, .9, rejected=True)], LRS)
t["S7 all rejected: no winner"] = r["winner"] is None
r = select_lr([mk(3e-5, .7), mk(1e-4, .8, rejected=True), mk(3e-4, .9, status="REJECT_NONFINITE_TRAIN_LOSS")], LRS)
t["S8 one eligible: selected by default (edge LR also triggers the extension flag)"] = r["winner"] == 3e-5 and r["edge_extension_proposed"]
r = select_lr([mk(3e-5, .60), mk(1e-4, .800, 0.20), mk(3e-4, .795, 0.10)], LRS)
t["S9 tie-break may pick the LOWER-scoring candidate when its val loss is lower"] = r["winner"] == 3e-4 and r["tie_break_used"]
r = select_lr([mk(3e-5, .800, 0.30), mk(1e-4, .799, 0.20), mk(3e-4, .795, 0.10)], LRS)
t["S10 third candidate within 0.01 is noted, never used (top two decide)"] = r["winner"] == 1e-4 and any("also within" in n for n in r["notes"])
r = select_lr([mk(3e-5, .5), mk(1e-4, .8, 0.1), mk(3e-4, .6)], LRS)
t["S11 primary score is the mean of the 3675 and 4900 mIoU4 values"] = abs(score_of({"miou4": {3675: .70, 4900: .80}}) - 0.75) < 1e-12 and r["winner"] == 1e-4
bad_c = dict(lr=3e-4, rejected=False, status="REJECT_NONFINITE_TRAIN_LOSS", miou4={1225: .5}, val_loss={1225: .2})
r = select_lr([mk(3e-5, .7), mk(1e-4, .8), bad_c], LRS)
t["S12 a stopped candidate with missing evaluations is excluded without crashing"] = r["winner"] == 1e-4 and len(r["excluded"]) == 1
P3 = [[2450, 3675], [3675, 4900]]
t["R1 mIoU3 drop 0.0501 is rejected"] = [x["pair"] for x in miou3_rejections({2450: 0.8000, 3675: 0.7499}, P3, 0.05)] == [[2450, 3675]]
t["R2 mIoU3 drop of exactly 0.05 is NOT rejected (float trap 0.80-0.75)"] = (0.80 - 0.75 > 0.05) and miou3_rejections({3675: 0.80, 4900: 0.75}, P3, 0.05) == []
t["R3 the 1225->2450 pair is never checked"] = miou3_rejections({1225: 0.9, 2450: 0.5}, P3, 0.05) == []
print("=== PART 2: SYNTHETIC TESTS (no real data read yet) ===")
for k, v in t.items():
    print(("PASS  " if v else "FAIL  ") + k)
if not all(t.values()):
    stop("synthetic tests failed")
print("TESTS: %d/%d PASS\n" % (sum(t.values()), len(t)))


# ======================= PART 3: verify the restored project against the restored manifest =======================
def good_manifest(m):
    return isinstance(m, dict) and isinstance(m.get("files"), dict) and \
        "results/lr_pilot/lr3e-04_summary.json" in m["files"] and "configs/lr_pilot_protocol.json" in m["files"]


def find_manifest():
    """Returns (manifest_dict, description). Looks at the restored project first, then other fixed-depth locations."""
    seen = []
    cand_files = [W / "MANIFEST.json", WORKING / "MANIFEST.json"]
    for base in (WORKING, INPUT):
        for depth in ("*", "*/*", "*/*/*"):
            cand_files += [Path(p) for p in sorted(glob.glob(str(base / depth / "MANIFEST.json")))]
    for p in cand_files:
        if p.is_file() and str(p) not in seen:
            seen.append(str(p))
            try:
                m = json.load(open(p))
            except Exception:
                continue
            if good_manifest(m):
                return m, str(p)
    zips = []
    for base in (WORKING, INPUT):
        for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
            zips += [Path(p) for p in sorted(glob.glob(str(base / depth / "backup_09*.zip")))]
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                m = json.loads(zf.read("MANIFEST.json"))
        except Exception:
            continue
        if good_manifest(m):
            return m, str(z) + " (MANIFEST.json inside the ZIP)"
    return None, None


def snapshot():
    snap = {}
    for root, _dirs, files in os.walk(W):
        for fn in files:
            p = Path(root) / fn
            st = p.stat()
            snap[p.relative_to(W).as_posix()] = (st.st_size, st.st_mtime_ns)
    return snap


print("=== PART 3: PROJECT VERIFICATION AGAINST THE BACKUP 09 MANIFEST ===")
if not W.is_dir():
    stop(str(W) + " does not exist - restore Backup 09 first")
manifest, msrc = find_manifest()
if manifest is None:
    stop("no Backup 09 MANIFEST.json was found (looked in the restored project, /kaggle/working and /kaggle/input)")
files = manifest["files"]
print("manifest used: %s | created %s | %d files listed" % (msrc, manifest.get("created"), len(files)))
missing = [f for f in files if not (W / f).is_file()]
if missing:
    stop("files listed in the manifest are missing from the restored project: " + str(missing))
mismatch = [f for f in files if (W / f).stat().st_size != files[f]["bytes"] or sha_file(W / f) != files[f]["sha256"]]
if mismatch:
    stop("files differ from the manifest: " + str(mismatch))
print("[ok] all %d manifest files present with matching size and SHA256" % len(files))
snap_before = snapshot()
print("[ok] metadata snapshot of %d files under %s taken (compared again at the end)" % (len(snap_before), W))

# ---- protocol (frozen)
protocol = json.load(open(W / "configs/lr_pilot_protocol.json"))
cfg = protocol["training_config"]
lrs = protocol["candidate_lrs"]
if sorted(lrs) != sorted(LRS):
    stop("protocol candidate list is " + str(lrs))
if cfg["eval_iters"] != EVAL_ITERS:
    stop("protocol eval_iters is " + str(cfg["eval_iters"]))
rule = protocol["decision_rule"]
if "3675 and 4900" not in rule["primary"] or "mIoU4" not in rule["primary"]:
    stop("the protocol's primary-score text is not as expected: " + rule["primary"])
if "0.01" not in rule["tie_break"] or "validation loss" not in rule["tie_break"] or "smaller LR" not in rule["tie_break"]:
    stop("the protocol's tie-break text is not as expected: " + rule["tie_break"])
PAIRS, THR = cfg["reject_miou3_pairs"], cfg["reject_miou3_drop"]
if PAIRS != [[2450, 3675], [3675, 4900]] or abs(THR - 0.05) > 1e-12:
    stop("protocol rejection settings are not as expected: %s %s" % (PAIRS, THR))
print("[ok] protocol: candidates %s | primary = mean val mIoU4 at 3675 and 4900 | tie threshold %g | reject pairs %s drop > %g" % (sorted(lrs), TIE, PAIRS, THR))

# ---- the three candidates
class_cols = ["iou_" + c for c in CLASSES] + ["recall_" + c for c in CLASSES] + ["precision_" + c for c in CLASSES]
need_cols = ["iteration", "val_loss", "miou4", "miou3", "pixacc"] + class_cols
need_keys = ["tag", "lr", "status", "rejected", "reject_reasons", "iterations_done", "config_hash",
             "plan_hash", "init_state_hash", "miou3_rule"]
cands, detail, problems, per_cand_hashes = [], {}, [], {}
print("\n=== CANDIDATE FILE CHECKS ===")
for lr in sorted(lrs):
    tag = "lr%.0e" % lr
    vp, sp = W / ("logs/lr_pilot/%s_val.csv" % tag), W / ("results/lr_pilot/%s_summary.json" % tag)
    if not vp.is_file() or not sp.is_file():
        stop("missing validation CSV or summary for " + tag)
    va = pd.read_csv(vp)
    sm = json.load(open(sp))
    miss_c = [c for c in need_cols if c not in va.columns]
    miss_k = [k for k in need_keys if k not in sm]
    if miss_c or miss_k:
        stop("%s: missing CSV columns %s / summary keys %s" % (tag, miss_c, miss_k))
    st = sm["status"]
    completed = (st == "completed")
    chk = {}
    chk["summary tag/lr match"] = sm["tag"] == tag and abs(sm["lr"] - lr) <= 1e-15
    chk["config hash equals the frozen value"] = sm["config_hash"] == EXPECT_CFG_HASH[tag]
    chk["plan hash and init-weights hash equal the protocol's"] = sm["plan_hash"] == cfg["plan_hash"] and sm["init_state_hash"] == cfg["init_state_hash"]
    chk["status is 'completed' or a recognised protocol stop (REJECT_...)"] = completed or str(st).startswith("REJECT_")
    v = va.set_index("iteration")
    evals3 = {int(i): float(v.loc[i, "miou3"]) for i in v.index}
    recomputed = miou3_rejections(evals3, PAIRS, THR)
    if completed:
        chk["4900 iterations done"] = sm["iterations_done"] == 4900
        chk["validation CSV has exactly the 4 expected iterations, each once"] = list(va.iteration) == EVAL_ITERS
        chk["all required CSV values are finite"] = bool(np.isfinite(va[need_cols].astype(float).values).all())
        chk["summary 'rejected' flag agrees with the rule re-applied to the CSV"] = bool(sm["rejected"]) == (len(recomputed) > 0)
        # per-evaluation JSON cross-check (independent copy of the same validation results)
        jok = True
        for i in EVAL_ITERS:
            jp = W / ("results/lr_pilot/%s_val_it%d.json" % (tag, i))
            if not jp.is_file() or i not in v.index:
                jok = False
                continue
            j = json.load(open(jp))
            jok = jok and j["iteration"] == i and abs(j["miou4"] - float(v.loc[i, "miou4"])) <= 1e-7 and abs(j["loss"] - float(v.loc[i, "val_loss"])) <= 1e-7
        chk["CSV agrees with the 4 per-evaluation JSON files (1e-7)"] = jok
    else:
        print("   note: %s did not complete (status %s); it is excluded from selection under the predeclared rule." % (tag, st))
    print("%s (lr=%g): status=%s, summary rejected=%s, rule re-applied to CSV: %s" % (
        tag, lr, st, sm["rejected"], [(x["pair"], round(x["drop"], 6)) for x in recomputed] or "no drop above 0.05"))
    for k, ok in chk.items():
        print("   " + ("PASS  " if ok else "FAIL  ") + k)
        if not ok:
            problems.append(tag + ": " + k)
    rejected_final = bool(sm["rejected"]) or len(recomputed) > 0
    cands.append(dict(lr=lr, tag=tag, rejected=rejected_final, status=st,
                      miou4={int(i): float(v.loc[i, "miou4"]) for i in v.index},
                      val_loss={int(i): float(v.loc[i, "val_loss"]) for i in v.index}))
    detail[lr] = v
    per_cand_hashes[tag] = dict(val_csv_sha256=sha_file(vp), summary_sha256=sha_file(sp))
if problems:
    stop("candidate checks failed:\n   " + "\n   ".join(problems))

# ======================= PART 4: full-precision comparison and the predeclared decision =======================
def at(c, key, it):
    return c[key].get(it, float("nan"))


print("\n=== REAL RESULTS (validation only; actual CSV values, 6 decimals shown) ===")
print("%-8s%-30s%-10s%12s%12s%11s%15s%10s%11s" % ("LR", "status", "rejected", "mIoU4@3675", "mIoU4@4900", "PRIMARY", "val loss@3675", "@4900", "mean loss"))
for c in cands:
    print("%-8g%-30s%-10s%12.6f%12.6f%11.6f%15.6f%10.6f%11.6f" % (
        c["lr"], c["status"], str(c["rejected"]), at(c, "miou4", 3675), at(c, "miou4", 4900), score_of(c),
        at(c, "val_loss", 3675), at(c, "val_loss", 4900), loss_of(c)))
print("\nPrimary scores at full precision:")
for c in cands:
    print("   lr %-8g primary = %.12f | mean val loss = %.12f" % (c["lr"], score_of(c), loss_of(c)))
print("\nPer-class IoU at 4900 | Big Rock detail (reported separately; NOT used for selection)")
print("%-8s%9s%9s%9s%9s | Big Rock IoU/recall/precision @3675      @4900" % ("LR", "soil", "bedrock", "sand", "bigrock"))
for lr in sorted(lrs):
    v = detail[lr]
    if 3675 not in v.index or 4900 not in v.index:
        print("%-8g not available (this candidate stopped before iteration 4900)" % lr)
        continue
    a, b = v.loc[3675], v.loc[4900]
    print("%-8g%9.4f%9.4f%9.4f%9.4f | %.3f/%.3f/%.3f          %.3f/%.3f/%.3f" % (
        lr, b.iou_soil, b.iou_bedrock, b.iou_sand, b.iou_bigrock,
        a.iou_bigrock, a.recall_bigrock, a.precision_bigrock, b.iou_bigrock, b.recall_bigrock, b.precision_bigrock))

res = select_lr(cands, sorted(lrs))
print("\n=== DECISION (predeclared rule) ===")
print("eligible:", sorted(c["lr"] for c in res["eligible"]), "| excluded:", res["excluded"])
if res["top_gap"] is not None:
    print("gap between the top two primary scores: %.9f (tie threshold %g is a convention, not derived from measured noise)" % (res["top_gap"], TIE))
print("tie-break used: %s%s" % (res["tie_break_used"], (" -> " + res["tie_break_reason"]) if res["tie_break_used"] else ""))
for n in res["notes"]:
    print("note:", n)
if res["winner"] is None:
    print("\nNO LR SELECTED")
else:
    print("\nSELECTED STARTING LR (pilot): %g" % res["winner"])
if res["edge_extension_proposed"]:
    print("EDGE RULE: the winner is an edge candidate (3e-5 or 3e-4). An extension should be PROPOSED to the user; NOTHING is run automatically.")
elif res["winner"] is not None:
    print("edge rule: the winner is the interior candidate; no extension is proposed.")
print("caveat: single seed, 4-epoch pilot, in-domain (Curiosity, crowdsourced) validation data; it selects a sensible starting LR, not a proven optimum.")

# ======================= PART 5: save the decision (new file only) and prove nothing else changed =======================
out = W / NEW_FILE
if out.exists():
    print("\n[skip] %s already exists - NOT overwritten" % NEW_FILE)
else:
    record = dict(
        created=time.strftime("%Y-%m-%d %H:%M:%S"), manifest_used=msrc, manifest_created=manifest.get("created"),
        rule=rule, tie_threshold=TIE, tolerance=TOL, selected_lr=res["winner"], tie_break_used=res["tie_break_used"],
        tie_break_reason=res["tie_break_reason"], top_gap=res["top_gap"], edge_extension_proposed=res["edge_extension_proposed"],
        excluded=res["excluded"], notes=res["notes"], source_file_hashes=per_cand_hashes,
        candidates=[dict(lr=c["lr"], tag=c["tag"], rejected=c["rejected"], status=c["status"], primary_score=score_of(c),
                         mean_val_loss=loss_of(c), miou4={str(k): x for k, x in c["miou4"].items()},
                         val_loss={str(k): x for k, x in c["val_loss"].items()}) for c in cands],
        caveats=protocol["caveats"],
        versions=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__))
    with open(out, "x") as f:                       # "x" = create only; fails instead of overwriting
        json.dump(record, f, indent=2)
    print("\n[saved] new file %s (sha256 %s)" % (NEW_FILE, sha_file(out)))

snap_after = snapshot()
added = sorted(set(snap_after) - set(snap_before))
removed = sorted(set(snap_before) - set(snap_after))
changed = sorted(k for k in snap_before if k in snap_after and snap_before[k] != snap_after[k])
mismatch2 = [f for f in files if sha_file(W / f) != files[f]["sha256"]]
print("\n=== NO-MODIFICATION CHECK ===")
print("manifest files re-hashed: %d | hash mismatches: %d" % (len(files), len(mismatch2)))
print("files changed: %s | removed: %s | added: %s (only %s may appear here)" % (changed, removed, added, NEW_FILE))
untouched = (not mismatch2) and (not changed) and (not removed) and set(added) <= {NEW_FILE}
print("NO ORIGINAL CANDIDATE, PROTOCOL, CHECKPOINT, CSV OR SUMMARY FILE WAS MODIFIED:", untouched)
print("\nFORMAL LR SELECTION:", "COMPLETE" if (untouched and res["winner"] is not None) else "FINISHED WITH A PROBLEM - paste output to Claude")


=== PART 2: SYNTHETIC TESTS (no real data read yet) ===
PASS  S1 clear middle winner (gap 0.05): no tie-break, no edge flag
PASS  S2 float trap: 0.80-0.79 > 0.01 in floats, still a tie -> lower val loss (3e-4) wins
PASS  S3 gap 0.010001 is NOT a tie -> higher score wins despite higher loss
PASS  S4 tie with equal losses -> smaller LR
PASS  S5 rejected leader excluded; next eligible wins; note recorded
PASS  S6a edge winner 3e-5 proposes an extension
PASS  S6b edge winner 3e-4 proposes an extension
PASS  S7 all rejected: no winner
PASS  S8 one eligible: selected by default (edge LR also triggers the extension flag)
PASS  S9 tie-break may pick the LOWER-scoring candidate when its val loss is lower
PASS  S10 third candidate within 0.01 is noted, never used (top two decide)
PASS  S11 primary score is the mean of the 3675 and 4900 mIoU4 values
PASS  S12 a stopped candidate with missing evaluations is excluded without crashing
PASS  R1 mIoU3 drop 0.0501 is rejected
PASS  R2 mIoU3 drop of exa

In [3]:
import os, json, hashlib, zipfile, time
from pathlib import Path

W = Path("/kaggle/working/mars_work")
OUT = Path("/kaggle/working/backup_10_lr_selected.zip")
SEL = "results/lr_pilot/lr_selection.json"


def stop(msg):
    raise SystemExit("STOP: " + msg + "\nNo ZIP was created. Paste this output to Claude.")


def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# ---- 1. the project must still equal Backup 09 (restored manifest), plus the new selection file
old_path = W / "MANIFEST.json"
if not old_path.is_file():
    stop("restored MANIFEST.json not found in " + str(W))
old = json.load(open(old_path))["files"]
bad = [f for f in old if (not (W / f).is_file()) or sha_file(W / f) != old[f]["sha256"]]
if bad:
    stop("files differ from the restored Backup 09 manifest: " + str(bad))
print("[ok] all %d Backup 09 files still match the restored manifest" % len(old))
if not (W / SEL).is_file():
    stop(SEL + " not found - run Cell 21 first")
sel = json.load(open(W / SEL))
print("[ok] %s present: selected_lr=%s, tie_break_used=%s, edge_extension_proposed=%s" % (
    SEL, sel["selected_lr"], sel["tie_break_used"], sel["edge_extension_proposed"]))

# ---- 2. collect files: arcname -> source path
src = {f: W / f for f in old}
src[SEL] = W / SEL
src["MANIFEST_backup09.json"] = old_path          # kept for provenance under a different name
extras = []
for p in sorted(W.rglob("*")):
    if not p.is_file():
        continue
    rel = p.relative_to(W).as_posix()
    if rel in src or rel == "MANIFEST.json":
        continue
    if rel.startswith(("backups/", "_dryrun", "_ruletest")) or "__pycache__" in rel or rel.endswith(".tmp"):
        continue
    if p.stat().st_size > 200e6:
        print("skipped (>200 MB):", rel)
        continue
    src[rel] = p
    extras.append(rel)
print("extra files added beyond Backup 09 + selection file:", extras if extras else "none")

# ---- 3. build the ZIP with a manifest, then verify it
manifest = {"created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "note": "backup after formal LR selection (selected lr 1e-4); Backup 09 files + lr_selection.json",
            "parent_manifest": "MANIFEST_backup09.json",
            "files": {a: {"bytes": p.stat().st_size, "sha256": sha_file(p)} for a, p in src.items()}}
OUT.unlink(missing_ok=True)
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for a, p in src.items():
        z.write(p, arcname=a)
    z.writestr("MANIFEST.json", json.dumps(manifest, indent=1))
with zipfile.ZipFile(OUT) as z:
    badcrc = z.testzip()
    names = [i.filename for i in z.infolist()]
    mism = [a for a in src if hashlib.sha256(z.read(a)).hexdigest() != manifest["files"][a]["sha256"]]
print("ZIP PATH : %s" % OUT)
print("ZIP SIZE : %.2f MB" % (OUT.stat().st_size / 1e6))
print("CRC TEST : %s" % ("OK (no bad files)" if badcrc is None else "FAILED at " + badcrc))
print("NO DUPLICATE NAMES IN ZIP: %s" % (len(names) == len(set(names))))
print("CONTENT HASH CHECK: %s" % ("all %d files match their manifest" % len(src) if not mism else "MISMATCH: " + str(mism)))
print("init weights SHA256 in ZIP: %s (expected b698a642...04d96)" % manifest["files"]["checkpoints/lr_pilot/init_seed42.pt"]["sha256"])
print("lr_selection.json SHA256 in ZIP: %s (Cell 21 printed 54e75b51...d5c3)" % manifest["files"][SEL]["sha256"])
print("\nZIP CONTENTS:")
for a in src:
    print("  %12s  %s" % (format(manifest["files"][a]["bytes"], ","), a))
print("  + MANIFEST.json")
print("\nZIP SHA256 (Kaggle): %s" % sha_file(OUT))

[ok] all 42 Backup 09 files still match the restored manifest
[ok] results/lr_pilot/lr_selection.json present: selected_lr=0.0001, tie_break_used=False, edge_extension_proposed=False
extra files added beyond Backup 09 + selection file: none
ZIP PATH : /kaggle/working/backup_10_lr_selected.zip
ZIP SIZE : 138.88 MB
CRC TEST : OK (no bad files)
NO DUPLICATE NAMES IN ZIP: True
CONTENT HASH CHECK: all 44 files match their manifest
init weights SHA256 in ZIP: b698a64211f103892055998142d3af04167121c356afa89a52baedee80004d96 (expected b698a642...04d96)
lr_selection.json SHA256 in ZIP: 54e75b515a4a69e41b46a90ba7db1a8288ce720d47568386726ea29c7bf2d5c3 (Cell 21 printed 54e75b51...d5c3)

ZIP CONTENTS:
         5,833  code/mars_common.py
         2,960  code/ckpt_utils.py
         2,343  code/tracking.py
         2,638  code/mars_det.py
         3,011  code/recovery.py
         9,127  code/lr_pilot.py
         2,653  configs/lr_pilot_protocol.json
           693  splits/split_v3_FROZEN.json
    14,9

In [3]:
from pathlib import Path

B = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/backup_10_lr_selected")

for p in sorted(B.iterdir()):
    print(("DIR  " if p.is_dir() else "FILE "), p.name)

FILE  MANIFEST.json
FILE  MANIFEST_backup09.json
DIR   checkpoints
DIR   code
DIR   configs
DIR   logs
DIR   results
DIR   splits


In [4]:
from pathlib import Path
import shutil, json, hashlib

SRC = Path("/kaggle/input/datasets/hudafatimah/mars-terrain-v3-512/backup_10_lr_selected")
DST = Path("/kaggle/working/mars_work")

# Restore Backup 10
DST.mkdir(parents=True, exist_ok=True)

for item in SRC.iterdir():
    target = DST / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Verify all files against Backup 10 manifest
manifest = json.loads((DST / "MANIFEST.json").read_text())
bad = []

for rel, info in manifest["files"].items():
    p = DST / rel
    if not p.is_file():
        bad.append(f"MISSING: {rel}")
    else:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        if h != info["sha256"]:
            bad.append(f"HASH MISMATCH: {rel}")

if bad:
    print("RESTORE FAILED")
    print("\n".join(bad))
else:
    print("RESTORE SUCCESSFUL")
    print("Verified files:", len(manifest["files"]))
    print("Selected LR:", json.loads(
        (DST / "results/lr_pilot/lr_selection.json").read_text()
    )["selected_lr"])
    print("Project:", DST)

RESTORE SUCCESSFUL
Verified files: 44
Selected LR: 0.0001
Project: /kaggle/working/mars_work
